# Phase 10 — Web Application Development

## Objective
Develop a full-stack web application for BoreWell Sentinel India
to present groundwater quality analytics, machine-learning estimates,
historical trends, and intelligent testing recommendations.

## Proposed technology stack

### Frontend
- React.js
- Tailwind CSS
- Recharts
- Interactive geographic visualizations

### Backend
- Python
- FastAPI
- Pandas
- Scikit-learn
- Joblib

### Data and model sources
- Integrated groundwater master dataset
- Phase 6 trained model bundle
- Phase 7 temporal analysis outputs
- Phase 8 testing-priority registers
- Phase 9 evaluation reports

## Application modules

1. National groundwater dashboard
2. Groundwater sample analysis
3. Multi-contaminant prediction
4. Testing priority explorer
5. Parameter-specific testing recommendations
6. Historical contamination trends
7. Sparse trace-element alerts
8. Model performance and limitations

## Development principles
- Preserve source-data traceability.
- Clearly distinguish measurements from predictions.
- Display model uncertainty and limitations.
- Never present predictions as laboratory-confirmed results.
- Never classify groundwater as safe using predictions alone.
- Keep sparse trace-element alerts separate from the primary score.

## Expected outcome
A functional, tested web application connected to the project's
existing analytical datasets and trained models.

**Status: IN PROGRESS**

In [1]:
# CELL 1 — Phase 10 environment and artifact audit

from pathlib import Path
import importlib.util
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Project root not found.")

PHASE6_DIR = PROJECT_ROOT / "reports/modeling/phase_6"
PHASE7_DIR = PROJECT_ROOT / "reports/temporal_analysis/phase_7"
PHASE8_DIR = PROJECT_ROOT / "reports/testing_prioritization/phase_8"
PHASE9_DIR = PROJECT_ROOT / "reports/model_evaluation/phase_9"

MODEL_BUNDLE = (
    PROJECT_ROOT / "models/phase_6/phase6_model_bundle.joblib"
)

APP_ARTIFACTS = {
    "Model bundle": MODEL_BUNDLE,
    "Site priority register": (
        PHASE8_DIR / "phase8_site_priority_register.csv"
    ),
    "Site action register": (
        PHASE8_DIR / "phase8_site_action_register.csv"
    ),
    "Parameter action register": (
        PHASE8_DIR / "phase8_parameter_action_register.csv"
    ),
    "Trace alert register": (
        PHASE8_DIR / "phase8_sparse_trace_alert_register.csv"
    ),
    "Model evaluation report": (
        PHASE9_DIR / "phase9_model_evaluation_and_validation_report.txt"
    ),
    "Temporal location trends": (
        PHASE7_DIR / "phase7_location_trends_may_only.csv"
    )
}

print("PHASE 10 — APPLICATION ENVIRONMENT")
print("Project root:", PROJECT_ROOT)

print("\nARTIFACT AVAILABILITY")

for name, path in APP_ARTIFACTS.items():
    print(
        f"{name}:",
        "AVAILABLE" if path.exists() else "NOT FOUND",
        f"| {round(path.stat().st_size / (1024**2), 2)} MB"
        if path.exists() else ""
    )

print("\nDEPENDENCY AVAILABILITY")

DEPENDENCIES = [
    "fastapi",
    "uvicorn",
    "pandas",
    "numpy",
    "sklearn",
    "joblib",
    "pydantic"
]

for package in DEPENDENCIES:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'AVAILABLE' if available else 'NOT INSTALLED'}")

PHASE 10 — APPLICATION ENVIRONMENT
Project root: c:\career\projects\BoreWell Sentinel India

ARTIFACT AVAILABILITY
Model bundle: AVAILABLE | 1802.74 MB
Site priority register: AVAILABLE | 15.1 MB
Site action register: AVAILABLE | 30.91 MB
Parameter action register: AVAILABLE | 66.38 MB
Trace alert register: AVAILABLE | 0.18 MB
Model evaluation report: AVAILABLE | 0.0 MB
Temporal location trends: AVAILABLE | 18.91 MB

DEPENDENCY AVAILABILITY
fastapi: NOT INSTALLED
uvicorn: NOT INSTALLED
pandas: AVAILABLE
numpy: AVAILABLE
sklearn: AVAILABLE
joblib: AVAILABLE
pydantic: NOT INSTALLED


In [2]:
# CELL 2 — Inspect saved application data contracts

SITE_PRIORITY_PATH = (
    PHASE8_DIR / "phase8_site_priority_register.csv"
)

SITE_ACTION_PATH = (
    PHASE8_DIR / "phase8_site_action_register.csv"
)

PARAMETER_ACTION_PATH = (
    PHASE8_DIR / "phase8_parameter_action_register.csv"
)

priority_preview = pd.read_csv(
    SITE_PRIORITY_PATH,
    nrows=5
)

action_preview = pd.read_csv(
    SITE_ACTION_PATH,
    nrows=5
)

parameter_action_preview = pd.read_csv(
    PARAMETER_ACTION_PATH,
    nrows=5
)

print("SITE PRIORITY REGISTER")
print(priority_preview.columns.tolist())
display(priority_preview.head(3))

print("\nSITE ACTION REGISTER")
print(action_preview.columns.tolist())
display(action_preview.head(3))

print("\nPARAMETER ACTION REGISTER")
print(parameter_action_preview.columns.tolist())
display(parameter_action_preview.head(3))

SITE PRIORITY REGISTER
['_spatial_group_id', 'State', 'District', 'Latitude', 'Longitude', 'First_observation', 'Latest_observation', 'Total_records', 'Distinct_years', 'Coordinate_missing_records', 'Years_since_last_sample', 'Observed_count__Potential of Hydrogen (pH)', 'Observed_count__Electric Conductivity (μS/cm)', 'Observed_count__Chloride (mg/L)', 'Observed_count__Bicarbonate (mg/L)', 'Observed_count__Sulphate (mg/L)', 'Observed_count__Calcium (mg/L)', 'Observed_count__Magnesium (mg/L)', 'Observed_count__Sodium (mg/L)', 'Smoothed_severity__pH', 'Valid_label_observations__pH', 'Smoothed_severity__Chloride', 'Valid_label_observations__Chloride', 'Smoothed_severity__Magnesium', 'Valid_label_observations__Magnesium', 'Smoothed_severity__Calcium', 'Valid_label_observations__Calcium', 'Smoothed_severity__Hardness', 'Valid_label_observations__Hardness', 'Smoothed_severity__Sulphate', 'Valid_label_observations__Sulphate', 'Smoothed_severity__Alkalinity', 'Valid_label_observations__Alkali

,_spatial_group_id,State,District,Latitude,Longitude,First_observation,Latest_observation,Total_records,Distinct_years,Coordinate_missing_records,...,Priority_explanation,Priority_percentile,Priority_rank,Priority_tier,Sulphate_conditional_unresolved_records,Weighted_contribution__Chemical,Weighted_contribution__Temporal,Weighted_contribution__Monitoring_gap,Weighted_contribution__Uncertainty,Primary_priority_driver
0,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.503889,92.859167,2004-05-25 00:00:00,2021-05-30 08:00:00,8,8,0,...,Routine monitoring evidence; maintain surveill...,13.521220,20928,P3 — Remaining locations,0,5.068336,0.0,10.962355,2.460938,Monitoring_gap
1,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.166944,92.765556,2004-05-25 00:00:00,2021-05-30 08:00:00,7,7,0,...,Routine monitoring evidence; maintain surveill...,22.769536,18690,P3 — Remaining locations,0,7.132636,0.0,10.962355,2.437500,Monitoring_gap
2,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.729167,92.884722,2004-05-25 00:00:00,2021-05-30 08:00:00,7,7,0,...,Routine monitoring evidence; maintain surveill...,16.827142,20128,P3 — Remaining locations,0,5.643343,0.0,10.962355,2.625000,Monitoring_gap



SITE ACTION REGISTER
['_spatial_group_id', 'State', 'District', 'Latitude', 'Longitude', 'First_observation', 'Latest_observation', 'Total_records', 'Distinct_years', 'Coordinate_missing_records', 'Years_since_last_sample', 'Observed_count__Potential of Hydrogen (pH)', 'Observed_count__Electric Conductivity (μS/cm)', 'Observed_count__Chloride (mg/L)', 'Observed_count__Bicarbonate (mg/L)', 'Observed_count__Sulphate (mg/L)', 'Observed_count__Calcium (mg/L)', 'Observed_count__Magnesium (mg/L)', 'Observed_count__Sodium (mg/L)', 'Smoothed_severity__pH', 'Valid_label_observations__pH', 'Smoothed_severity__Chloride', 'Valid_label_observations__Chloride', 'Smoothed_severity__Magnesium', 'Valid_label_observations__Magnesium', 'Smoothed_severity__Calcium', 'Valid_label_observations__Calcium', 'Smoothed_severity__Hardness', 'Valid_label_observations__Hardness', 'Smoothed_severity__Sulphate', 'Valid_label_observations__Sulphate', 'Smoothed_severity__Alkalinity', 'Valid_label_observations__Alkalin

,_spatial_group_id,State,District,Latitude,Longitude,First_observation,Latest_observation,Total_records,Distinct_years,Coordinate_missing_records,...,Baseline_test_count,Targeted_retest_count,Overdue_retest_count,Routine_retest_due_count,Baseline_parameters,Targeted_parameters,Overdue_parameters,Routine_retest_parameters,Action_reason_summary,Sparse_trace_alert_parameters
0,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.503889,92.859167,2004-05-25 00:00:00,2021-05-30 08:00:00,8,8,0,...,0,2,0,6,NaN,"Magnesium (mg/L), Total Hardness (mgCaCO3/L)",NaN,"Calcium (mg/L), Chloride (mg/L), Potential of ...",Potential of Hydrogen (pH): Latest valid param...,NaN
1,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.166944,92.765556,2004-05-25 00:00:00,2021-05-30 08:00:00,7,7,0,...,0,2,0,6,NaN,"Magnesium (mg/L), Total Hardness (mgCaCO3/L)",NaN,"Calcium (mg/L), Chloride (mg/L), Potential of ...",Potential of Hydrogen (pH): Latest valid param...,NaN
2,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.729167,92.884722,2004-05-25 00:00:00,2021-05-30 08:00:00,7,7,0,...,0,2,0,6,NaN,"Magnesium (mg/L), Total Hardness (mgCaCO3/L)",NaN,"Calcium (mg/L), Chloride (mg/L), Potential of ...",Potential of Hydrogen (pH): Latest valid param...,NaN



PARAMETER ACTION REGISTER
['_spatial_group_id', 'State', 'District', 'Latitude', 'Longitude', 'Priority_rank', 'Priority_tier', 'Testing_priority_score', 'Valid_measurements', 'Distinct_measurement_years', 'Latest_measurement', 'Historical_exceedance_count', 'Historical_severe_count', 'Conditional_unresolved_count', 'Mean_historical_severity', 'Parameter', 'Parameter_key', 'Years_since_parameter_sample', 'Has_valid_measurement', 'Testing_reason', 'Testing_recommendation', 'Exceedance_count', 'Severe_exceedance_count', 'Distinct_exceedance_years', 'Latest_exceedance_date', 'Years_since_last_exceedance', 'Exceedance_recency', 'Significant_increasing_trend']


,_spatial_group_id,State,District,Latitude,Longitude,Priority_rank,Priority_tier,Testing_priority_score,Valid_measurements,Distinct_measurement_years,...,Has_valid_measurement,Testing_reason,Testing_recommendation,Exceedance_count,Severe_exceedance_count,Distinct_exceedance_years,Latest_exceedance_date,Years_since_last_exceedance,Exceedance_recency,Significant_increasing_trend
0,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.503889,92.859167,20928,P3 — Remaining locations,18.491628,8,8,...,True,Latest valid parameter measurement is 5–10 yea...,Routine retest due,0,0,0,NaN,NaN,No historical exceedance,False
1,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.166944,92.765556,18690,P3 — Remaining locations,20.532491,7,7,...,True,Latest valid parameter measurement is 5–10 yea...,Routine retest due,0,0,0,NaN,NaN,No historical exceedance,False
2,andaman and nicobar islands|north middle andam...,andaman and nicobar islands,north middle andaman,12.729167,92.884722,20128,P3 — Remaining locations,19.230697,7,7,...,True,Latest valid parameter measurement is 5–10 yea...,Routine retest due,0,0,0,NaN,NaN,No historical exceedance,False


In [3]:
# CELL 3 — Validate required application fields

REQUIRED_APP_COLUMNS = {
    "site_priority": [
        "_spatial_group_id",
        "Priority_rank",
        "Testing_priority_score",
        "Priority_tier"
    ],
    "site_actions": [
        "_spatial_group_id",
        "Recommended_test_count",
        "Baseline_test_count",
        "Targeted_retest_count",
        "Overdue_retest_count"
    ]
}

DATA_PREVIEWS = {
    "site_priority": priority_preview,
    "site_actions": action_preview
}

for dataset_name, required_columns in REQUIRED_APP_COLUMNS.items():
    available_columns = DATA_PREVIEWS[dataset_name].columns.tolist()

    missing = [
        col for col in required_columns
        if col not in available_columns
    ]

    print(f"\n{dataset_name.upper()}")

    if missing:
        print("Missing required fields:", missing)
    else:
        print("All required fields available.")

    assert not missing, (
        f"{dataset_name} does not satisfy its application data contract."
    )

print("\nPHASE 10 INITIAL DATA CONTRACT VALIDATION PASSED")


SITE_PRIORITY
All required fields available.

SITE_ACTIONS
All required fields available.

PHASE 10 INITIAL DATA CONTRACT VALIDATION PASSED


In [4]:
# CELL 4 — Development tools and model bundle inspection

import subprocess
import shutil
import json
import sys

print("PYTHON ENVIRONMENT")
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("\nDEVELOPMENT TOOLS")

for command in ["node", "npm", "git"]:
    executable = shutil.which(command)

    if executable:
        result = subprocess.run(
            [executable, "--version"],
            capture_output=True,
            text=True,
            timeout=15
        )
        print(f"{command}: {result.stdout.strip() or result.stderr.strip()}")
    else:
        print(f"{command}: NOT FOUND")

print("\nMODEL BUNDLE FILE INSPECTION")

bundle_path = PROJECT_ROOT / "models" / "phase_6" / "phase6_model_bundle.joblib"

with open(bundle_path, "rb") as file:
    signature = file.read(32)

print("File size (GB):", round(bundle_path.stat().st_size / (1024**3), 3))
print("First 32 bytes (hex):", signature.hex())

print("\nPHASE 6 MODEL SAVING CODE")

phase6_notebook_path = (
    PROJECT_ROOT / "notebooks" / "06_multi_contaminant_prediction.ipynb"
)

if phase6_notebook_path.exists():

    notebook = json.loads(
        phase6_notebook_path.read_text(encoding="utf-8")
    )

    found_saving_code = False

    for cell_number, cell in enumerate(notebook["cells"], start=1):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        matches = [
            i for i, line in enumerate(lines)
            if "joblib.dump" in line
        ]

        for match in matches:
            found_saving_code = True

            start = max(0, match - 12)
            end = min(len(lines), match + 8)

            print(f"\nNotebook cell {cell_number}:")
            print("\n".join(lines[start:end]))

    if not found_saving_code:
        print("No direct joblib.dump call found.")
else:
    print("Phase 6 notebook not found at expected path.")

PYTHON ENVIRONMENT
Python executable: c:\career\projects\BoreWell Sentinel India\.venv\Scripts\python.exe
Python version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jun 18 2026, 21:18:42) [MSC v.1942 64 bit (AMD64)]

DEVELOPMENT TOOLS
node: v24.14.0
npm: 11.21.3
git: git version 2.53.0.windows.1

MODEL BUNDLE FILE INSPECTION
File size (GB): 1.76
First 32 bytes (hex): 785ed4bd077813c713382a8a71a1994e284110126a8829a10590e804530ca185

PHASE 6 MODEL SAVING CODE
Phase 6 notebook not found at expected path.


In [5]:
# CELL 5 — Full application data contract audit

SCHEMA_FILES = {
    "site_priority": PHASE8_DIR / "phase8_site_priority_register.csv",
    "site_actions": PHASE8_DIR / "phase8_site_action_register.csv",
    "parameter_actions": PHASE8_DIR / "phase8_parameter_action_register.csv",
    "trace_alerts": PHASE8_DIR / "phase8_sparse_trace_alert_register.csv",
    "temporal_trends": PHASE7_DIR / "phase7_location_trends_may_only.csv",
    "validation_summary": PHASE9_DIR / "phase9_target_model_summary.csv"
}

schema_registry = {}

for dataset_name, path in SCHEMA_FILES.items():

    if not path.exists():
        print(f"\n{dataset_name}: FILE MISSING")
        continue

    columns = pd.read_csv(path, nrows=0).columns.tolist()
    schema_registry[dataset_name] = columns

    print(f"\n{'=' * 65}")
    print(f"{dataset_name.upper()}")
    print("Columns:", len(columns))
    print(", ".join(columns))

# Validate site-level uniqueness and counts.
priority_keys = pd.read_csv(
    SCHEMA_FILES["site_priority"],
    usecols=["_spatial_group_id"]
)

action_keys = pd.read_csv(
    SCHEMA_FILES["site_actions"],
    usecols=["_spatial_group_id"]
)

parameter_keys = pd.read_csv(
    SCHEMA_FILES["parameter_actions"],
    usecols=["_spatial_group_id"]
)

priority_unique = priority_keys["_spatial_group_id"].is_unique
action_unique = action_keys["_spatial_group_id"].is_unique

parameter_counts = (
    parameter_keys.groupby("_spatial_group_id")
    .size()
)

print("\nAPPLICATION RECORD INTEGRITY")
print("Priority records:", len(priority_keys))
print("Unique priority locations:", priority_keys["_spatial_group_id"].nunique())
print("Action records:", len(action_keys))
print("Unique action locations:", action_keys["_spatial_group_id"].nunique())
print("Parameter action records:", len(parameter_keys))
print("Unique parameter-action locations:", parameter_keys["_spatial_group_id"].nunique())
print("Minimum parameter rows per location:", parameter_counts.min())
print("Maximum parameter rows per location:", parameter_counts.max())

assert priority_unique
assert action_unique
assert len(priority_keys) == 24_199
assert len(action_keys) == 24_199
assert len(parameter_keys) == 193_592
assert parameter_counts.eq(8).all()

print("\nPASS: Application data contracts and record integrity verified.")


SITE_PRIORITY
Columns: 74
_spatial_group_id, State, District, Latitude, Longitude, First_observation, Latest_observation, Total_records, Distinct_years, Coordinate_missing_records, Years_since_last_sample, Observed_count__Potential of Hydrogen (pH), Observed_count__Electric Conductivity (μS/cm), Observed_count__Chloride (mg/L), Observed_count__Bicarbonate (mg/L), Observed_count__Sulphate (mg/L), Observed_count__Calcium (mg/L), Observed_count__Magnesium (mg/L), Observed_count__Sodium (mg/L), Smoothed_severity__pH, Valid_label_observations__pH, Smoothed_severity__Chloride, Valid_label_observations__Chloride, Smoothed_severity__Magnesium, Valid_label_observations__Magnesium, Smoothed_severity__Calcium, Valid_label_observations__Calcium, Smoothed_severity__Hardness, Valid_label_observations__Hardness, Smoothed_severity__Sulphate, Valid_label_observations__Sulphate, Smoothed_severity__Alkalinity, Valid_label_observations__Alkalinity, Smoothed_severity__TDS, Valid_label_observations__TDS, C

In [6]:
# CELL 6 — Locate notebooks and Phase 6 model serialization code

import os
import json

EXCLUDED_DIRS = {
    ".venv",
    "node_modules",
    ".git",
    "__pycache__",
    "data",
    "models"
}

notebook_files = []

for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [
        directory for directory in dirs
        if directory not in EXCLUDED_DIRS
    ]

    for filename in files:
        if filename.lower().endswith(".ipynb"):
            notebook_files.append(Path(root) / filename)

print("NOTEBOOKS FOUND:", len(notebook_files))

for path in notebook_files:
    print("-", path.relative_to(PROJECT_ROOT))

print("\nSEARCHING FOR MODEL SERIALIZATION CODE")

search_terms = [
    "phase6_model_bundle",
    "joblib.dump",
    "compress=",
    "model_bundle"
]

serialization_matches = []

for path in notebook_files:
    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print("Could not inspect:", path.name, "|", exc)
        continue

    for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lowered = source.lower()

        if any(term.lower() in lowered for term in search_terms):
            serialization_matches.append({
                "notebook": path,
                "cell": cell_number,
                "source": source
            })

for match in serialization_matches:
    print("\nNOTEBOOK:", match["notebook"].name)
    print("CELL:", match["cell"])
    print(match["source"][:5000])

if not serialization_matches:
    print("No serialization code found in the notebooks.")

NOTEBOOKS FOUND: 11
- notebooks\00_environment_check.ipynb
- notebooks\01_dataset_collection.ipynb
- notebooks\02_dataset_audit.ipynb
- notebooks\03_data_cleaning_integration.ipynb
- notebooks\04_exploratory_analysis.ipynb
- notebooks\05_feature_engineering_target_preparation.ipynb
- notebooks\06_multicontaminant_prediction.ipynb
- notebooks\07_temporal_analysis.ipynb
- notebooks\08_testing_prioritization.ipynb
- notebooks\09_model_evaluation.ipynb
- notebooks\10_web_application_deployment.ipynb

SEARCHING FOR MODEL SERIALIZATION CODE

NOTEBOOK: 06_multicontaminant_prediction.ipynb
CELL: 26
import joblib
import sklearn

# Dedicated Phase 6 directories
MODEL_DIR = PROJECT_ROOT / "models" / "phase_6"
REPORT_DIR = PROJECT_ROOT / "reports" / "modeling" / "phase_6"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Build input feature schemas for each chemistry-assisted model.
assisted_input_columns = {}

for target in TARGETS:
    other_features 

In [7]:
# CELL 7 — Install FastAPI backend dependencies

%pip install "fastapi[standard]"

  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.5-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
Using cached opentelemetry_api-1.45.2-py3-none-any.whl (60 kB)
Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl (456 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.3 MB 960.7 kB/s eta 0:00:02
   --------- ----------------

In [8]:
# CELL 8 — Verify installation and Python environment

import sys
import importlib.metadata as metadata

import fastapi
import uvicorn
import pydantic

print("PYTHON EXECUTABLE")
print(sys.executable)

print("\nINSTALLED BACKEND PACKAGES")

for package in ["fastapi", "uvicorn", "pydantic", "starlette"]:
    print(f"{package}: {metadata.version(package)}")

print("\nENVIRONMENT CHECK")

assert ".venv" in sys.executable.lower()
assert fastapi is not None
assert uvicorn is not None
assert pydantic is not None

print("PASS: FastAPI backend environment is ready.")

PYTHON EXECUTABLE
c:\career\projects\BoreWell Sentinel India\.venv\Scripts\python.exe

INSTALLED BACKEND PACKAGES
fastapi: 0.165.0
uvicorn: 0.54.0
pydantic: 2.14.4
starlette: 1.10.0

ENVIRONMENT CHECK
PASS: FastAPI backend environment is ready.


In [9]:
# CELL 9 — Inspect Phase 6 model bundle construction

import json
from pathlib import Path

phase6_notebook_path = (
    PROJECT_ROOT
    / "notebooks"
    / "06_multicontaminant_prediction.ipynb"
)

notebook = json.loads(
    phase6_notebook_path.read_text(encoding="utf-8")
)

found = False

for cell_number, cell in enumerate(notebook["cells"], start=1):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if (
        "joblib.dump" in source
        or "phase6_model_bundle" in source
    ):
        found = True

        print("\n" + "=" * 75)
        print("MATCHING CELL:", cell_number)
        print("TOTAL LINES:", len(source.splitlines()))
        print("=" * 75)

        for line_number, line in enumerate(source.splitlines(), start=1):
            print(f"{line_number:03d}: {line}")

if not found:
    print("No model serialization cell found.")


MATCHING CELL: 26
TOTAL LINES: 152
001: import joblib
002: import sklearn
003: 
004: # Dedicated Phase 6 directories
005: MODEL_DIR = PROJECT_ROOT / "models" / "phase_6"
006: REPORT_DIR = PROJECT_ROOT / "reports" / "modeling" / "phase_6"
007: 
008: MODEL_DIR.mkdir(parents=True, exist_ok=True)
009: REPORT_DIR.mkdir(parents=True, exist_ok=True)
010: 
011: # Build input feature schemas for each chemistry-assisted model.
012: assisted_input_columns = {}
013: 
014: for target in TARGETS:
015:     other_features = [
016:         CHEM_LOG_FEATURES[t]
017:         for t in TARGETS
018:         if t != target
019:     ]
020: 
021:     assisted_input_columns[target] = (
022:         NUMERIC_FEATURES
023:         + other_features
024:         + CATEGORICAL_FEATURES
025:     )
026: 
027: # Create a reproducible model bundle.
028: model_bundle = {
029:     "project": "Borewell Sentinel India",
030:     "phase": 6,
031:     "phase_name": "Multi-Contaminant Prediction",
032: 
033:     "targets": TAR

In [10]:
# CELL 10 — Model directory inventory

MODEL_DIR = PROJECT_ROOT / "models" / "phase_6"

print("MODEL DIRECTORY:", MODEL_DIR)

for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        print(
            path.name,
            "| Size (GB):",
            round(path.stat().st_size / (1024**3), 3)
        )
    else:
        print(path.name, "| DIRECTORY")

print("\nSERIALIZATION LIBRARY VERSIONS")

import sklearn
import joblib

print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("Python:", sys.version)

MODEL DIRECTORY: c:\career\projects\BoreWell Sentinel India\models\phase_6
phase6_model_bundle.joblib | Size (GB): 1.76

SERIALIZATION LIBRARY VERSIONS
scikit-learn: 1.9.1
joblib: 1.6.0
Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jun 18 2026, 21:18:42) [MSC v.1942 64 bit (AMD64)]


In [11]:
# CELL 11 — Inspect Phase 6 serialization logic (Part 1)

import json

phase6_path = (
    PROJECT_ROOT
    / "notebooks"
    / "06_multicontaminant_prediction.ipynb"
)

nb = json.loads(phase6_path.read_text(encoding="utf-8"))

phase6_cell26 = "".join(
    nb["cells"][25]["source"]
)

phase6_lines = phase6_cell26.splitlines()

print("\n".join(
    f"{i+1:03d}: {phase6_lines[i]}"
    for i in range(0, min(55, len(phase6_lines)))
))

001: import joblib
002: import sklearn
003: 
004: # Dedicated Phase 6 directories
005: MODEL_DIR = PROJECT_ROOT / "models" / "phase_6"
006: REPORT_DIR = PROJECT_ROOT / "reports" / "modeling" / "phase_6"
007: 
008: MODEL_DIR.mkdir(parents=True, exist_ok=True)
009: REPORT_DIR.mkdir(parents=True, exist_ok=True)
010: 
011: # Build input feature schemas for each chemistry-assisted model.
012: assisted_input_columns = {}
013: 
014: for target in TARGETS:
015:     other_features = [
016:         CHEM_LOG_FEATURES[t]
017:         for t in TARGETS
018:         if t != target
019:     ]
020: 
021:     assisted_input_columns[target] = (
022:         NUMERIC_FEATURES
023:         + other_features
024:         + CATEGORICAL_FEATURES
025:     )
026: 
027: # Create a reproducible model bundle.
028: model_bundle = {
029:     "project": "Borewell Sentinel India",
030:     "phase": 6,
031:     "phase_name": "Multi-Contaminant Prediction",
032: 
033:     "targets": TARGETS,
034:     "target_transformatio

In [12]:
# CELL 12 — Inspect Phase 6 serialization logic (Part 2)

print("\n".join(
    f"{i+1:03d}: {phase6_lines[i]}"
    for i in range(55, min(110, len(phase6_lines)))
))

056:     },
057: 
058:     "environment": {
059:         "python_version": __import__("sys").version,
060:         "pandas_version": pd.__version__,
061:         "numpy_version": np.__version__,
062:         "scikit_learn_version": sklearn.__version__
063:     }
064: }
065: 
066: MODEL_PATH = MODEL_DIR / "phase6_model_bundle.joblib"
067: 
068: joblib.dump(model_bundle, MODEL_PATH, compress=3)
069: 
070: # Save final model comparison.
071: metrics_export = final_comparison_df.reset_index()
072: 
073: metrics_export["Context training time (seconds)"] = [
074:     training_times[t] for t in TARGETS
075: ]
076: 
077: metrics_export["Assisted training time (seconds)"] = [
078:     final_assisted_training_times[t] for t in TARGETS
079: ]
080: 
081: metrics_export.to_csv(
082:     REPORT_DIR / "phase6_final_model_comparison.csv",
083:     index=False
084: )
085: 
086: # Save test predictions with record traceability.
087: prediction_export = model_df.iloc[test_idx][
088:     [
089:         "_

In [13]:
# CELL 13 — Inspect Phase 6 serialization logic (Part 3)

print("\n".join(
    f"{i+1:03d}: {phase6_lines[i]}"
    for i in range(110, len(phase6_lines))
))

111: prediction_export.to_csv(
112:     REPORT_DIR / "phase6_test_predictions.csv",
113:     index=False
114: )
115: 
116: # Save interpretation and diagnostic reports.
117: feature_importance_df.to_csv(
118:     REPORT_DIR / "phase6_feature_importance.csv",
119:     index=False
120: )
121: 
122: residual_summary.to_csv(
123:     REPORT_DIR / "phase6_residual_diagnostics.csv"
124: )
125: 
126: range_error_summary.to_csv(
127:     REPORT_DIR / "phase6_error_by_value_range.csv",
128:     index=False
129: )
130: 
131: corrected_coverage_df.to_csv(
132:     REPORT_DIR / "phase6_chemistry_assisted_coverage.csv"
133: )
134: 
135: print("PHASE 6 ARTIFACTS SAVED")
136: print("=" * 65)
137: 
138: for path in [
139:     MODEL_PATH,
140:     REPORT_DIR / "phase6_final_model_comparison.csv",
141:     REPORT_DIR / "phase6_test_predictions.csv",
142:     REPORT_DIR / "phase6_feature_importance.csv",
143:     REPORT_DIR / "phase6_residual_diagnostics.csv",
144:     REPORT_DIR / "phase6_error_by_value

In [14]:
# CELL 14 — Focused inspection of model bundle construction

print("MODEL BUNDLE CONSTRUCTION — LINES 20–68")
print("=" * 65)

for i in range(19, 68):
    if i < len(phase6_lines):
        print(f"{i+1:03d}: {phase6_lines[i]}")

print("\nMODEL BUNDLE REFERENCES THROUGHOUT THE CELL")
print("=" * 65)

for i, line in enumerate(phase6_lines):
    if any(term in line.lower() for term in [
        "model_bundle",
        "context_models",
        "assisted_models",
        "context_input",
        "assisted_input",
        "target_transforms",
        "input_columns",
        "preprocessor",
        "pipelines"
    ]):
        print(f"{i+1:03d}: {line}")

MODEL BUNDLE CONSTRUCTION — LINES 20–68
020: 
021:     assisted_input_columns[target] = (
022:         NUMERIC_FEATURES
023:         + other_features
024:         + CATEGORICAL_FEATURES
025:     )
026: 
027: # Create a reproducible model bundle.
028: model_bundle = {
029:     "project": "Borewell Sentinel India",
030:     "phase": 6,
031:     "phase_name": "Multi-Contaminant Prediction",
032: 
033:     "targets": TARGETS,
034:     "target_transformations": TARGET_TRANSFORMS,
035: 
036:     "context_numeric_features": NUMERIC_FEATURES,
037:     "context_categorical_features": CATEGORICAL_FEATURES,
038: 
039:     "chemistry_feature_map": CHEM_LOG_FEATURES,
040:     "chemistry_assisted_input_columns": assisted_input_columns,
041: 
042:     "context_only_models": trained_models,
043:     "chemistry_assisted_models": final_assisted_models,
044: 
045:     "context_model_settings": MODEL_SETTINGS,
046: 
047:     "validation": {
048:         "method": "GroupShuffleSplit",
049:         "random_

In [15]:
# CELL 15 — Inspect model objects and preprocessing definitions
import re

search_terms = [
    "trained_models =",
    "trained_models[",
    "final_assisted_models =",
    "final_assisted_models[",
    "TARGETS =",
    "TARGET_TRANSFORMS =",
    "NUMERIC_FEATURES =",
    "CATEGORICAL_FEATURES =",
    "CHEM_LOG_FEATURES =",
    "Pipeline(",
    "ColumnTransformer(",
    "RandomForestRegressor",
    "FunctionTransformer",
    "TransformedTargetRegressor"
]

for term in search_terms:
    print("\n" + "=" * 65)
    print("SEARCH:", term)
    print("=" * 65)

    matches = 0

    for cell_number, cell in enumerate(nb["cells"], start=1):
        if cell.get("cell_type") != "code":
            continue

        lines = "".join(cell.get("source", [])).splitlines()

        for i, line in enumerate(lines):
            if term.lower() in line.lower():
                start = max(0, i - 3)
                end = min(len(lines), i + 6)

                print(f"\nNotebook cell {cell_number}, line {i+1}")
                print("\n".join(
                    f"{j+1:03d}: {lines[j]}"
                    for j in range(start, end)
                ))

                matches += 1

                if matches >= 5:
                    break

        if matches >= 5:
            break

    if matches == 0:
        print("No matching definition found.")


SEARCH: trained_models =

Notebook cell 12, line 1
001: trained_models = {}
002: training_times = {}
003: 
004: MODEL_SETTINGS = {
005:     "n_estimators": 150,
006:     "max_features": 0.7,

SEARCH: trained_models[

Notebook cell 12, line 32
029: 
030:     elapsed = perf_counter() - start_time
031: 
032:     trained_models[target] = model
033:     training_times[target] = elapsed
034: 
035:     print(f"Completed in {elapsed:.2f} seconds")
036: 
037: print("\n" + "=" * 65)

Notebook cell 13, line 7
004: for target in TARGETS:
005: 
006:     # Model prediction on the transformed scale
007:     predicted_transformed = trained_models[target].predict(X_test)
008: 
009:     # Convert predictions back to original units.
010:     if TARGET_TRANSFORMS[target] == "log1p":
011:         predicted_original = np.expm1(predicted_transformed)
012:     else:

SEARCH: final_assisted_models =

Notebook cell 22, line 1
001: final_assisted_models = {}
002: final_assisted_training_times = {}
003: 
004: pr

In [16]:
# CELL 16 — Print the complete bundle dictionary definition

print("MODEL BUNDLE STRUCTURE")
print("=" * 65)

for i in range(27, min(65, len(phase6_lines))):
    print(f"{i+1:03d}: {phase6_lines[i]}")

MODEL BUNDLE STRUCTURE
028: model_bundle = {
029:     "project": "Borewell Sentinel India",
030:     "phase": 6,
031:     "phase_name": "Multi-Contaminant Prediction",
032: 
033:     "targets": TARGETS,
034:     "target_transformations": TARGET_TRANSFORMS,
035: 
036:     "context_numeric_features": NUMERIC_FEATURES,
037:     "context_categorical_features": CATEGORICAL_FEATURES,
038: 
039:     "chemistry_feature_map": CHEM_LOG_FEATURES,
040:     "chemistry_assisted_input_columns": assisted_input_columns,
041: 
042:     "context_only_models": trained_models,
043:     "chemistry_assisted_models": final_assisted_models,
044: 
045:     "context_model_settings": MODEL_SETTINGS,
046: 
047:     "validation": {
048:         "method": "GroupShuffleSplit",
049:         "random_state": 42,
050:         "test_size": 0.20,
051:         "training_records": len(X_train),
052:         "testing_records": len(X_test),
053:         "training_spatial_groups": len(train_groups),
054:         "testing_spatia

In [17]:
# CELL 17 — Focused inspection of model training pipelines

import json

phase6_path = (
    PROJECT_ROOT
    / "notebooks"
    / "06_multicontaminant_prediction.ipynb"
)

nb = json.loads(phase6_path.read_text(encoding="utf-8"))

search_terms = [
    "TARGETS =",
    "TARGET_TRANSFORMS",
    "NUMERIC_FEATURES =",
    "CATEGORICAL_FEATURES =",
    "CHEM_LOG_FEATURES",
    "ColumnTransformer(",
    "Pipeline(",
    "final_assisted_models",
    "chemistry_assisted",
    "log1p",
    "expm1",
    "inverse_transform"
]

relevant_cells = {}

for cell_number, cell in enumerate(nb["cells"], start=1):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matched_terms = [
        term for term in search_terms
        if term.lower() in source.lower()
    ]

    if matched_terms:
        relevant_cells[cell_number] = {
            "source": source,
            "matches": matched_terms
        }

print("RELEVANT NOTEBOOK CELLS")

for cell_number, details in relevant_cells.items():
    print(
        f"Cell {cell_number}: "
        f"{', '.join(details['matches'])}"
    )

print("\nTRAINING AND PREPROCESSING DEFINITIONS")
print("=" * 75)

for cell_number, details in relevant_cells.items():

    source_lines = details["source"].splitlines()

    # Only show the most important definitions here.
    important = any(
        term in " ".join(details["matches"])
        for term in [
            "final_assisted_models",
            "ColumnTransformer(",
            "Pipeline(",
            "TARGET_TRANSFORMS",
            "CHEM_LOG_FEATURES"
        ]
    )

    if not important:
        continue

    print(f"\n{'=' * 75}")
    print(f"NOTEBOOK CELL {cell_number}")
    print(f"{'=' * 75}")

    for line_number, line in enumerate(source_lines[:110], start=1):
        print(f"{line_number:03d}: {line}")

    if len(source_lines) > 110:
        print(
            f"... {len(source_lines) - 110} additional lines omitted"
        )

RELEVANT NOTEBOOK CELLS
Cell 3: TARGETS =
Cell 4: NUMERIC_FEATURES =, CATEGORICAL_FEATURES =
Cell 10: TARGETS =, TARGET_TRANSFORMS, log1p
Cell 11: ColumnTransformer(, Pipeline(
Cell 12: Pipeline(
Cell 13: TARGET_TRANSFORMS, log1p, expm1
Cell 15: TARGET_TRANSFORMS, log1p
Cell 16: CHEM_LOG_FEATURES, log1p
Cell 17: TARGET_TRANSFORMS, NUMERIC_FEATURES =, CHEM_LOG_FEATURES, ColumnTransformer(, Pipeline(, log1p, expm1
Cell 18: TARGET_TRANSFORMS, NUMERIC_FEATURES =, CHEM_LOG_FEATURES, ColumnTransformer(, Pipeline(, log1p, expm1
Cell 20: TARGETS =
Cell 21: TARGETS =
Cell 22: NUMERIC_FEATURES =, CHEM_LOG_FEATURES, Pipeline(, final_assisted_models
Cell 23: TARGET_TRANSFORMS, NUMERIC_FEATURES =, CHEM_LOG_FEATURES, final_assisted_models, log1p, expm1
Cell 25: final_assisted_models
Cell 26: TARGET_TRANSFORMS, CHEM_LOG_FEATURES, final_assisted_models, chemistry_assisted
Cell 27: chemistry_assisted

TRAINING AND PREPROCESSING DEFINITIONS

NOTEBOOK CELL 10
001: PH_TARGET = "Potential of Hydrogen (pH)"

In [18]:
# CELL 18 — Find the target inverse-transformation logic

for cell_number, cell in enumerate(nb["cells"], start=1):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    matching_lines = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in [
            "expm1(",
            "np.exp(",
            "inverse_transform",
            "target_transformations",
            "prediction_data"
        ])
    ]

    if matching_lines:
        print(f"\nNOTEBOOK CELL {cell_number}")

        shown = set()

        for i in matching_lines:
            for j in range(max(0, i - 3), min(len(lines), i + 5)):
                if j not in shown:
                    print(f"{j+1:03d}: {lines[j]}")
                    shown.add(j)


NOTEBOOK CELL 13
008: 
009:     # Convert predictions back to original units.
010:     if TARGET_TRANSFORMS[target] == "log1p":
011:         predicted_original = np.expm1(predicted_transformed)
012:     else:
013:         predicted_original = predicted_transformed
014: 
015:     actual_original = y_test[target].to_numpy()
023:     )
024: 
025:     if TARGET_TRANSFORMS[target] == "log1p":
026:         baseline_original = np.expm1(baseline_transformed)
027:     else:
028:         baseline_original = baseline_transformed
029: 
030:     # Original-scale evaluation

NOTEBOOK CELL 17
066:     )
067: 
068:     if TARGET_TRANSFORMS[target] == "log1p":
069:         context_pred = np.expm1(context_pred_transformed)
070:     else:
071:         context_pred = context_pred_transformed
072: 
073:     # --------------------------------------------------
125:     )
126: 
127:     if TARGET_TRANSFORMS[target] == "log1p":
128:         assisted_pred = np.expm1(assisted_pred_transformed)
129:     else:
1

In [19]:
# CELL 19 — Extract exact target and feature definitions

import json

phase6_path = (
    PROJECT_ROOT / "notebooks" / "06_multicontaminant_prediction.ipynb"
)

phase6_nb = json.loads(
    phase6_path.read_text(encoding="utf-8")
)

for cell_no in [3, 4, 10]:
    cell = phase6_nb["cells"][cell_no - 1]

    print("\n" + "=" * 70)
    print("PHASE 6 NOTEBOOK CELL:", cell_no)
    print("=" * 70)

    print("".join(cell.get("source", [])))


PHASE 6 NOTEBOOK CELL: 3
TARGETS = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Chloride (mg/L)",
    "Bicarbonate (mg/L)",
    "Sulphate (mg/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)"
]

missing_targets = [col for col in TARGETS if col not in model_df.columns]

if missing_targets:
    print("Missing target columns:", missing_targets)
    raise KeyError("Target column names still do not match the dataset.")

target_rows = []

for col in TARGETS:
    values = pd.to_numeric(model_df[col], errors="coerce")
    valid = values.dropna()

    target_rows.append({
        "Target": col,
        "Valid records": len(valid),
        "Missing records": int(values.isna().sum()),
        "Minimum": valid.min(),
        "Median": valid.median(),
        "Mean": valid.mean(),
        "95th percentile": valid.quantile(0.95),
        "Maximum": valid.max(),
        "Skewness": valid.skew(),
        "Zero values": int((valid == 0).sum()),
   

In [20]:
# CELL 20 — Extract context model pipeline definitions

for cell_no in [11, 12]:
    cell = phase6_nb["cells"][cell_no - 1]

    print("\n" + "=" * 70)
    print("PHASE 6 NOTEBOOK CELL:", cell_no)
    print("=" * 70)

    print("".join(cell.get("source", [])))


PHASE 6 NOTEBOOK CELL: 11
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)
from sklearn.base import clone
from time import perf_counter

# Numeric feature preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical feature preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=5
    ))
])

# Combine both preprocessing pipelines.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)
])

# Leakage check
assert not s

In [21]:
# CELL 21 — Extract chemistry-assisted model definitions

for cell_no in [17, 18, 22, 23]:
    cell = phase6_nb["cells"][cell_no - 1]

    print("\n" + "=" * 70)
    print("PHASE 6 NOTEBOOK CELL:", cell_no)
    print("=" * 70)

    print("".join(cell.get("source", [])))


PHASE 6 NOTEBOOK CELL: 17
VALIDATION_RF_SETTINGS = {
    "n_estimators": 60,
    "max_features": 0.7,
    "min_samples_leaf": 3,
    "max_samples": 0.8,
    "random_state": 42,
    "n_jobs": -1
}

def build_model_preprocessor(numeric_columns):
    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=5
                ))
            ]),
            CATEGORICAL_FEATURES
        )
    ])

validation_results = []
validation_predictions = {}

print("CONTEXT-ONLY VS CHEMISTRY-ASSISTED TRAINING")
print("=" * 65)

for target in TARGETS:

    print(f"\nTARGET: {target}")

    y_fit = y_train_transformed

In [22]:
# CELL 22 — Extract exact Phase 6 feature configuration

for cell_no in [4, 10, 16]:

    source = "".join(
        phase6_nb["cells"][cell_no - 1].get("source", [])
    )

    print("\n" + "=" * 65)
    print(f"NOTEBOOK CELL {cell_no}")
    print("=" * 65)
    print(source)


NOTEBOOK CELL 4
# Context-only predictors for the first modelling baseline.
NUMERIC_FEATURES = [
    "_year_index",
    "_month_sin",
    "_month_cos",
    "_dayofyear_sin",
    "_dayofyear_cos",
    "_latitude",
    "_longitude",
    "_coordinates_missing"
]

CATEGORICAL_FEATURES = [
    "_state_feature",
    "_district_feature"
]

GROUP_COLUMN = "_spatial_group_id"

required_columns = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
    + [GROUP_COLUMN]
)

missing_columns = [
    col for col in required_columns
    if col not in model_df.columns
]

if missing_columns:
    raise KeyError(f"Required modelling columns missing: {missing_columns}")

print("PREDICTOR READINESS")
print("-" * 55)

print("\nNumeric features:")
for col in NUMERIC_FEATURES:
    print(f"{col:28s} | dtype={model_df[col].dtype} | missing={model_df[col].isna().sum():,}")

print("\nCategorical features:")
for col in CATEGORICAL_FEATURES:
    print(
        f"{col:28s} | dtype={model_df[col].dtype} "
        f"| mi

In [23]:
# CELL 23 — Extract context model pipeline definitions

for cell_no in [11, 12]:

    source = "".join(
        phase6_nb["cells"][cell_no - 1].get("source", [])
    )

    print("\n" + "=" * 65)
    print(f"NOTEBOOK CELL {cell_no}")
    print("=" * 65)
    print(source)


NOTEBOOK CELL 11
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)
from sklearn.base import clone
from time import perf_counter

# Numeric feature preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical feature preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=5
    ))
])

# Combine both preprocessing pipelines.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)
])

# Leakage check
assert not set(FEATUR

In [24]:
# CELL 24 — Extract final assisted model construction

for cell_no in [22, 23]:

    source = "".join(
        phase6_nb["cells"][cell_no - 1].get("source", [])
    )

    print("\n" + "=" * 65)
    print(f"NOTEBOOK CELL {cell_no}")
    print("=" * 65)
    print(source)


NOTEBOOK CELL 22
final_assisted_models = {}
final_assisted_training_times = {}

print("FINAL CHEMISTRY-ASSISTED MODEL TRAINING")
print("=" * 65)

for target in TARGETS:

    print(f"\nTraining: {target}")

    own_target_feature = CHEM_LOG_FEATURES[target]

    other_chemical_features = [
        CHEM_LOG_FEATURES[t]
        for t in TARGETS
        if t != target
    ]

    assisted_numeric_features = (
        NUMERIC_FEATURES + other_chemical_features
    )

    assisted_feature_columns = (
        assisted_numeric_features + CATEGORICAL_FEATURES
    )

    # Verify the target itself is not included as an input.
    assert own_target_feature not in assisted_feature_columns

    assisted_preprocessor = build_model_preprocessor(
        assisted_numeric_features
    )

    assisted_model = Pipeline([
        ("preprocessing", assisted_preprocessor),
        ("regressor", RandomForestRegressor(**MODEL_SETTINGS))
    ])

    X_assisted_train = X_chem_full.iloc[train_idx][
        assis

In [25]:
# CELL 25 — Extract Phase 6 final assisted training code

def print_phase6_cell(cell_number):
    source = "".join(
        phase6_nb["cells"][cell_number - 1].get("source", [])
    )

    lines = source.splitlines()

    print(f"NOTEBOOK CELL {cell_number}")
    print(f"TOTAL LINES: {len(lines)}")
    print("=" * 70)

    for i, line in enumerate(lines, start=1):
        print(f"{i:03d}: {line}")


print_phase6_cell(22)

NOTEBOOK CELL 22
TOTAL LINES: 65
001: final_assisted_models = {}
002: final_assisted_training_times = {}
003: 
004: print("FINAL CHEMISTRY-ASSISTED MODEL TRAINING")
005: print("=" * 65)
006: 
007: for target in TARGETS:
008: 
009:     print(f"\nTraining: {target}")
010: 
011:     own_target_feature = CHEM_LOG_FEATURES[target]
012: 
013:     other_chemical_features = [
014:         CHEM_LOG_FEATURES[t]
015:         for t in TARGETS
016:         if t != target
017:     ]
018: 
019:     assisted_numeric_features = (
020:         NUMERIC_FEATURES + other_chemical_features
021:     )
022: 
023:     assisted_feature_columns = (
024:         assisted_numeric_features + CATEGORICAL_FEATURES
025:     )
026: 
027:     # Verify the target itself is not included as an input.
028:     assert own_target_feature not in assisted_feature_columns
029: 
030:     assisted_preprocessor = build_model_preprocessor(
031:         assisted_numeric_features
032:     )
033: 
034:     assisted_model = Pipeline([
0

In [26]:
# CELL 26 — Exact feature and target transformations

print_phase6_cell(4)

NOTEBOOK CELL 4
TOTAL LINES: 68
001: # Context-only predictors for the first modelling baseline.
002: NUMERIC_FEATURES = [
003:     "_year_index",
004:     "_month_sin",
005:     "_month_cos",
006:     "_dayofyear_sin",
007:     "_dayofyear_cos",
008:     "_latitude",
009:     "_longitude",
010:     "_coordinates_missing"
011: ]
012: 
013: CATEGORICAL_FEATURES = [
014:     "_state_feature",
015:     "_district_feature"
016: ]
017: 
018: GROUP_COLUMN = "_spatial_group_id"
019: 
020: required_columns = (
021:     NUMERIC_FEATURES
022:     + CATEGORICAL_FEATURES
023:     + [GROUP_COLUMN]
024: )
025: 
026: missing_columns = [
027:     col for col in required_columns
028:     if col not in model_df.columns
029: ]
030: 
031: if missing_columns:
032:     raise KeyError(f"Required modelling columns missing: {missing_columns}")
033: 
034: print("PREDICTOR READINESS")
035: print("-" * 55)
036: 
037: print("\nNumeric features:")
038: for col in NUMERIC_FEATURES:
039:     print(f"{col:28s} | dtype

In [27]:
# CELL 27 — Chemistry feature construction

print_phase6_cell(16)

NOTEBOOK CELL 16
TOTAL LINES: 52
001: # Create log-transformed chemical input features.
002: CHEM_LOG_FEATURES = {}
003: 
004: X_chem_full = X.copy()
005: 
006: for target in TARGETS:
007:     feature_name = f"_chem_log1p__{target}"
008: 
009:     X_chem_full[feature_name] = np.log1p(
010:         pd.to_numeric(model_df[target], errors="coerce")
011:     )
012: 
013:     CHEM_LOG_FEATURES[target] = feature_name
014: 
015: # Work exclusively within the existing training data.
016: X_chem_train = X_chem_full.iloc[train_idx].copy()
017: 
018: groups_train = groups.iloc[train_idx].reset_index(drop=True)
019: 
020: # Create a separate group-aware validation subset.
021: inner_splitter = GroupShuffleSplit(
022:     n_splits=1,
023:     test_size=0.20,
024:     random_state=77
025: )
026: 
027: inner_fit_idx, inner_val_idx = next(
028:     inner_splitter.split(X_train, groups=groups_train)
029: )
030: 
031: fit_groups = set(groups_train.iloc[inner_fit_idx])
032: validation_groups = set(groups

In [28]:
# CELL 28 — Context and assisted preprocessing

print("CONTEXT PREPROCESSOR")
print_phase6_cell(11)

print("\nCONTEXT MODEL TRAINING")
print_phase6_cell(12)

CONTEXT PREPROCESSOR
NOTEBOOK CELL 11
TOTAL LINES: 47
001: from sklearn.compose import ColumnTransformer
002: from sklearn.pipeline import Pipeline
003: from sklearn.preprocessing import OneHotEncoder
004: from sklearn.impute import SimpleImputer
005: from sklearn.ensemble import RandomForestRegressor
006: from sklearn.metrics import (
007:     mean_absolute_error,
008:     mean_squared_error,
009:     median_absolute_error,
010:     r2_score
011: )
012: from sklearn.base import clone
013: from time import perf_counter
014: 
015: # Numeric feature preprocessing
016: numeric_pipeline = Pipeline([
017:     ("imputer", SimpleImputer(strategy="median"))
018: ])
019: 
020: # Categorical feature preprocessing
021: categorical_pipeline = Pipeline([
022:     ("imputer", SimpleImputer(strategy="most_frequent")),
023:     ("onehot", OneHotEncoder(
024:         handle_unknown="ignore",
025:         min_frequency=5
026:     ))
027: ])
028: 
029: # Combine both preprocessing pipelines.
030: preproc

In [29]:
# CELL 29 — Exact final assisted model construction

source = "".join(
    phase6_nb["cells"][21].get("source", [])
)

lines = source.splitlines()

print("PHASE 6 CELL 22 — LINES 18–65")
print("=" * 70)

for i in range(17, len(lines)):
    print(f"{i+1:03d}: {lines[i]}")

PHASE 6 CELL 22 — LINES 18–65
018: 
019:     assisted_numeric_features = (
020:         NUMERIC_FEATURES + other_chemical_features
021:     )
022: 
023:     assisted_feature_columns = (
024:         assisted_numeric_features + CATEGORICAL_FEATURES
025:     )
026: 
027:     # Verify the target itself is not included as an input.
028:     assert own_target_feature not in assisted_feature_columns
029: 
030:     assisted_preprocessor = build_model_preprocessor(
031:         assisted_numeric_features
032:     )
033: 
034:     assisted_model = Pipeline([
035:         ("preprocessing", assisted_preprocessor),
036:         ("regressor", RandomForestRegressor(**MODEL_SETTINGS))
037:     ])
038: 
039:     X_assisted_train = X_chem_full.iloc[train_idx][
040:         assisted_feature_columns
041:     ]
042: 
043:     start_time = perf_counter()
044: 
045:     assisted_model.fit(
046:         X_assisted_train,
047:         y_train_transformed[target]
048:     )
049: 
050:     elapsed = perf_counter

In [30]:
# CELL 30 — Locate exact preprocessing and transformations

terms = [
    "def build_model_preprocessor",
    "ColumnTransformer(",
    "OneHotEncoder(",
    "SimpleImputer(",
    "RandomForestRegressor(",
    "final_assisted_models[target]",
    "TARGET_TRANSFORMS =",
    "np.log1p(",
    "np.expm1("
]

for term in terms:
    print(f"\n{'='*65}\nSEARCH: {term}\n{'='*65}")

    found = 0

    for cell_number, cell in enumerate(phase6_nb["cells"], start=1):
        if cell.get("cell_type") != "code":
            continue

        cell_lines = "".join(cell.get("source", [])).splitlines()

        for i, line in enumerate(cell_lines):
            if term.lower() in line.lower():
                start = max(0, i - 3)
                end = min(len(cell_lines), i + 7)

                print(f"\nCell {cell_number}, line {i+1}")

                for j in range(start, end):
                    print(f"{j+1:03d}: {cell_lines[j]}")

                found += 1
                if found >= 3:
                    break

        if found >= 3:
            break

    if found == 0:
        print("No match found.")


SEARCH: def build_model_preprocessor

Cell 17, line 10
007:     "n_jobs": -1
008: }
009: 
010: def build_model_preprocessor(numeric_columns):
011:     return ColumnTransformer([
012:         (
013:             "numeric",
014:             Pipeline([
015:                 ("imputer", SimpleImputer(strategy="median"))
016:             ]),

Cell 18, line 10
007:     "n_jobs": -1
008: }
009: 
010: def build_model_preprocessor(numeric_columns):
011:     return ColumnTransformer([
012:         (
013:             "numeric",
014:             Pipeline([
015:                 ("imputer", SimpleImputer(strategy="median"))
016:             ]),

SEARCH: ColumnTransformer(

Cell 11, line 30
027: ])
028: 
029: # Combine both preprocessing pipelines.
030: preprocessor = ColumnTransformer([
031:     ("numeric", numeric_pipeline, NUMERIC_FEATURES),
032:     ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)
033: ])
034: 
035: # Leakage check
036: assert not set(FEATURE_COLUMNS).intersection(TARGE

In [31]:
# CELL 31 — Exact Phase 6 inference configuration

def print_selected_lines(cell_number, start_line, end_line):
    source = "".join(
        phase6_nb["cells"][cell_number - 1].get("source", [])
    )
    lines = source.splitlines()

    print(f"\nCELL {cell_number} | LINES {start_line}–{end_line}")
    print("=" * 65)

    for i in range(start_line - 1, min(end_line, len(lines))):
        print(f"{i+1:03d}: {lines[i]}")


# Complete preprocessing function
print_selected_lines(17, 1, 35)

# Final assisted training, including feature selection and fitting
print_selected_lines(22, 25, 65)


CELL 17 | LINES 1–35
001: VALIDATION_RF_SETTINGS = {
002:     "n_estimators": 60,
003:     "max_features": 0.7,
004:     "min_samples_leaf": 3,
005:     "max_samples": 0.8,
006:     "random_state": 42,
007:     "n_jobs": -1
008: }
009: 
010: def build_model_preprocessor(numeric_columns):
011:     return ColumnTransformer([
012:         (
013:             "numeric",
014:             Pipeline([
015:                 ("imputer", SimpleImputer(strategy="median"))
016:             ]),
017:             numeric_columns
018:         ),
019:         (
020:             "categorical",
021:             Pipeline([
022:                 ("imputer", SimpleImputer(strategy="most_frequent")),
023:                 ("onehot", OneHotEncoder(
024:                     handle_unknown="ignore",
025:                     min_frequency=5
026:                 ))
027:             ]),
028:             CATEGORICAL_FEATURES
029:         )
030:     ])
031: 
032: validation_results = []
033: validation_predictions = {}


In [32]:
# CELL 32 — Exact model settings and target transformations

terms_to_find = [
    "MODEL_SETTINGS =",
    "TARGET_TRANSFORMS =",
    "y_train_transformed",
    "final_assisted_models[target]"
]

for term in terms_to_find:
    print(f"\nSEARCH: {term}")
    print("-" * 60)

    for cell_number, cell in enumerate(phase6_nb["cells"], start=1):
        if cell.get("cell_type") != "code":
            continue

        lines = "".join(cell.get("source", [])).splitlines()

        for i, line in enumerate(lines):
            if term.lower() in line.lower():
                start = max(0, i - 3)
                end = min(len(lines), i + 6)

                print(f"\nCell {cell_number}")
                for j in range(start, end):
                    print(f"{j+1:03d}: {lines[j]}")


SEARCH: MODEL_SETTINGS =
------------------------------------------------------------

Cell 12
001: trained_models = {}
002: training_times = {}
003: 
004: MODEL_SETTINGS = {
005:     "n_estimators": 150,
006:     "max_features": 0.7,
007:     "min_samples_leaf": 3,
008:     "random_state": 42,
009:     "n_jobs": -1

SEARCH: TARGET_TRANSFORMS =
------------------------------------------------------------

Cell 10
005:     if target != PH_TARGET
006: ]
007: 
008: TARGET_TRANSFORMS = {
009:     target: "identity" if target == PH_TARGET else "log1p"
010:     for target in TARGETS
011: }
012: 
013: # Check that log1p is valid for every concentration target.

SEARCH: y_train_transformed
------------------------------------------------------------

Cell 10
016:         raise ValueError(f"Negative values found in {target}; review before log1p.")
017: 
018: # Create transformed copies, retaining original-scale targets.
019: y_train_transformed = y_train.copy()
020: y_test_transformed = y_test

In [33]:
# CELL 33 — Verify temporal feature formulas and preprocessing

import json
from pathlib import Path

# --------------------------------------------------
# 1. Inspect Phase 5 year-index and cyclic features
# --------------------------------------------------

phase5_path = (
    PROJECT_ROOT
    / "notebooks"
    / "05_feature_engineering_target_preparation.ipynb"
)

phase5_nb = json.loads(
    phase5_path.read_text(encoding="utf-8")
)

print("PHASE 5 TEMPORAL FEATURE DEFINITIONS")
print("=" * 70)

for cell_number, cell in enumerate(phase5_nb["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in [
            "_year_index",
            "_month_sin",
            "_dayofyear_sin"
        ])
    ]

    if matches:
        print(f"\nPHASE 5 CELL {cell_number}")

        shown = set()

        for i in matches:
            for j in range(max(0, i - 4), min(len(lines), i + 7)):
                if j not in shown:
                    print(f"{j+1:03d}: {lines[j]}")
                    shown.add(j)

# --------------------------------------------------
# 2. Inspect the complete Phase 6 preprocessor
# --------------------------------------------------

print("\n\nPHASE 6 PREPROCESSOR DEFINITIONS")
print("=" * 70)

for cell_number, cell in enumerate(phase6_nb["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if "def build_model_preprocessor" in source:
        print(f"\nPHASE 6 CELL {cell_number}")
        print(source)

PHASE 5 TEMPORAL FEATURE DEFINITIONS

PHASE 5 CELL 11
005: feature_df["_quarter"] = model_df["_parsed_date"].dt.quarter
006: feature_df["_day_of_year"] = model_df["_parsed_date"].dt.dayofyear
007: 
008: # Numeric year trend
009: feature_df["_year_index"] = feature_df["_year"] - 2000
010: 
011: # Cyclical month encoding
012: feature_df["_month_sin"] = np.sin(
013:     2 * np.pi * feature_df["_month"] / 12
014: )
015: 
016: feature_df["_month_cos"] = np.cos(
017:     2 * np.pi * feature_df["_month"] / 12
018: )
019: 
020: # Cyclical day-of-year encoding
021: feature_df["_dayofyear_sin"] = np.sin(
022:     2 * np.pi * feature_df["_day_of_year"] / 365.25
023: )
024: 
025: feature_df["_dayofyear_cos"] = np.cos(
026:     2 * np.pi * feature_df["_day_of_year"] / 365.25
027: )
032:         [
033:             "_year",
034:             "_month",
035:             "_quarter",
036:             "_year_index",
037:             "_month_sin",
038:             "_month_cos",
039:             "_dayofyear_

In [34]:
# CELL 34 — Extract exact inference requirements

import json
from pathlib import Path

phase6_path = (
    PROJECT_ROOT / "notebooks" / "06_multicontaminant_prediction.ipynb"
)

phase5_path = (
    PROJECT_ROOT / "notebooks" / "05_feature_engineering_target_preparation.ipynb"
)

p6 = json.loads(phase6_path.read_text(encoding="utf-8"))
p5 = json.loads(phase5_path.read_text(encoding="utf-8"))

def show_section(notebook, cell_number, start, end, title):
    source = "".join(notebook["cells"][cell_number - 1]["source"])
    lines = source.splitlines()

    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1:03d}: {lines[i]}")


show_section(
    p5, 11, 1, 35,
    "PHASE 5 TEMPORAL FEATURE ENGINEERING"
)

show_section(
    p6, 11, 15, 47,
    "PHASE 6 PREPROCESSING PIPELINE"
)

show_section(
    p6, 22, 25, 65,
    "PHASE 6 FINAL ASSISTED MODEL TRAINING"
)

show_section(
    p6, 26, 27, 65,
    "PHASE 6 MODEL BUNDLE METADATA"
)


PHASE 5 TEMPORAL FEATURE ENGINEERING
001: feature_df = pd.DataFrame(index=model_df.index)
002: 
003: feature_df["_year"] = model_df["_parsed_date"].dt.year
004: feature_df["_month"] = model_df["_parsed_date"].dt.month
005: feature_df["_quarter"] = model_df["_parsed_date"].dt.quarter
006: feature_df["_day_of_year"] = model_df["_parsed_date"].dt.dayofyear
007: 
008: # Numeric year trend
009: feature_df["_year_index"] = feature_df["_year"] - 2000
010: 
011: # Cyclical month encoding
012: feature_df["_month_sin"] = np.sin(
013:     2 * np.pi * feature_df["_month"] / 12
014: )
015: 
016: feature_df["_month_cos"] = np.cos(
017:     2 * np.pi * feature_df["_month"] / 12
018: )
019: 
020: # Cyclical day-of-year encoding
021: feature_df["_dayofyear_sin"] = np.sin(
022:     2 * np.pi * feature_df["_day_of_year"] / 365.25
023: )
024: 
025: feature_df["_dayofyear_cos"] = np.cos(
026:     2 * np.pi * feature_df["_day_of_year"] / 365.25
027: )
028: 
029: print("TEMPORAL FEATURES CREATED")
030: disp

In [35]:
# CELL 35 — Create BoreWell Sentinel application structure

APP_ROOT = PROJECT_ROOT / "app"
BACKEND_ROOT = APP_ROOT / "backend"
FRONTEND_ROOT = APP_ROOT / "frontend"

BACKEND_DIRS = [
    BACKEND_ROOT / "routes",
    BACKEND_ROOT / "services",
    BACKEND_ROOT / "schemas",
    FRONTEND_ROOT / "src",
    FRONTEND_ROOT / "public"
]

for directory in BACKEND_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

for init_file in [
    BACKEND_ROOT / "__init__.py",
    BACKEND_ROOT / "routes" / "__init__.py",
    BACKEND_ROOT / "services" / "__init__.py",
    BACKEND_ROOT / "schemas" / "__init__.py"
]:
    init_file.touch(exist_ok=True)

print("APPLICATION STRUCTURE CREATED")

for directory in [APP_ROOT, BACKEND_ROOT, FRONTEND_ROOT]:
    print("-", directory)

print("\nExisting project data and models have not been modified.")

APPLICATION STRUCTURE CREATED
- c:\career\projects\BoreWell Sentinel India\app
- c:\career\projects\BoreWell Sentinel India\app\backend
- c:\career\projects\BoreWell Sentinel India\app\frontend

Existing project data and models have not been modified.


In [36]:
# CELL 36 — Write backend configuration and data service

config_source = r'''
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[2]

REPORTS_DIR = PROJECT_ROOT / "reports"

PHASE7_DIR = REPORTS_DIR / "temporal_analysis" / "phase_7"
PHASE8_DIR = REPORTS_DIR / "testing_prioritization" / "phase_8"
PHASE9_DIR = REPORTS_DIR / "model_evaluation" / "phase_9"

MODEL_DIR = PROJECT_ROOT / "models" / "phase_6"

SITE_PRIORITY_PATH = (
    PHASE8_DIR / "phase8_site_priority_register.csv"
)

SITE_ACTION_PATH = (
    PHASE8_DIR / "phase8_site_action_register.csv"
)

PARAMETER_ACTION_PATH = (
    PHASE8_DIR / "phase8_parameter_action_register.csv"
)

TRACE_ALERT_PATH = (
    PHASE8_DIR / "phase8_sparse_trace_alert_register.csv"
)

TEMPORAL_TRENDS_PATH = (
    PHASE7_DIR / "phase7_location_trends_may_only.csv"
)

MODEL_BUNDLE_PATH = (
    MODEL_DIR / "phase6_model_bundle.joblib"
)

APP_VERSION = "0.1.0"
DATASET_LOCATION_COUNT = 24199
PARAMETERS_PER_LOCATION = 8
'''

data_service_source = r'''
from functools import lru_cache
from typing import Any

import numpy as np
import pandas as pd

from backend.config import (
    SITE_PRIORITY_PATH,
    SITE_ACTION_PATH,
    PARAMETER_ACTION_PATH,
    TRACE_ALERT_PATH
)


SITE_COLUMNS = [
    "_spatial_group_id",
    "State",
    "District",
    "Latitude",
    "Longitude",
    "Latest_observation",
    "Total_records",
    "Distinct_years",
    "Years_since_last_sample",
    "Chemical_parameters_tested",
    "Chemical_risk_score",
    "Chemical_coverage_fraction",
    "Data_uncertainty_score",
    "Any_sparse_trace_alert",
    "Significant_increasing_trend_count",
    "Testing_priority_score",
    "Priority_rank",
    "Priority_tier",
    "Priority_explanation",
    "Primary_priority_driver"
]

ACTION_COLUMNS = [
    "_spatial_group_id",
    "Recommended_test_count",
    "Baseline_test_count",
    "Targeted_retest_count",
    "Overdue_retest_count",
    "Routine_retest_due_count",
    "Baseline_parameters",
    "Targeted_parameters",
    "Overdue_parameters",
    "Routine_retest_parameters",
    "Action_reason_summary",
    "Sparse_trace_alert_parameters"
]

PARAMETER_COLUMNS = [
    "_spatial_group_id",
    "Parameter",
    "Parameter_key",
    "Valid_measurements",
    "Distinct_measurement_years",
    "Latest_measurement",
    "Historical_exceedance_count",
    "Historical_severe_count",
    "Conditional_unresolved_count",
    "Years_since_parameter_sample",
    "Has_valid_measurement",
    "Testing_reason",
    "Testing_recommendation",
    "Exceedance_count",
    "Severe_exceedance_count",
    "Distinct_exceedance_years",
    "Latest_exceedance_date",
    "Years_since_last_exceedance",
    "Exceedance_recency",
    "Significant_increasing_trend"
]

TRACE_COLUMNS = [
    "_spatial_group_id",
    "State",
    "District",
    "Latitude",
    "Longitude",
    "Priority_rank",
    "Priority_tier",
    "Testing_priority_score",
    "Sparse_trace_alert_parameters",
    "Any_sparse_trace_alert",
    "Trace_alerts__Arsenic",
    "Trace_alerts__Copper",
    "Trace_alerts__Lead",
    "Trace_alerts__Zinc",
    "Trace_alerts__Nickel",
    "Trace_alerts__Cadmium",
    "Trace_alerts__Chromium",
    "Trace_alerts__Iron_Manganese"
]


def read_selected_csv(path, desired_columns, required_columns):
    if not path.exists():
        raise FileNotFoundError(f"Required application data is missing: {path}")

    available_columns = pd.read_csv(path, nrows=0).columns.tolist()

    missing = [
        column for column in required_columns
        if column not in available_columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns in {path.name}: {missing}"
        )

    selected = [
        column for column in desired_columns
        if column in available_columns
    ]

    frame = pd.read_csv(
        path,
        usecols=selected,
        low_memory=False
    )

    return frame


def validate_unique_locations(frame, name):
    if frame["_spatial_group_id"].duplicated().any():
        raise ValueError(
            f"{name} contains duplicate spatial group identifiers."
        )


@lru_cache(maxsize=1)
def load_app_data():
    site = read_selected_csv(
        SITE_PRIORITY_PATH,
        SITE_COLUMNS,
        [
            "_spatial_group_id",
            "State",
            "District",
            "Testing_priority_score",
            "Priority_rank",
            "Priority_tier"
        ]
    )

    actions = read_selected_csv(
        SITE_ACTION_PATH,
        ACTION_COLUMNS,
        [
            "_spatial_group_id",
            "Recommended_test_count",
            "Baseline_test_count",
            "Targeted_retest_count",
            "Overdue_retest_count",
            "Routine_retest_due_count"
        ]
    )

    parameters = read_selected_csv(
        PARAMETER_ACTION_PATH,
        PARAMETER_COLUMNS,
        [
            "_spatial_group_id",
            "Parameter",
            "Testing_recommendation"
        ]
    )

    traces = read_selected_csv(
        TRACE_ALERT_PATH,
        TRACE_COLUMNS,
        [
            "_spatial_group_id",
            "Sparse_trace_alert_parameters"
        ]
    )

    validate_unique_locations(site, "Site priority register")
    validate_unique_locations(actions, "Site action register")
    validate_unique_locations(traces, "Trace alert register")

    if len(site) != 24199:
        raise ValueError(
            f"Unexpected site count: {len(site)}; expected 24,199."
        )

    if len(actions) != len(site):
        raise ValueError("Site priority and action counts do not match.")

    parameter_counts = parameters.groupby("_spatial_group_id").size()

    if not parameter_counts.eq(8).all():
        raise ValueError(
            "Not every location has exactly eight parameter records."
        )

    site_ids = set(site["_spatial_group_id"])
    action_ids = set(actions["_spatial_group_id"])
    parameter_ids = set(parameters["_spatial_group_id"])

    if site_ids != action_ids or site_ids != parameter_ids:
        raise ValueError(
            "Spatial identifiers do not match across application registers."
        )

    return {
        "sites": site,
        "actions": actions,
        "parameters": parameters,
        "traces": traces
    }


def clean_value(value: Any):
    if value is None:
        return None

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.generic):
        value = value.item()

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return value


def clean_record(record):
    return {
        str(key): clean_value(value)
        for key, value in record.items()
    }


def records_from_frame(frame):
    return [
        clean_record(row)
        for row in frame.to_dict(orient="records")
    ]
'''

(BACKEND_ROOT / "config.py").write_text(
    config_source.strip() + "\n",
    encoding="utf-8"
)

(BACKEND_ROOT / "services" / "data_service.py").write_text(
    data_service_source.strip() + "\n",
    encoding="utf-8"
)

print("BACKEND FILES CREATED")
print(BACKEND_ROOT / "config.py")
print(BACKEND_ROOT / "services" / "data_service.py")

BACKEND FILES CREATED
c:\career\projects\BoreWell Sentinel India\app\backend\config.py
c:\career\projects\BoreWell Sentinel India\app\backend\services\data_service.py


In [37]:
# CELL 37 — Write FastAPI application

main_source = r'''
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware

from backend.config import APP_VERSION
from backend.services.data_service import (
    load_app_data,
    records_from_frame,
    clean_record
)


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Load application datasets once at startup.
    app.state.app_data = load_app_data()
    app.state.data_ready = True

    yield

    app.state.data_ready = False


app = FastAPI(
    title="BoreWell Sentinel India API",
    description=(
        "Groundwater quality analytics, testing prioritization "
        "and research prediction services."
    ),
    version=APP_VERSION,
    lifespan=lifespan
)


app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)


def get_data():
    data = getattr(app.state, "app_data", None)

    if data is None:
        raise HTTPException(
            status_code=503,
            detail="Application datasets are not ready."
        )

    return data


@app.get("/")
def root():
    return {
        "project": "BoreWell Sentinel India",
        "api": "BoreWell Sentinel API",
        "version": APP_VERSION,
        "docs": "/docs"
    }


@app.get("/health")
def health():
    ready = bool(getattr(app.state, "data_ready", False))

    return {
        "status": "healthy" if ready else "starting",
        "data_ready": ready,
        "model_status": "not_loaded",
        "version": APP_VERSION
    }


@app.get("/api/v1/dashboard/summary")
def dashboard_summary():
    data = get_data()
    sites = data["sites"]
    actions = data["actions"]
    traces = data["traces"]

    tier_counts = (
        sites["Priority_tier"]
        .value_counts()
        .to_dict()
    )

    return {
        "total_locations": int(len(sites)),
        "priority_tiers": {
            "P1": int(tier_counts.get("P1 — Highest 10%", 0)),
            "P2": int(tier_counts.get("P2 — Next 15%", 0)),
            "P3": int(tier_counts.get("P3 — Remaining locations", 0))
        },
        "trace_alert_locations": int(len(traces)),
        "low_chemical_evidence_locations": int(
            sites["Chemical_parameters_tested"].le(2).sum()
        ),
        "mean_testing_priority_score": round(
            float(sites["Testing_priority_score"].mean()), 3
        ),
        "mean_recommended_tests": round(
            float(actions["Recommended_test_count"].mean()), 3
        ),
        "total_parameter_action_records": int(len(data["parameters"]))
    }


@app.get("/api/v1/dashboard/states")
def dashboard_states(
    limit: int = Query(default=15, ge=1, le=40)
):
    sites = get_data()["sites"]

    grouped = (
        sites.groupby("State", dropna=False)
        .agg(
            Locations=("_spatial_group_id", "size"),
            Mean_priority_score=("Testing_priority_score", "mean")
        )
        .reset_index()
        .sort_values("Locations", ascending=False)
        .head(limit)
    )

    return {
        "states": records_from_frame(grouped)
    }


@app.get("/api/v1/sites")
def search_sites(
    query: str | None = Query(default=None, max_length=120),
    state: str | None = Query(default=None, max_length=100),
    district: str | None = Query(default=None, max_length=100),
    tier: str | None = Query(default=None),
    low_evidence_only: bool = False,
    limit: int = Query(default=25, ge=1, le=100),
    offset: int = Query(default=0, ge=0)
):
    sites = get_data()["sites"]

    filtered = sites

    if query:
        query = query.strip()

        if query:
            mask = (
                filtered["_spatial_group_id"].astype(str).str.contains(
                    query, case=False, regex=False, na=False
                )
                | filtered["State"].astype(str).str.contains(
                    query, case=False, regex=False, na=False
                )
                | filtered["District"].astype(str).str.contains(
                    query, case=False, regex=False, na=False
                )
            )
            filtered = filtered.loc[mask]

    if state:
        filtered = filtered.loc[
            filtered["State"].str.casefold().eq(state.strip().casefold())
        ]

    if district:
        filtered = filtered.loc[
            filtered["District"].str.casefold().eq(district.strip().casefold())
        ]

    if tier:
        filtered = filtered.loc[
            filtered["Priority_tier"].eq(tier.strip())
        ]

    if low_evidence_only:
        filtered = filtered.loc[
            filtered["Chemical_parameters_tested"].le(2)
        ]

    filtered = filtered.sort_values("Priority_rank")

    total = len(filtered)
    page = filtered.iloc[offset:offset + limit]

    return {
        "total": int(total),
        "limit": limit,
        "offset": offset,
        "sites": records_from_frame(page)
    }


@app.get("/api/v1/sites/{site_id}")
def site_details(site_id: str):
    data = get_data()

    matching = data["sites"].loc[
        data["sites"]["_spatial_group_id"].eq(site_id)
    ]

    if matching.empty:
        raise HTTPException(status_code=404, detail="Location not found.")

    actions = data["actions"].loc[
        data["actions"]["_spatial_group_id"].eq(site_id)
    ]

    parameter_actions = data["parameters"].loc[
        data["parameters"]["_spatial_group_id"].eq(site_id)
    ].sort_values("Parameter")

    return {
        "site": clean_record(matching.iloc[0].to_dict()),
        "actions": (
            clean_record(actions.iloc[0].to_dict())
            if not actions.empty else None
        ),
        "parameter_actions": records_from_frame(parameter_actions)
    }


@app.get("/api/v1/alerts/trace")
def trace_alerts(
    query: str | None = Query(default=None, max_length=120),
    state: str | None = Query(default=None, max_length=100),
    limit: int = Query(default=50, ge=1, le=150),
    offset: int = Query(default=0, ge=0)
):
    traces = get_data()["traces"]

    filtered = traces

    if query:
        mask = (
            filtered["_spatial_group_id"].astype(str).str.contains(
                query.strip(), case=False, regex=False, na=False
            )
            | filtered["District"].astype(str).str.contains(
                query.strip(), case=False, regex=False, na=False
            )
            | filtered["State"].astype(str).str.contains(
                query.strip(), case=False, regex=False, na=False
            )
        )
        filtered = filtered.loc[mask]

    if state:
        filtered = filtered.loc[
            filtered["State"].str.casefold().eq(state.strip().casefold())
        ]

    if "Priority_rank" in filtered.columns:
        filtered = filtered.sort_values("Priority_rank")

    total = len(filtered)

    return {
        "total": int(total),
        "limit": limit,
        "offset": offset,
        "alerts": records_from_frame(
            filtered.iloc[offset:offset + limit]
        )
    }


@app.get("/api/v1/metadata/targets")
def target_metadata():
    return {
        "targets": [
            {
                "name": "Potential of Hydrogen (pH)",
                "unit": "pH",
                "transformation": "identity"
            },
            {
                "name": "Electric Conductivity (μS/cm)",
                "unit": "μS/cm",
                "transformation": "log1p"
            },
            {
                "name": "Chloride (mg/L)",
                "unit": "mg/L",
                "transformation": "log1p"
            },
            {
                "name": "Bicarbonate (mg/L)",
                "unit": "mg/L",
                "transformation": "log1p"
            },
            {
                "name": "Sulphate (mg/L)",
                "unit": "mg/L",
                "transformation": "log1p",
                "conditional_screening": True
            },
            {
                "name": "Calcium (mg/L)",
                "unit": "mg/L",
                "transformation": "log1p"
            },
            {
                "name": "Magnesium (mg/L)",
                "unit": "mg/L",
                "transformation": "log1p"
            },
            {
                "name": "Sodium (mg/L)",
                "unit": "mg/L",
                "transformation": "log1p"
            }
        ],
        "prediction_notice": (
            "Predictions are estimates, not laboratory measurements "
            "or certifications of groundwater safety."
        )
    }
'''

(BACKEND_ROOT / "main.py").write_text(
    main_source.strip() + "\n",
    encoding="utf-8"
)

print("FASTAPI APPLICATION CREATED")
print(BACKEND_ROOT / "main.py")

FASTAPI APPLICATION CREATED
c:\career\projects\BoreWell Sentinel India\app\backend\main.py


In [38]:
# CELL 38 — Backend integration smoke test

import sys
from fastapi.testclient import TestClient

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

from backend.main import app

with TestClient(app) as client:

    endpoints = [
        "/",
        "/health",
        "/api/v1/dashboard/summary",
        "/api/v1/dashboard/states",
        "/api/v1/sites?limit=3",
        "/api/v1/alerts/trace?limit=3",
        "/api/v1/metadata/targets"
    ]

    api_results = []

    for endpoint in endpoints:
        response = client.get(endpoint)

        api_results.append({
            "Endpoint": endpoint,
            "HTTP_status": response.status_code,
            "Success": response.status_code == 200
        })

        print(
            endpoint,
            "->",
            response.status_code
        )

        if response.status_code != 200:
            print("Error:", response.text[:1000])

    api_test_summary = pd.DataFrame(api_results)

    print("\nAPI TEST SUMMARY")
    display(api_test_summary)

    assert api_test_summary["Success"].all()

    summary_response = client.get(
        "/api/v1/dashboard/summary"
    ).json()

    print("\nDASHBOARD SUMMARY")
    display(pd.DataFrame([summary_response]))

    first_site = client.get(
        "/api/v1/sites?limit=1"
    ).json()["sites"][0]

    site_id = first_site["_spatial_group_id"]

    detail_response = client.get(
        f"/api/v1/sites/{site_id}"
    )

    assert detail_response.status_code == 200

    details = detail_response.json()

    print("\nLOCATION DETAIL CHECK")
    print("Location:", details["site"]["_spatial_group_id"])
    print("Parameter recommendations:", len(details["parameter_actions"]))

    assert len(details["parameter_actions"]) == 8

print("\nPASS: Backend data API smoke tests completed.")

/ -> 200
/health -> 200
/api/v1/dashboard/summary -> 200
/api/v1/dashboard/states -> 200
/api/v1/sites?limit=3 -> 200
/api/v1/alerts/trace?limit=3 -> 200
/api/v1/metadata/targets -> 200

API TEST SUMMARY


,Endpoint,HTTP_status,Success
0,/,200,True
1,/health,200,True
2,/api/v1/dashboard/summary,200,True
3,/api/v1/dashboard/states,200,True
4,/api/v1/sites?limit=3,200,True
5,/api/v1/alerts/trace?limit=3,200,True
6,/api/v1/metadata/targets,200,True



DASHBOARD SUMMARY


,total_locations,priority_tiers,trace_alert_locations,low_chemical_evidence_locations,mean_testing_priority_score,mean_recommended_tests,total_parameter_action_records
0,24199,"{'P1': 2421, 'P2': 3629, 'P3': 18149}",150,795,26.301,7.048,193592



LOCATION DETAIL CHECK
Location: gujarat|kachchh|lilpur|site|23.51111|70.65
Parameter recommendations: 8

PASS: Backend data API smoke tests completed.


In [39]:
# CELL 39 — Create React frontend configuration

import json

FRONTEND_ROOT = PROJECT_ROOT / "app" / "frontend"
SRC_DIR = FRONTEND_ROOT / "src"

package_json = {
    "name": "borewell-sentinel-frontend",
    "private": True,
    "version": "0.1.0",
    "type": "module",
    "scripts": {
        "dev": "vite --host 127.0.0.1",
        "build": "vite build",
        "preview": "vite preview --host 127.0.0.1"
    },
    "dependencies": {
        "react": "latest",
        "react-dom": "latest",
        "recharts": "latest",
        "lucide-react": "latest"
    },
    "devDependencies": {
        "vite": "latest",
        "@vitejs/plugin-react": "latest"
    }
}

(FRONTEND_ROOT / "package.json").write_text(
    json.dumps(package_json, indent=2),
    encoding="utf-8"
)

vite_config = '''
import { defineConfig } from "vite";
import react from "@vitejs/plugin-react";

export default defineConfig({
  plugins: [react()],
  server: {
    host: "127.0.0.1",
    port: 5173,
    strictPort: true
  }
});
'''

(FRONTEND_ROOT / "vite.config.js").write_text(
    vite_config.strip() + "\n",
    encoding="utf-8"
)

index_html = '''
<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <meta name="theme-color" content="#101d2b" />
    <meta
      name="description"
      content="BoreWell Sentinel India — Groundwater intelligence and testing prioritization."
    />
    <title>BoreWell Sentinel India</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>
'''

(FRONTEND_ROOT / "index.html").write_text(
    index_html.strip() + "\n",
    encoding="utf-8"
)

main_jsx = '''
import React from "react";
import ReactDOM from "react-dom/client";
import App from "./App";
import "./styles.css";

ReactDOM.createRoot(document.getElementById("root")).render(
  <React.StrictMode>
    <App />
  </React.StrictMode>
);
'''

(SRC_DIR / "main.jsx").write_text(
    main_jsx.strip() + "\n",
    encoding="utf-8"
)

print("FRONTEND CONFIGURATION CREATED")
print("Root:", FRONTEND_ROOT)
print("Files: package.json, vite.config.js, index.html, src/main.jsx")

FRONTEND CONFIGURATION CREATED
Root: c:\career\projects\BoreWell Sentinel India\app\frontend
Files: package.json, vite.config.js, index.html, src/main.jsx


In [42]:
# CELL 41 — Create the missing frontend stylesheet

styles = r"""
* { box-sizing: border-box; }

:root {
  font-family: Inter, "Segoe UI", sans-serif;
  color: #1c2935;
  background: #f4f6f8;
}

body { margin: 0; min-width: 320px; }
button, input, select { font: inherit; }
button { cursor: pointer; }

.app-shell { display: flex; min-height: 100vh; }

.sidebar {
  width: 248px;
  flex-shrink: 0;
  background: #101d2b;
  color: white;
  padding: 25px 17px;
  display: flex;
  flex-direction: column;
}

.brand { display: flex; align-items: center; gap: 12px; padding: 0 8px 38px; }
.brand-mark { background: #244454; color: #91d9d0; padding: 12px; border-radius: 11px; }
.brand-name { font-size: 14px; font-weight: 800; letter-spacing: 1px; }
.brand-subtitle { font-size: 9px; letter-spacing: 2px; color: #91a8b5; margin-top: 4px; }

.nav-caption, .eyebrow { font-size: 9px; letter-spacing: 1.5px; font-weight: 700; color: #8b9ba5; }
.nav-caption { padding: 0 10px 13px; }

.main-nav { display: grid; gap: 7px; }
.nav-item {
  display: flex; align-items: center; gap: 12px;
  border: 0; border-radius: 8px; padding: 13px;
  background: transparent; color: #aab9c2; text-align: left;
}
.nav-item.active, .nav-item:hover { background: #244454; color: #a5e0d7; }
.nav-count { margin-left: auto; font-size: 10px; }

.sidebar-bottom { margin-top: auto; }
.data-card { background: #192c3b; border: 1px solid #344957; padding: 15px; border-radius: 10px; margin-bottom: 20px; }
.data-card-icon { color: #8bd0c9; }
.data-card-title { font-size: 11px; margin-top: 12px; }
.data-card-value { font-size: 19px; font-weight: 700; margin-top: 7px; }
.data-card-note, .sidebar-version { color: #91a2ad; font-size: 10px; margin-top: 6px; }
.sidebar-version { text-align: center; font-size: 8px; letter-spacing: 1px; }

.main-area { flex: 1; min-width: 0; }
.topbar {
  min-height: 78px; background: white; border-bottom: 1px solid #e9edf0;
  display: flex; justify-content: space-between; align-items: center;
  padding: 15px 32px;
}
.breadcrumb { font-size: 9px; color: #8a969e; letter-spacing: 1px; font-weight: 700; }
.breadcrumb span { padding: 0 5px; }
.topbar-title { font-size: 18px; font-weight: 700; margin-top: 5px; }
.topbar-actions { display: flex; align-items: center; gap: 12px; }
.api-pill { border: 1px solid #e7edef; padding: 7px 11px; border-radius: 20px; font-size: 10px; text-transform: uppercase; }
.status-dot, .live-dot { display: inline-block; width: 7px; height: 7px; border-radius: 50%; background: #d19a45; margin-right: 6px; }
.api-pill.online .status-dot, .live-dot { background: #42a788; }
.api-pill.offline .status-dot { background: #d65c5c; }

.page-content { max-width: 1500px; margin: auto; padding: 28px 32px; }

.hero-panel {
  background: linear-gradient(110deg, #142e3d, #28596a);
  color: white; border-radius: 15px; padding: 34px;
  min-height: 230px; display: flex; align-items: center;
  justify-content: space-between; overflow: hidden;
}
.hero-kicker { color: #a5d8d2; font-size: 9px; letter-spacing: 1.5px; }
.hero-panel h1 { font-size: 35px; line-height: 1.2; margin: 18px 0 10px; }
.hero-panel h1 span { color: #91d7ce; }
.hero-panel p { color: #c3d1d8; font-size: 12px; line-height: 1.7; max-width: 500px; }
.hero-button { display: flex; align-items: center; gap: 9px; margin-top: 18px; padding: 11px 16px; border: 0; border-radius: 7px; background: #a2ddd3; color: #173b46; font-weight: 700; font-size: 11px; }

.stat-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin: 18px 0; }
.stat-card, .panel { background: white; border: 1px solid #e8edf0; border-radius: 11px; }
.stat-card { padding: 16px; }
.stat-icon { display: inline-grid; place-items: center; width: 36px; height: 36px; border-radius: 9px; margin-bottom: 13px; }
.stat-icon.blue { color: #477d96; background: #eaf2f7; }
.stat-icon.red { color: #c76060; background: #fbefef; }
.stat-icon.amber { color: #bd8b43; background: #fbf4e9; }
.stat-icon.green { color: #46977e; background: #eaf5f0; }
.stat-label { font-size: 9px; color: #89959d; font-weight: 700; letter-spacing: 1px; }
.stat-value { font-size: 25px; font-weight: 700; margin-top: 7px; }
.stat-detail { font-size: 10px; color: #89959d; margin-top: 5px; }

.chart-grid { display: grid; grid-template-columns: 1fr 1.15fr; gap: 14px; margin-bottom: 15px; }
.chart-panel, .table-panel { padding: 18px; }
.panel-heading { display: flex; justify-content: space-between; align-items: flex-start; }
.panel-heading h3 { font-size: 15px; margin: 6px 0; }
.chart-area { height: 205px; margin-top: 14px; }
.chart-footnote { border-top: 1px solid #eee; padding-top: 10px; font-size: 9px; color: #98a2a9; }

.section-title { display: flex; justify-content: space-between; margin-bottom: 20px; }
.section-title h2 { font-size: 24px; margin: 7px 0; }
.section-title p { color: #7d8991; font-size: 11px; }

.filter-panel { display: flex; gap: 9px; padding: 12px; background: white; border: 1px solid #e9edf0; border-radius: 10px; }
.search-field { display: flex; align-items: center; gap: 9px; flex: 1; padding: 0 10px; background: #f7f9fa; border-radius: 7px; }
.search-field input { width: 100%; background: transparent; border: 0; outline: 0; padding: 10px 0; font-size: 11px; }
.filter-panel select { border: 1px solid #e5eaed; border-radius: 7px; padding: 8px; font-size: 10px; background: white; }

.primary-button, .secondary-button { display: inline-flex; align-items: center; gap: 7px; padding: 10px 13px; border-radius: 7px; font-size: 10px; font-weight: 700; }
.primary-button { background: #234f62; border: 0; color: white; }
.secondary-button { background: white; border: 1px solid #e5eaed; color: #52616b; }
.secondary-button:disabled { opacity: .4; cursor: not-allowed; }

.results-line { display: flex; justify-content: space-between; padding: 14px 2px 9px; color: #87939b; font-size: 10px; }
.table-wrap { overflow-x: auto; margin-top: 14px; }
table { width: 100%; border-collapse: collapse; text-align: left; white-space: nowrap; }
th { color: #909ba3; font-size: 8px; letter-spacing: 1px; padding: 11px; border-bottom: 1px solid #edf0f2; }
td { color: #52616b; font-size: 10px; padding: 11px; border-bottom: 1px solid #f0f2f4; }
tbody tr:hover { background: #f8fafb; }
.location-name { font-weight: 700; color: #253541; font-size: 11px; }
.sub-cell { color: #929da5; font-size: 9px; margin-top: 4px; max-width: 320px; overflow: hidden; text-overflow: ellipsis; }
.score-cell, .rank-cell { font-weight: 700; }
.tier-badge { padding: 4px 7px; border-radius: 5px; font-size: 9px; font-weight: 700; }
.tier-badge.red { background: #fbeeee; color: #b85151; }
.tier-badge.amber { background: #fbf3e6; color: #a87930; }
.tier-badge.green { background: #eaf5ef; color: #39866d; }

.icon-button { display: inline-grid; place-items: center; width: 36px; height: 36px; border: 1px solid #e7edef; border-radius: 8px; background: white; color: #657580; }
.icon-button.small { width: 29px; height: 29px; }
.text-action { display: inline-flex; align-items: center; gap: 6px; color: #477e91; border: 0; background: transparent; font-size: 10px; font-weight: 700; }

.pagination { display: flex; justify-content: space-between; align-items: center; margin-top: 13px; padding-top: 13px; border-top: 1px solid #eee; font-size: 10px; color: #839099; }

.detail-panel { background: white; border: 1px solid #dfe8eb; padding: 19px; border-radius: 12px; margin-top: 17px; }
.detail-heading { display: flex; justify-content: space-between; }
.detail-heading h3 { font-size: 17px; }
.detail-id { overflow-wrap: anywhere; }
.detail-stats, .action-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 9px; margin-top: 15px; }
.detail-stats > div, .action-grid > div { background: #f6f8f9; padding: 11px; border-radius: 8px; }
.detail-stats span, .action-grid span { display: block; color: #89959c; font-size: 9px; }
.detail-stats strong, .action-grid strong { display: block; margin-top: 6px; font-size: 17px; }
.detail-section { margin-top: 20px; }
.detail-section h4 { font-size: 12px; }
.recommendation-row { display: flex; justify-content: space-between; gap: 12px; padding: 11px 3px; border-bottom: 1px solid #eee; }
.recommendation-name { font-size: 10px; font-weight: 700; }
.recommendation-type { font-size: 9px; color: #477e91; font-weight: 700; }
.recommendation-row .sub-cell { white-space: normal; max-width: 550px; }
.notice { display: flex; gap: 9px; padding: 12px; margin-top: 17px; border-radius: 8px; background: #f0f6f8; color: #4a6d7c; font-size: 10px; line-height: 1.6; }
.notice-amber { background: #fbf6ec; color: #94713a; }
.trace-parameters { white-space: normal; max-width: 380px; }
.empty-state, .loading-state { min-height: 150px; display: flex; justify-content: center; align-items: center; gap: 10px; color: #85939b; font-size: 11px; }
.loading-state { height: 55vh; }
.loading-state.compact { height: auto; min-height: 60px; }
.error-banner { margin: 15px 32px 0; padding: 12px; background: #fff0ef; color: #a64f4d; border-radius: 8px; font-size: 11px; display: flex; gap: 9px; }
.app-footer { display: flex; justify-content: space-between; padding: 14px 32px 22px; color: #a0aab1; font-size: 8px; }
.app-footer span:first-child { font-weight: 700; letter-spacing: 1px; }

@media (max-width: 1000px) {
  .stat-grid { grid-template-columns: repeat(2, 1fr); }
  .sidebar { width: 210px; }
  .chart-grid { grid-template-columns: 1fr; }
}

@media (max-width: 700px) {
  .sidebar { width: 62px; padding: 16px 8px; }
  .brand { padding: 0 0 28px; }
  .brand > div:last-child, .nav-caption, .nav-item span, .sidebar-bottom { display: none; }
  .nav-item { justify-content: center; padding: 12px 0; }
  .topbar { padding: 12px; }
  .page-content { padding: 16px 12px; }
  .hero-panel { padding: 24px 18px; }
  .hero-panel h1 { font-size: 26px; }
  .filter-panel { flex-wrap: wrap; }
  .search-field { flex-basis: 100%; min-height: 38px; }
  .filter-panel select { flex: 1; min-width: 0; }
  .detail-stats, .action-grid { grid-template-columns: repeat(2, 1fr); }
  .app-footer { padding: 12px; flex-direction: column; gap: 6px; }
}
"""

style_path = SRC_DIR / "styles.css"
style_path.write_text(styles.strip() + "\n", encoding="utf-8")

print("STYLESHEET CREATED:", style_path)
print("File exists:", style_path.exists())
print("Size:", round(style_path.stat().st_size / 1024, 2), "KB")

STYLESHEET CREATED: c:\career\projects\BoreWell Sentinel India\app\frontend\src\styles.css
File exists: True
Size: 10.0 KB


In [45]:
# CELL 42 — Rebuild React frontend using the resolved npm executable

import subprocess
import shutil
from pathlib import Path

FRONTEND_ROOT = (
    PROJECT_ROOT / "app" / "frontend"
).resolve()

# Verify frontend files exist
assert FRONTEND_ROOT.exists(), f"Frontend folder missing: {FRONTEND_ROOT}"
assert (FRONTEND_ROOT / "package.json").exists(), "package.json missing"
assert (FRONTEND_ROOT / "src" / "styles.css").exists(), "styles.css missing"
assert (FRONTEND_ROOT / "src" / "App.jsx").exists(), "App.jsx missing"

# Resolve npm using Windows PATH/PATHEXT
npm_executable = (
    shutil.which("npm")
    or shutil.which("npm.cmd")
)

if not npm_executable:
    raise FileNotFoundError(
        "npm was not found. Check your Node.js installation and PATH."
    )

print("Frontend directory:", FRONTEND_ROOT)
print("npm executable:", npm_executable)
print("\nBUILDING FRONTEND...")

result = subprocess.run(
    [npm_executable, "run", "build"],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)

if result.returncode != 0:
    print("\nBUILD ERROR:")
    print(result.stderr[-8000:])
    raise RuntimeError("Frontend build failed.")

print("\nFRONTEND BUILD PASSED")
print("Build output:", FRONTEND_ROOT / "dist")

Frontend directory: C:\career\projects\BoreWell Sentinel India\app\frontend
npm executable: C:\Program Files\nodejs\npm.CMD

BUILDING FRONTEND...

> borewell-sentinel-frontend@0.1.0 build
> vite build

vite v8.3.8 building client environment for production...
transforming...
âœ“ 2529 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                   0.61 kB â”‚ gzip:   0.36 kB
dist/assets/index-Dg4hFwtd.css    8.59 kB â”‚ gzip:   2.46 kB
dist/assets/index-CmQTbKjy.js   658.75 kB â”‚ gzip: 193.50 kB

âœ“ built in 751ms


FRONTEND BUILD PASSED
Build output: C:\career\projects\BoreWell Sentinel India\app\frontend\dist


In [46]:
# CELL 43 — Add geographic map endpoint to FastAPI

from pathlib import Path

main_path = BACKEND_ROOT / "main.py"

map_endpoint_source = r'''

@app.get("/api/v1/map/sites")
def map_sites(
    state: str | None = Query(default=None, max_length=100),
    tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
):
    import pandas as pd

    sites = get_data()["sites"]
    filtered = sites.copy()

    if state:
        filtered = filtered.loc[
            filtered["State"].astype(str).str.casefold()
            == state.strip().casefold()
        ]

    if tier:
        filtered = filtered.loc[
            filtered["Priority_tier"]
            .fillna("")
            .astype(str)
            .str.startswith(tier.upper())
        ]

    total_filtered = len(filtered)

    latitude = pd.to_numeric(
        filtered["Latitude"], errors="coerce"
    )

    longitude = pd.to_numeric(
        filtered["Longitude"], errors="coerce"
    )

    valid_coordinates = (
        latitude.between(-90, 90)
        & longitude.between(-180, 180)
    )

    geo = filtered.loc[valid_coordinates].copy()

    geo["Latitude"] = latitude.loc[valid_coordinates]
    geo["Longitude"] = longitude.loc[valid_coordinates]

    map_columns = [
        "_spatial_group_id",
        "State",
        "District",
        "Latitude",
        "Longitude",
        "Priority_rank",
        "Priority_tier",
        "Testing_priority_score",
        "Primary_priority_driver",
        "Total_records",
        "Distinct_years"
    ]

    geo = geo[map_columns].sort_values(
        "Priority_rank",
        ascending=True
    )

    return {
        "filtered_locations": int(total_filtered),
        "mapped_locations": int(len(geo)),
        "excluded_invalid_coordinates": int(
            total_filtered - len(geo)
        ),
        "points": records_from_frame(geo)
    }
'''

current_source = main_path.read_text(encoding="utf-8")

if '"/api/v1/map/sites"' not in current_source:
    with main_path.open("a", encoding="utf-8") as file:
        file.write(map_endpoint_source)

print("MAP ENDPOINT ADDED:", main_path)

# Reload the application so the new route is registered.
import importlib
import backend.main as backend_main

backend_main = importlib.reload(backend_main)

from fastapi.testclient import TestClient

with TestClient(backend_main.app) as client:
    response = client.get("/api/v1/map/sites")

    print("\nHTTP STATUS:", response.status_code)

    response.raise_for_status()
    map_data = response.json()

    print("Filtered locations:", map_data["filtered_locations"])
    print("Mapped locations:", map_data["mapped_locations"])
    print("Invalid/missing coordinates:",
          map_data["excluded_invalid_coordinates"])

    print("\nFIRST 3 MAP POINTS")
    display(pd.DataFrame(map_data["points"][:3]))

    assert map_data["filtered_locations"] == 24_199
    assert (
        map_data["mapped_locations"]
        + map_data["excluded_invalid_coordinates"]
        == 24_199
    )
    assert len(map_data["points"]) == map_data["mapped_locations"]

print("\nPASS: Geographic API endpoint is working.")

MAP ENDPOINT ADDED: c:\career\projects\BoreWell Sentinel India\app\backend\main.py

HTTP STATUS: 200
Filtered locations: 24199
Mapped locations: 24172
Invalid/missing coordinates: 27

FIRST 3 MAP POINTS


,_spatial_group_id,State,District,Latitude,Longitude,Priority_rank,Priority_tier,Testing_priority_score,Primary_priority_driver,Total_records,Distinct_years
0,gujarat|kachchh|lilpur|site|23.51111|70.65,gujarat,kachchh,23.511111,70.650,1,P1 — Highest 10%,44.267053,Chemical,16,16
1,odisha|malkangiri|m.v.79|site|17.985|81.595,odisha,malkangiri,17.985000,81.595,2,P1 — Highest 10%,44.229041,Monitoring_gap,1,1
2,maharashtra|jalgaon|sadavan bk.|site|20.85278|...,maharashtra,jalgaon,20.852778,75.100,3,P1 — Highest 10%,43.369090,Chemical,9,9



PASS: Geographic API endpoint is working.


In [47]:
# CELL 44 — Install geographic map dependencies

import subprocess
import shutil

npm_executable = shutil.which("npm") or shutil.which("npm.cmd")

if not npm_executable:
    raise FileNotFoundError("npm not found.")

result = subprocess.run(
    [
        npm_executable,
        "install",
        "leaflet@1.9.4",
        "leaflet.markercluster@1.5.3"
    ],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr[-5000:])
    raise RuntimeError("Leaflet installation failed.")

print("\nPASS: Leaflet and marker clustering installed.")


added 2 packages, and audited 62 packages in 3s

9 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


PASS: Leaflet and marker clustering installed.


In [48]:
# CELL 45 — Install React Leaflet

import subprocess
import shutil

npm_executable = shutil.which("npm") or shutil.which("npm.cmd")

result = subprocess.run(
    [npm_executable, "install", "react-leaflet@^5.0.0"],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr[-6000:])
    raise RuntimeError("React Leaflet installation failed.")

print("\nINSTALLED VERSIONS")

for package in ["react", "react-dom", "leaflet", "react-leaflet", "leaflet.markercluster"]:
    package_json = json.loads(
        (FRONTEND_ROOT / "node_modules" / package / "package.json")
        .read_text(encoding="utf-8")
    )
    print(package, package_json["version"])


added 2 packages, and audited 64 packages in 3s

9 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


INSTALLED VERSIONS
react 19.3.1
react-dom 19.3.1
leaflet 1.9.4
react-leaflet 5.0.0
leaflet.markercluster 1.5.3


In [49]:
# CELL 46 — Coordinate sanity audit for India map

import pandas as pd

map_points_df = pd.DataFrame(map_data["points"])

# Broad sanity envelope covering mainland India and island territories.
INDIA_BOUNDS = {
    "min_latitude": 5.0,
    "max_latitude": 38.0,
    "min_longitude": 67.0,
    "max_longitude": 99.0
}

inside_india = (
    map_points_df["Latitude"].between(
        INDIA_BOUNDS["min_latitude"],
        INDIA_BOUNDS["max_latitude"]
    )
    & map_points_df["Longitude"].between(
        INDIA_BOUNDS["min_longitude"],
        INDIA_BOUNDS["max_longitude"]
    )
)

map_points_df["Inside_India_Map_Envelope"] = inside_india

inside_count = int(inside_india.sum())
outside_count = int((~inside_india).sum())

print("COORDINATE QUALITY AUDIT")
print("Valid coordinate records:", len(map_points_df))
print("Inside broad India envelope:", inside_count)
print("Outside envelope — review:", outside_count)
print("\nCoordinate ranges")
display(
    map_points_df[["Latitude", "Longitude"]]
    .describe()
    .round(5)
)

if outside_count:
    print("\nOUTSIDE-ENVELOPE RECORDS — REVIEW ONLY")
    display(
        map_points_df.loc[
            ~inside_india,
            [
                "_spatial_group_id",
                "State",
                "District",
                "Latitude",
                "Longitude"
            ]
        ].head(25)
    )

print("\nNo source records or coordinates have been modified.")

COORDINATE QUALITY AUDIT
Valid coordinate records: 24172
Inside broad India envelope: 24172
Outside envelope — review: 0

Coordinate ranges


,Latitude,Longitude
count,24172.00000,24172.00000
mean,21.38361,79.58023
std,5.77823,4.87599
min,8.08333,68.53333
25%,17.36667,76.04167
50%,22.04422,78.36486
75%,25.79333,82.97611
max,34.65028,96.13333



No source records or coordinates have been modified.


In [50]:
# CELL 47 — Create interactive groundwater map component

map_component = r'''
import React, { useCallback, useEffect, useMemo, useRef, useState } from "react";
import L from "leaflet";
import "leaflet.markercluster";
import {
  MapContainer, TileLayer, useMap
} from "react-leaflet";

import {
  AlertTriangle, Crosshair, Layers3, LoaderCircle,
  MapPin, RefreshCw, X
} from "lucide-react";

import "leaflet/dist/leaflet.css";
import "leaflet.markercluster/dist/MarkerCluster.css";
import "leaflet.markercluster/dist/MarkerCluster.Default.css";

const API = import.meta.env.VITE_API_BASE_URL || "http://127.0.0.1:8000";

const number = (value) =>
  Number(value || 0).toLocaleString("en-IN");

const tierClass = (tier) => {
  if (String(tier).startsWith("P1")) return "p1";
  if (String(tier).startsWith("P2")) return "p2";
  return "p3";
};

const escapeHTML = (value) =>
  String(value ?? "").replace(/[&<>"']/g, (char) => ({
    "&": "&amp;",
    "<": "&lt;",
    ">": "&gt;",
    '"': "&quot;",
    "'": "&#39;"
  })[char]);

function MapMarkers({ points, onSelect }) {
  const map = useMap();
  const selectRef = useRef(onSelect);

  useEffect(() => {
    selectRef.current = onSelect;
  }, [onSelect]);

  useEffect(() => {
    if (!points?.length) return;

    const clusters = L.markerClusterGroup({
      chunkedLoading: true,
      chunkInterval: 150,
      chunkDelay: 35,
      maxClusterRadius: 55,
      spiderfyOnMaxZoom: true,
      showCoverageOnHover: false,
      zoomToBoundsOnClick: true,
      animate: false,
      removeOutsideVisibleBounds: true,

      iconCreateFunction(cluster) {
        const count = cluster.getChildCount();
        const size = count < 10 ? "small" : count < 100 ? "medium" : "large";

        return L.divIcon({
          html: `<div class="cluster-bubble ${size}"><span>${number(count)}</span></div>`,
          className: "cluster-icon-wrapper",
          iconSize: [42, 42],
          iconAnchor: [21, 21]
        });
      }
    });

    for (const point of points) {
      const lat = Number(point.Latitude);
      const lng = Number(point.Longitude);

      if (
        !Number.isFinite(lat) ||
        !Number.isFinite(lng) ||
        lat < -90 || lat > 90 ||
        lng < -180 || lng > 180
      ) {
        continue;
      }

      const tier = tierClass(point.Priority_tier);

      const icon = L.divIcon({
        html: '<span class="site-marker-core"></span>',
        className: `site-marker-icon ${tier}`,
        iconSize: [16, 16],
        iconAnchor: [8, 8]
      });

      const marker = L.marker([lat, lng], {
        icon,
        title: `${point.District}, ${point.State}`
      });

      const popup = `
        <div class="map-popup">
          <div class="map-popup-title">
            ${escapeHTML(point.District)}
          </div>
          <div class="map-popup-subtitle">
            ${escapeHTML(point.State)}
          </div>
          <div class="map-popup-grid">
            <span>Priority rank</span>
            <strong>#${number(point.Priority_rank)}</strong>
            <span>Testing score</span>
            <strong>${Number(point.Testing_priority_score || 0).toFixed(2)}</strong>
            <span>Tier</span>
            <strong>${escapeHTML(String(point.Priority_tier || "").split(" — ")[0])}</strong>
          </div>
          <div class="map-popup-hint">Click marker to open the site profile</div>
        </div>
      `;

      marker.bindPopup(popup, {
        maxWidth: 250,
        closeButton: true
      });

      marker.on("click", () => {
        selectRef.current(point._spatial_group_id);
      });

      clusters.addLayer(marker);
    }

    map.addLayer(clusters);

    return () => {
      map.removeLayer(clusters);
      clusters.clearLayers();
    };
  }, [map, points]);

  return null;
}

function MapLegend() {
  return (
    <div className="map-legend">
      <div className="map-legend-title">TESTING PRIORITY</div>
      <div><span className="legend-dot p1" /> P1 — Highest 10%</div>
      <div><span className="legend-dot p2" /> P2 — Next 15%</div>
      <div><span className="legend-dot p3" /> P3 — Remaining</div>
    </div>
  );
}

function SiteProfile({ detail, loading, onClose }) {
  if (loading) {
    return (
      <div className="map-profile-loading">
        <LoaderCircle className="map-spin" size={19} />
        Loading location profile...
      </div>
    );
  }

  if (!detail) return null;

  const site = detail.site;
  const actions = detail.actions;

  return (
    <div className="map-profile">
      <div className="map-profile-heading">
        <div>
          <div className="map-section-eyebrow">SELECTED LOCATION</div>
          <h3>{site.District}, {site.State}</h3>
          <div className="map-profile-id">{site._spatial_group_id}</div>
        </div>
        <button className="icon-button" onClick={onClose} title="Close profile">
          <X size={17} />
        </button>
      </div>

      <div className="map-profile-metrics">
        <div>
          <span>Priority rank</span>
          <strong>#{number(site.Priority_rank)}</strong>
        </div>
        <div>
          <span>Priority score</span>
          <strong>{Number(site.Testing_priority_score || 0).toFixed(2)}</strong>
        </div>
        <div>
          <span>Historical records</span>
          <strong>{number(site.Total_records)}</strong>
        </div>
        <div>
          <span>Distinct years</span>
          <strong>{number(site.Distinct_years)}</strong>
        </div>
      </div>

      <div className="map-profile-subheading">TESTING RECOMMENDATIONS</div>

      <div className="map-action-grid">
        <div><span>Baseline</span><strong>{number(actions?.Baseline_test_count)}</strong></div>
        <div><span>Targeted retest</span><strong>{number(actions?.Targeted_retest_count)}</strong></div>
        <div><span>Overdue</span><strong>{number(actions?.Overdue_retest_count)}</strong></div>
        <div><span>Routine due</span><strong>{number(actions?.Routine_retest_due_count)}</strong></div>
      </div>

      <div className="map-profile-subheading">PARAMETER ACTIONS</div>

      <div className="map-parameter-list">
        {(detail.parameter_actions || []).map((item, index) => (
          <div className="map-parameter-row" key={`${item.Parameter}-${index}`}>
            <span>{item.Parameter}</span>
            <strong>{item.Testing_recommendation}</strong>
          </div>
        ))}
      </div>

      <div className="map-profile-warning">
        <AlertTriangle size={15} />
        Testing priority is not a water-safety classification. Laboratory testing is required.
      </div>
    </div>
  );
}

export default function MapExplorer() {
  const [mapData, setMapData] = useState(null);
  const [states, setStates] = useState([]);
  const [state, setState] = useState("");
  const [tier, setTier] = useState("ALL");

  const [loading, setLoading] = useState(true);
  const [error, setError] = useState("");

  const [selectedId, setSelectedId] = useState(null);
  const [detail, setDetail] = useState(null);
  const [detailLoading, setDetailLoading] = useState(false);

  const loadMap = useCallback(async () => {
    setLoading(true);
    setError("");

    try {
      const params = new URLSearchParams();

      if (state) params.set("state", state);
      if (tier !== "ALL") params.set("tier", tier);

      const suffix = params.toString()
        ? `?${params.toString()}`
        : "";

      const [mapResponse, statesResponse] = await Promise.all([
        fetch(`${API}/api/v1/map/sites${suffix}`),
        fetch(`${API}/api/v1/dashboard/states?limit=40`)
      ]);

      if (!mapResponse.ok) {
        throw new Error(`Map endpoint returned ${mapResponse.status}`);
      }

      if (!statesResponse.ok) {
        throw new Error(`State endpoint returned ${statesResponse.status}`);
      }

      const mapJson = await mapResponse.json();
      const statesJson = await statesResponse.json();

      setMapData(mapJson);
      setStates((statesJson.states || []).map((item) => item.State).sort());

      setSelectedId(null);
      setDetail(null);
    } catch (err) {
      console.error(err);
      setError(err.message || "Unable to load map data.");
    } finally {
      setLoading(false);
    }
  }, [state, tier]);

  useEffect(() => {
    loadMap();
  }, [loadMap]);

  const openSite = useCallback(async (siteId) => {
    setSelectedId(siteId);
    setDetail(null);
    setDetailLoading(true);

    try {
      const response = await fetch(
        `${API}/api/v1/sites/${encodeURIComponent(siteId)}`
      );

      if (!response.ok) {
        throw new Error("Could not retrieve this location profile.");
      }

      setDetail(await response.json());
    } catch (err) {
      console.error(err);
      setError(err.message);
    } finally {
      setDetailLoading(false);
    }
  }, []);

  const summaryLabel = useMemo(() => {
    if (!mapData) return "Loading map data";

    return `${number(mapData.mapped_locations)} mapped locations`;
  }, [mapData]);

  return (
    <div className="page-content map-page">
      <div className="section-title">
        <div>
          <div className="eyebrow">GEOGRAPHIC INTELLIGENCE</div>
          <h2>India groundwater map</h2>
          <p>
            Explore provisional spatial groups and their testing-priority distribution.
          </p>
        </div>

        <button
          className="secondary-button map-refresh"
          onClick={loadMap}
          disabled={loading}
        >
          <RefreshCw size={15} className={loading ? "map-spin" : ""} />
          Refresh
        </button>
      </div>

      <div className="map-warning">
        <AlertTriangle size={16} />
        Coordinates represent provisional source locations. Administrative boundaries
        and station identities have not been independently verified for every point.
      </div>

      <div className="map-filters">
        <label>
          <span>STATE</span>
          <select
            value={state}
            onChange={(event) => setState(event.target.value)}
          >
            <option value="">All states</option>
            {states.map((item) => (
              <option value={item} key={item}>{item}</option>
            ))}
          </select>
        </label>

        <label>
          <span>PRIORITY TIER</span>
          <select
            value={tier}
            onChange={(event) => setTier(event.target.value)}
          >
            <option value="ALL">All tiers</option>
            <option value="P1">P1 — Highest 10%</option>
            <option value="P2">P2 — Next 15%</option>
            <option value="P3">P3 — Remaining</option>
          </select>
        </label>

        <div className="map-filter-summary">
          <Layers3 size={16} />
          {summaryLabel}
        </div>
      </div>

      {error && (
        <div className="error-banner">
          <AlertTriangle size={16} />
          {error}
        </div>
      )}

      <div className="map-status-row">
        <span>
          {number(mapData?.filtered_locations)} locations in current filter
        </span>
        <span>
          {number(mapData?.excluded_invalid_coordinates)} excluded for missing or invalid coordinates
        </span>
      </div>

      <div className="map-container">
        <MapContainer
          center={[22.0, 79.0]}
          zoom={5}
          minZoom={4}
          maxZoom={17}
          scrollWheelZoom={true}
          maxBounds={[[5, 66], [38, 100]]}
          maxBoundsViscosity={0.8}
          preferCanvas={true}
        >
          <TileLayer
            attribution='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
            url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png"
            maxZoom={19}
          />

          {mapData?.points && (
            <MapMarkers
              points={mapData.points}
              onSelect={openSite}
            />
          )}
        </MapContainer>

        <MapLegend />

        {loading && (
          <div className="map-loading-overlay">
            <LoaderCircle size={23} className="map-spin" />
            Preparing geographic markers...
          </div>
        )}

        {!loading && !error && !mapData?.mapped_locations && (
          <div className="map-empty-overlay">
            No geocoded locations match these filters.
          </div>
        )}
      </div>

      <div className="map-source-note">
        <span><MapPin size={13} /> Source: BoreWell Sentinel India processed location register</span>
        <span>Map tiles: OpenStreetMap contributors</span>
      </div>

      {selectedId && (
        <SiteProfile
          detail={detail}
          loading={detailLoading}
          onClose={() => {
            setSelectedId(null);
            setDetail(null);
          }}
        />
      )}
    </div>
  );
}
'''

map_component_path = SRC_DIR / "MapExplorer.jsx"

map_component_path.write_text(
    map_component.strip() + "\n",
    encoding="utf-8"
)

print("MAP COMPONENT CREATED:")
print(map_component_path)
print("Size:", round(map_component_path.stat().st_size / 1024, 2), "KB")

MAP COMPONENT CREATED:
c:\career\projects\BoreWell Sentinel India\app\frontend\src\MapExplorer.jsx
Size: 13.0 KB


In [51]:
# CELL 48 — Add map-specific styling

map_styles = r"""

/* INDIA GROUNDWATER MAP */

.map-page { padding-top: 27px; }

.map-warning {
  display: flex;
  align-items: flex-start;
  gap: 9px;
  background: #fbf6ec;
  color: #94713a;
  border: 1px solid #f0e5d2;
  border-radius: 9px;
  padding: 12px 14px;
  font-size: 10px;
  line-height: 1.6;
  margin-bottom: 14px;
}

.map-filters {
  display: flex;
  gap: 13px;
  align-items: flex-end;
  background: #fff;
  border: 1px solid #e8edf0;
  border-radius: 11px;
  padding: 13px;
  margin-bottom: 12px;
}

.map-filters label {
  display: grid;
  gap: 6px;
  min-width: 175px;
}

.map-filters label span {
  font-size: 8px;
  letter-spacing: 1.2px;
  color: #89959d;
  font-weight: 700;
}

.map-filters select {
  border: 1px solid #e5eaed;
  background: #fff;
  border-radius: 7px;
  padding: 9px 11px;
  color: #43545f;
  font-size: 10px;
  min-height: 37px;
}

.map-filter-summary {
  display: flex;
  align-items: center;
  gap: 8px;
  color: #667782;
  font-size: 10px;
  margin-left: auto;
  padding: 10px 5px;
}

.map-status-row {
  display: flex;
  justify-content: space-between;
  gap: 12px;
  padding: 3px 2px 9px;
  color: #87939b;
  font-size: 10px;
}

.map-container {
  width: 100%;
  height: 615px;
  position: relative;
  overflow: hidden;
  border: 1px solid #dfe7eb;
  border-radius: 12px;
  background: #eaf0f1;
  isolation: isolate;
}

.map-container .leaflet-container {
  width: 100%;
  height: 100%;
  background: #e9eff0;
  font-family: "DM Sans", sans-serif;
  z-index: 1;
}

.site-marker-icon {
  background: transparent;
  border: 0;
}

.site-marker-core {
  display: block;
  width: 13px;
  height: 13px;
  border-radius: 50%;
  border: 2px solid #fff;
  box-shadow: 0 1px 5px rgba(0,0,0,.4);
}

.site-marker-icon.p1 .site-marker-core { background: #d65c5c; }
.site-marker-icon.p2 .site-marker-core { background: #d69b42; }
.site-marker-icon.p3 .site-marker-core { background: #319777; }

.cluster-icon-wrapper {
  background: transparent;
  border: 0;
}

.cluster-bubble {
  height: 42px;
  width: 42px;
  border-radius: 50%;
  display: grid;
  place-items: center;
  color: #fff;
  font-size: 10px;
  font-weight: 700;
  border: 3px solid rgba(255,255,255,.85);
  box-shadow: 0 2px 7px rgba(17,37,49,.25);
}

.cluster-bubble.small { background: #398f81; }
.cluster-bubble.medium { background: #347a88; }
.cluster-bubble.large { background: #23596b; width: 48px; height: 48px; margin: -3px; }

.map-legend {
  position: absolute;
  bottom: 19px;
  right: 18px;
  z-index: 500;
  background: rgba(255,255,255,.96);
  border: 1px solid #e1e8eb;
  border-radius: 9px;
  padding: 12px 14px;
  min-width: 155px;
  box-shadow: 0 3px 12px rgba(24,45,57,.12);
  display: grid;
  gap: 9px;
  font-size: 10px;
  color: #586975;
}

.map-legend-title {
  font-size: 8px;
  font-weight: 700;
  color: #82919a;
  letter-spacing: 1.2px;
  margin-bottom: 2px;
}

.map-legend > div:not(.map-legend-title) {
  display: flex;
  align-items: center;
  gap: 8px;
}

.legend-dot {
  display: inline-block;
  width: 9px;
  height: 9px;
  border-radius: 50%;
}

.legend-dot.p1 { background: #d65c5c; }
.legend-dot.p2 { background: #d69b42; }
.legend-dot.p3 { background: #319777; }

.map-popup {
  font-family: "DM Sans", sans-serif;
  min-width: 190px;
}

.map-popup-title {
  font-size: 13px;
  font-weight: 700;
  color: #203441;
}

.map-popup-subtitle {
  color: #82909a;
  font-size: 10px;
  margin-top: 3px;
  text-transform: capitalize;
}

.map-popup-grid {
  display: grid;
  grid-template-columns: 1fr auto;
  gap: 8px;
  margin-top: 12px;
  font-size: 10px;
  color: #76848e;
}

.map-popup-grid strong { color: #2d4959; }
.map-popup-hint { font-size: 9px; color: #47816f; margin-top: 12px; }

.map-loading-overlay,
.map-empty-overlay {
  position: absolute;
  z-index: 600;
  left: 50%;
  top: 50%;
  transform: translate(-50%, -50%);
  background: rgba(255,255,255,.96);
  color: #526875;
  border: 1px solid #dfe7eb;
  box-shadow: 0 5px 22px rgba(20,40,50,.12);
  border-radius: 10px;
  padding: 14px 18px;
  display: flex;
  align-items: center;
  gap: 10px;
  font-size: 11px;
}

.map-loading-overlay { pointer-events: none; }
.map-empty-overlay { pointer-events: none; }

.map-spin { animation: map-spin 1.1s linear infinite; }

@keyframes map-spin {
  to { transform: rotate(360deg); }
}

.map-source-note {
  display: flex;
  justify-content: space-between;
  gap: 12px;
  padding: 10px 2px 3px;
  color: #919da5;
  font-size: 9px;
}

.map-source-note span {
  display: flex;
  align-items: center;
  gap: 5px;
}

.map-profile {
  background: #fff;
  border: 1px solid #dfe8eb;
  padding: 19px;
  border-radius: 12px;
  margin-top: 16px;
  box-shadow: 0 5px 20px rgba(20,45,60,.04);
}

.map-profile-heading {
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  gap: 12px;
}

.map-profile-heading h3 {
  font-size: 17px;
  margin: 6px 0;
}

.map-section-eyebrow {
  color: #84929a;
  font-size: 9px;
  font-weight: 700;
  letter-spacing: 1.3px;
}

.map-profile-id {
  color: #8b979e;
  overflow-wrap: anywhere;
  font-size: 9px;
  margin-top: 5px;
}

.map-profile-metrics {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 9px;
  margin-top: 16px;
}

.map-profile-metrics > div,
.map-action-grid > div {
  padding: 11px;
  background: #f6f8f9;
  border-radius: 8px;
}

.map-profile-metrics span,
.map-action-grid span {
  display: block;
  font-size: 9px;
  color: #89959c;
}

.map-profile-metrics strong,
.map-action-grid strong {
  display: block;
  font-size: 17px;
  margin-top: 5px;
  color: #263d4a;
}

.map-profile-subheading {
  margin: 20px 0 10px;
  font-size: 9px;
  font-weight: 700;
  color: #7c8991;
  letter-spacing: 1.1px;
}

.map-action-grid {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 8px;
}

.map-parameter-list {
  border-top: 1px solid #edf0f2;
}

.map-parameter-row {
  display: flex;
  justify-content: space-between;
  align-items: center;
  gap: 12px;
  padding: 10px 3px;
  border-bottom: 1px solid #f0f2f4;
  font-size: 10px;
}

.map-parameter-row > span { color: #354650; font-weight: 600; }
.map-parameter-row strong { color: #477e91; font-size: 9px; text-align: right; }

.map-profile-warning {
  display: flex;
  gap: 8px;
  padding: 12px;
  margin-top: 15px;
  border-radius: 8px;
  background: #fbf6ec;
  color: #94713a;
  font-size: 10px;
  line-height: 1.6;
}

.map-profile-loading {
  min-height: 85px;
  display: flex;
  justify-content: center;
  align-items: center;
  gap: 10px;
  color: #74858f;
  font-size: 11px;
}

@media (max-width: 760px) {
  .map-container { height: 490px; }
  .map-filters { flex-wrap: wrap; }
  .map-filters label { flex: 1; min-width: 135px; }
  .map-filter-summary { margin-left: 0; }
  .map-profile-metrics, .map-action-grid { grid-template-columns: repeat(2, 1fr); }
  .map-source-note { flex-direction: column; }
}

@media (max-width: 430px) {
  .map-container { height: 430px; }
  .map-legend { right: 8px; bottom: 8px; transform: scale(.9); transform-origin: bottom right; }
  .map-status-row { flex-direction: column; gap: 5px; }
}
"""

styles_path = SRC_DIR / "styles.css"

with styles_path.open("a", encoding="utf-8") as file:
    file.write("\n" + map_styles.strip() + "\n")

print("MAP STYLES APPENDED")
print("Stylesheet:", styles_path)
print("Total stylesheet size:", round(styles_path.stat().st_size / 1024, 2), "KB")

MAP STYLES APPENDED
Stylesheet: c:\career\projects\BoreWell Sentinel India\app\frontend\src\styles.css
Total stylesheet size: 17.28 KB


In [52]:
# CELL 49 — Integrate map into existing React application

app_path = SRC_DIR / "App.jsx"
app_source = app_path.read_text(encoding="utf-8")

# 1. Import the separate map component.
map_import = 'import MapExplorer from "./MapExplorer";'

if map_import not in app_source:
    anchor = 'const API = import.meta.env.VITE_API_BASE_URL'

    if anchor not in app_source:
        raise ValueError("Could not locate API configuration in App.jsx.")

    app_source = app_source.replace(
        anchor,
        map_import + "\n\n" + anchor,
        1
    )

# 2. Add a map item to sidebar navigation.
nav_anchor = '{ id: "overview", label: "Overview", icon: Gauge },'

if nav_anchor not in app_source:
    raise ValueError("Could not locate Overview navigation item.")

if 'id: "map", label: "Groundwater map"' not in app_source:
    app_source = app_source.replace(
        nav_anchor,
        nav_anchor + '\n    { id: "map", label: "Groundwater map", icon: Layers3 },',
        1
    )

# 3. Update the topbar page title.
old_title = (
    '{view === "overview" ? "Groundwater overview" : '
    'view === "sites" ? "Location intelligence" : "Trace-element review"}'
)

new_title = (
    '{view === "overview" ? "Groundwater overview" : '
    'view === "map" ? "India groundwater map" : '
    'view === "sites" ? "Location intelligence" : "Trace-element review"}'
)

if old_title in app_source:
    app_source = app_source.replace(old_title, new_title, 1)
elif new_title not in app_source:
    raise ValueError("Could not locate topbar title expression.")

# 4. Render the map page.
render_anchor = '            {view === "sites" && ('

map_render = '''            {view === "map" && (
              <MapExplorer />
            )}

'''

if render_anchor not in app_source:
    raise ValueError("Could not locate Site Explorer rendering block.")

if '<MapExplorer />' not in app_source:
    app_source = app_source.replace(
        render_anchor,
        map_render + render_anchor,
        1
    )

app_path.write_text(app_source, encoding="utf-8")

print("MAP INTEGRATION COMPLETE")
print("Map component imported:", map_import in app_source)
print("Map navigation item:", 'id: "map", label: "Groundwater map"' in app_source)
print("Map page rendering:", "<MapExplorer />" in app_source)
print("Original sections preserved:",
      all(section in app_source for section in [
          'id: "overview"',
          'id: "sites"',
          'id: "alerts"'
      ]))

MAP INTEGRATION COMPLETE
Map component imported: True
Map navigation item: True
Map page rendering: True
Original sections preserved: True


In [53]:
# CELL 50 — Production build for dashboard and map

import subprocess
import shutil

npm_executable = shutil.which("npm") or shutil.which("npm.cmd")

if not npm_executable:
    raise FileNotFoundError("npm executable not found.")

required_files = [
    FRONTEND_ROOT / "src" / "App.jsx",
    FRONTEND_ROOT / "src" / "MapExplorer.jsx",
    FRONTEND_ROOT / "src" / "styles.css"
]

for path in required_files:
    assert path.exists(), f"Missing frontend file: {path}"

result = subprocess.run(
    [npm_executable, "run", "build"],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)

if result.returncode != 0:
    print("\nBUILD ERRORS")
    print(result.stderr[-10000:])
    raise RuntimeError("Integrated map build failed.")

print("\nMAP BUILD PASSED")
print("Frontend output:", FRONTEND_ROOT / "dist")


> borewell-sentinel-frontend@0.1.0 build
> vite build

vite v8.3.8 building client environment for production...
transforming...
âœ“ 2575 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                   0.61 kB â”‚ gzip:   0.36 kB
dist/assets/index-BwRtUluW.css   30.94 kB â”‚ gzip:  10.15 kB
dist/assets/index-B-8akJJD.js   853.80 kB â”‚ gzip: 247.97 kB

âœ“ built in 988ms


MAP BUILD PASSED
Frontend output: C:\career\projects\BoreWell Sentinel India\app\frontend\dist


In [54]:
# CELL 51 — Test map state and priority filters

import importlib
from fastapi.testclient import TestClient

import backend.main as backend_main

backend_main = importlib.reload(backend_main)

with TestClient(backend_main.app) as client:

    tests = [
        ("/api/v1/map/sites", 24172),
        ("/api/v1/map/sites?tier=P1", None),
        ("/api/v1/map/sites?tier=P2", None),
        ("/api/v1/map/sites?tier=P3", None),
        ("/api/v1/map/sites?state=telangana", None)
    ]

    map_filter_results = []

    for endpoint, expected_count in tests:
        response = client.get(endpoint)
        response.raise_for_status()

        result = response.json()

        assert (
            result["mapped_locations"] == len(result["points"])
        )

        if expected_count is not None:
            assert result["mapped_locations"] == expected_count

        map_filter_results.append({
            "Endpoint": endpoint,
            "Filtered_locations": result["filtered_locations"],
            "Mapped_locations": result["mapped_locations"],
            "Excluded_coordinates": result["excluded_invalid_coordinates"],
            "Status": response.status_code
        })

    display(pd.DataFrame(map_filter_results))

print("\nPASS: Map endpoint and filters verified.")

,Endpoint,Filtered_locations,Mapped_locations,Excluded_coordinates,Status
0,/api/v1/map/sites,24199,24172,27,200
1,/api/v1/map/sites?tier=P1,2421,2416,5,200
2,/api/v1/map/sites?tier=P2,3629,3623,6,200
3,/api/v1/map/sites?tier=P3,18149,18133,16,200
4,/api/v1/map/sites?state=telangana,947,946,1,200



PASS: Map endpoint and filters verified.


In [55]:
# CELL 52 — Nationwide duplicate-coordinate audit

import pandas as pd
import numpy as np
from fastapi.testclient import TestClient
import backend.main as backend_main

# Retrieve the complete geographic register.
with TestClient(backend_main.app) as client:
    response = client.get("/api/v1/map/sites")
    response.raise_for_status()
    full_map_data = response.json()

coordinate_df = pd.DataFrame(full_map_data["points"])

coordinate_df["Latitude"] = pd.to_numeric(
    coordinate_df["Latitude"], errors="coerce"
)
coordinate_df["Longitude"] = pd.to_numeric(
    coordinate_df["Longitude"], errors="coerce"
)

# Every map record should represent one provisional spatial group.
assert coordinate_df["_spatial_group_id"].is_unique

print("GEOGRAPHIC REGISTER")
print("Total mapped location records:", len(coordinate_df))
print("Unique spatial group IDs:",
      coordinate_df["_spatial_group_id"].nunique())

# --------------------------------------------------
# 1. EXACT COORDINATE REUSE
# --------------------------------------------------

exact_duplicates = (
    coordinate_df
    .groupby(["Latitude", "Longitude"], dropna=False)
    .agg(
        Location_count=("_spatial_group_id", "nunique"),
        State_count=("State", "nunique"),
        District_count=("District", "nunique"),
        States=("State", lambda x: ", ".join(sorted(set(x.dropna())))),
        Districts=("District", lambda x: ", ".join(sorted(set(x.dropna())))),
        Spatial_group_IDs=(
            "_spatial_group_id",
            lambda x: list(x)[:8]
        )
    )
    .reset_index()
)

exact_duplicates = exact_duplicates.loc[
    exact_duplicates["Location_count"] > 1
].sort_values(
    ["State_count", "Location_count"],
    ascending=False
)

exact_reused_locations = int(
    exact_duplicates["Location_count"].sum()
)

exact_excess_locations = int(
    (exact_duplicates["Location_count"] - 1).sum()
)

print("\nEXACT DUPLICATE COORDINATE SUMMARY")
print("Repeated coordinate pairs:", len(exact_duplicates))
print("Locations participating in repeats:", exact_reused_locations)
print("Additional locations beyond first per pair:", exact_excess_locations)
print(
    "Repeated coordinate pairs shared across states:",
    int(exact_duplicates["State_count"].gt(1).sum())
)

display(exact_duplicates.head(25))

# --------------------------------------------------
# 2. APPROXIMATE COORDINATE REUSE
# --------------------------------------------------

precision_results = []

for decimals in [5, 4, 3]:

    rounded = coordinate_df.copy()

    rounded["Latitude_rounded"] = (
        rounded["Latitude"].round(decimals)
    )
    rounded["Longitude_rounded"] = (
        rounded["Longitude"].round(decimals)
    )

    grouped = (
        rounded.groupby(
            ["Latitude_rounded", "Longitude_rounded"]
        )
        .agg(
            Location_count=("_spatial_group_id", "nunique"),
            State_count=("State", "nunique")
        )
    )

    repeated = grouped.loc[
        grouped["Location_count"] > 1
    ]

    precision_results.append({
        "Decimal_places": decimals,
        "Repeated_coordinate_groups": len(repeated),
        "Locations_in_repeated_groups": int(
            repeated["Location_count"].sum()
        ),
        "Extra_locations_beyond_first": int(
            (repeated["Location_count"] - 1).sum()
        ),
        "Groups_shared_across_states": int(
            repeated["State_count"].gt(1).sum()
        )
    })

precision_summary = pd.DataFrame(precision_results)

print("\nAPPROXIMATE COORDINATE REUSE")
display(precision_summary)

# --------------------------------------------------
# 3. REPEATED COORDINATES WITHIN THE SAME STATE
# --------------------------------------------------

within_state_repeats = (
    coordinate_df
    .groupby(
        ["State", "Latitude", "Longitude"],
        dropna=False
    )
    .agg(
        Location_count=("_spatial_group_id", "nunique"),
        District_count=("District", "nunique"),
        Spatial_group_IDs=(
            "_spatial_group_id",
            lambda x: list(x)[:8]
        )
    )
    .reset_index()
)

within_state_repeats = within_state_repeats.loc[
    within_state_repeats["Location_count"] > 1
].sort_values(
    "Location_count",
    ascending=False
)

print("\nREPEATED COORDINATES WITHIN THE SAME STATE")
print("Repeated state-coordinate groups:", len(within_state_repeats))
display(within_state_repeats.head(25))

# --------------------------------------------------
# 4. STATE-WISE MAPPING SUMMARY
# --------------------------------------------------

state_coordinate_summary = (
    coordinate_df.groupby("State")
    .agg(
        Mapped_locations=("_spatial_group_id", "size"),
        Unique_exact_coordinates=(
            ["Latitude", "Longitude"],
            "nunique"
        ) if False else ("_spatial_group_id", "nunique")
    )
    .reset_index()
    .sort_values("Mapped_locations", ascending=False)
)

print("\nSTATE-WISE LOCATION COUNTS")
display(state_coordinate_summary.head(20))

print("\nAUDIT COMPLETE")
print("No coordinates or source records have been modified.")

GEOGRAPHIC REGISTER
Total mapped location records: 24172
Unique spatial group IDs: 24172

EXACT DUPLICATE COORDINATE SUMMARY
Repeated coordinate pairs: 808
Locations participating in repeats: 1700
Additional locations beyond first per pair: 892
Repeated coordinate pairs shared across states: 1


,Latitude,Longitude,Location_count,State_count,District_count,States,Districts,Spatial_group_IDs
5786,17.383333,78.433333,2,2,2,"andhra pradesh, telangana","anantapur, hyderabad",[telangana|hyderabad|gudimalkapur2|site|17.383...
20099,27.500000,94.544444,9,1,1,assam,dhemaji,"[assam|dhemaji|gogamukh|site|27.5|94.54444, as..."
988,10.686111,76.869444,8,1,1,kerala,palakkad,[kerala|palakkad|kambilichungam|site|10.68611|...
1349,11.250000,75.791667,6,1,1,kerala,kozhikode,[kerala|kozhikode|chevayur|site|11.25|75.79167...
642,9.919444,76.274722,5,1,1,kerala,ernakulam,[kerala|ernakulam|kallorkad|site|9.91944|76.27...
1096,10.857222,76.515556,5,1,1,kerala,palakkad,[kerala|palakkad|chakkanthara|site|10.85722|76...
15960,24.731111,92.890278,5,1,1,assam,cachar,[assam|cachar|sonaimukh|site|24.73111|92.89028...
752,10.152778,76.724722,4,1,1,kerala,ernakulam,[kerala|ernakulam|keerampara|site|10.15278|76....
908,10.543056,76.276944,4,1,1,kerala,trichur,[kerala|trichur|dyyal ambalam jn|site|10.54306...
2904,13.206667,79.101667,4,1,1,andhra pradesh,chittoor,[andhra pradesh|chittoor|chittoor|site|13.2066...



APPROXIMATE COORDINATE REUSE


,Decimal_places,Repeated_coordinate_groups,Locations_in_repeated_groups,Extra_locations_beyond_first,Groups_shared_across_states
0,5,815,1714,899,1
1,4,827,1739,912,1
2,3,932,1958,1026,1



REPEATED COORDINATES WITHIN THE SAME STATE
Repeated state-coordinate groups: 807


,State,Latitude,Longitude,Location_count,District_count,Spatial_group_IDs
1881,assam,27.500000,94.544444,9,1,"[assam|dhemaji|gogamukh|site|27.5|94.54444, as..."
9389,kerala,10.686111,76.869444,8,1,[kerala|palakkad|kambilichungam|site|10.68611|...
9544,kerala,11.250000,75.791667,6,1,[kerala|kozhikode|chevayur|site|11.25|75.79167...
9447,kerala,10.857222,76.515556,5,1,[kerala|palakkad|chakkanthara|site|10.85722|76...
9217,kerala,9.919444,76.274722,5,1,[kerala|ernakulam|kallorkad|site|9.91944|76.27...
1595,assam,24.731111,92.890278,5,1,[assam|cachar|sonaimukh|site|24.73111|92.89028...
9279,kerala,10.152778,76.724722,4,1,[kerala|ernakulam|keerampara|site|10.15278|76....
9350,kerala,10.543056,76.276944,4,1,[kerala|trichur|dyyal ambalam jn|site|10.54306...
10922,madhya pradesh,24.259722,80.764444,4,1,[madhya pradesh|satna|maihar-d|site|24.25972|8...
5657,haryana,29.293056,76.979167,4,1,[haryana|panipat|khelella pehl m|site|29.29306...



STATE-WISE LOCATION COUNTS


,State,Mapped_locations,Unique_exact_coordinates
15,karnataka,2043,2043
28,uttar pradesh,1965,1965
21,odisha,1952,1952
18,maharashtra,1855,1855
17,madhya pradesh,1707,1707
24,rajasthan,1632,1632
30,west bengal,1588,1588
1,andhra pradesh,1550,1550
6,chhattisgarh,1312,1312
25,tamil nadu,1218,1218



AUDIT COMPLETE
No coordinates or source records have been modified.


In [56]:
# CELL 53 — Detailed coordinate duplication and state anomaly audit

import numpy as np
import pandas as pd

audit_df = coordinate_df.copy()

audit_df["Site_name"] = (
    audit_df["_spatial_group_id"]
    .astype(str)
    .str.split("|", regex=False)
    .str[2]
)

audit_df["_coordinate_key"] = list(
    zip(audit_df["Latitude"], audit_df["Longitude"])
)

# Count exact coordinate reuse within each state.
audit_df["_same_state_coordinate_count"] = (
    audit_df.groupby(
        ["State", "Latitude", "Longitude"]
    )["_spatial_group_id"]
    .transform("size")
)

# Correct state-wise coordinate statistics.
state_coordinate_audit = (
    audit_df.groupby("State")
    .agg(
        Mapped_locations=("_spatial_group_id", "size"),
        Unique_exact_coordinate_pairs=("_coordinate_key", "nunique"),
        Locations_in_reused_coordinates=(
            "_same_state_coordinate_count",
            lambda x: int((x > 1).sum())
        )
    )
    .reset_index()
)

state_coordinate_audit["Extra_locations_at_reused_coordinates"] = (
    state_coordinate_audit["Locations_in_reused_coordinates"]
    - (
        state_coordinate_audit.groupby("State")["Locations_in_reused_coordinates"]
        .transform("size") * 0
    )
)

print("CORRECTED STATE-WISE COORDINATE AUDIT")
display(
    state_coordinate_audit.sort_values(
        "Locations_in_reused_coordinates",
        ascending=False
    ).head(20)
)

# Full detail of all exact repeated coordinate groups.
repeated_coordinate_details = audit_df.loc[
    audit_df["_same_state_coordinate_count"] > 1
].sort_values(
    ["State", "Latitude", "Longitude", "Site_name"]
)

print("\nREPEATED COORDINATE EXAMPLES")
display(
    repeated_coordinate_details[
        [
            "State",
            "District",
            "Site_name",
            "Latitude",
            "Longitude",
            "_same_state_coordinate_count",
            "_spatial_group_id"
        ]
    ].head(40)
)

# Cross-state exact coordinate conflict.
cross_state_pairs = (
    audit_df.groupby(["Latitude", "Longitude"])
    .agg(
        State_count=("State", "nunique"),
        Location_count=("_spatial_group_id", "nunique")
    )
    .reset_index()
)

cross_state_pairs = cross_state_pairs.loc[
    cross_state_pairs["State_count"] > 1
]

cross_state_detail = audit_df.merge(
    cross_state_pairs[["Latitude", "Longitude"]],
    on=["Latitude", "Longitude"],
    how="inner"
)

print("\nCROSS-STATE COORDINATE CONFLICTS")
display(
    cross_state_detail[
        [
            "State",
            "District",
            "Site_name",
            "Latitude",
            "Longitude",
            "_spatial_group_id"
        ]
    ]
)

# Distance from the median coordinate for each source-labelled state.
# This is an anomaly triage, NOT an administrative boundary test.
centres = (
    audit_df.groupby("State")[["Latitude", "Longitude"]]
    .median()
    .rename(columns={
        "Latitude": "_state_median_lat",
        "Longitude": "_state_median_lon"
    })
)

distance_df = audit_df.join(
    centres,
    on="State"
)

lat1 = np.radians(distance_df["Latitude"].to_numpy())
lon1 = np.radians(distance_df["Longitude"].to_numpy())
lat2 = np.radians(distance_df["_state_median_lat"].to_numpy())
lon2 = np.radians(distance_df["_state_median_lon"].to_numpy())

a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin((lon2 - lon1) / 2) ** 2
)

distance_df["Distance_from_state_median_km"] = (
    6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

for state_name in ["andhra pradesh", "chhattisgarh"]:

    state_outliers = (
        distance_df.loc[
            distance_df["State"].eq(state_name)
        ]
        .sort_values(
            "Distance_from_state_median_km",
            ascending=False
        )
        .head(20)
    )

    print(f"\nFARTHEST LOCATIONS FROM {state_name.upper()} MEDIAN")
    display(
        state_outliers[
            [
                "State",
                "District",
                "Site_name",
                "Latitude",
                "Longitude",
                "Distance_from_state_median_km",
                "_spatial_group_id"
            ]
        ].round(3)
    )

print("\nAUDIT COMPLETE — SOURCE RECORDS UNCHANGED")

CORRECTED STATE-WISE COORDINATE AUDIT


,State,Mapped_locations,Unique_exact_coordinate_pairs,Locations_in_reused_coordinates,Extra_locations_at_reused_coordinates
17,madhya pradesh,1707,1587,227,227
1,andhra pradesh,1550,1462,168,168
15,karnataka,2043,1959,166,166
16,kerala,1060,973,150,150
11,haryana,834,765,129,129
23,punjab,735,681,102,102
26,telangana,946,895,101,101
28,uttar pradesh,1965,1921,87,87
6,chhattisgarh,1312,1267,87,87
24,rajasthan,1632,1595,73,73



REPEATED COORDINATE EXAMPLES


,State,District,Site_name,Latitude,Longitude,_same_state_coordinate_count,_spatial_group_id
3196,andhra pradesh,chittoor,chittoor,13.206667,79.101667,4,andhra pradesh|chittoor|chittoor|site|13.20667...
10901,andhra pradesh,chittoor,chittoor-alt,13.206667,79.101667,4,andhra pradesh|chittoor|chittoor-alt|site|13.2...
9961,andhra pradesh,chittoor,puttur-alt,13.206667,79.101667,4,andhra pradesh|chittoor|puttur-alt|site|13.206...
6266,andhra pradesh,chittoor,srikalahasthi alt,13.206667,79.101667,4,andhra pradesh|chittoor|srikalahasthi alt|site...
1500,andhra pradesh,chittoor,g.nellore,13.233333,79.233333,2,andhra pradesh|chittoor|g.nellore|site|13.2333...
2643,andhra pradesh,chittoor,g.nellore1,13.233333,79.233333,2,andhra pradesh|chittoor|g.nellore1|site|13.233...
6722,andhra pradesh,chittoor,kanipakam,13.301667,79.056667,2,andhra pradesh|chittoor|kanipakam|site|13.3016...
4197,andhra pradesh,chittoor,kanipakam1,13.301667,79.056667,2,andhra pradesh|chittoor|kanipakam1|site|13.301...
3220,andhra pradesh,chittoor,tirupathi1,13.633333,79.359722,2,andhra pradesh|chittoor|tirupathi1|site|13.633...
10082,andhra pradesh,chittoor,tirupati(u),13.633333,79.359722,2,andhra pradesh|chittoor|tirupati(u)|site|13.63...



CROSS-STATE COORDINATE CONFLICTS


,State,District,Site_name,Latitude,Longitude,_spatial_group_id
0,telangana,hyderabad,gudimalkapur2,17.383333,78.433333,telangana|hyderabad|gudimalkapur2|site|17.3833...
1,andhra pradesh,anantapur,gudimalkapur-pz,17.383333,78.433333,andhra pradesh|anantapur|gudimalkapur-pz|site|...



FARTHEST LOCATIONS FROM ANDHRA PRADESH MEDIAN


,State,District,Site_name,Latitude,Longitude,Distance_from_state_median_km,_spatial_group_id
21133,andhra pradesh,anantapur,mohtli,32.217,75.671,1794.543,andhra pradesh|anantapur|mohtli|site|32.21667|...
14265,andhra pradesh,anantapur,channaur,32.146,75.688,1786.527,andhra pradesh|anantapur|channaur|site|32.1458...
15664,andhra pradesh,anantapur,bhalad,32.193,75.994,1784.521,andhra pradesh|anantapur|bhalad|site|32.19306|...
17801,andhra pradesh,anantapur,andaura,32.125,75.692,1784.187,andhra pradesh|anantapur|andaura|site|32.125|7...
22094,andhra pradesh,anantapur,bharmar,32.179,75.949,1784.035,andhra pradesh|anantapur|bharmar|site|32.17917...
12044,andhra pradesh,anantapur,kathgarh,32.103,75.688,1781.898,andhra pradesh|anantapur|kathgarh|site|32.1027...
3946,andhra pradesh,anantapur,tamota,32.071,75.612,1780.287,andhra pradesh|anantapur|tamota|site|32.07083|...
11101,andhra pradesh,anantapur,olherian,32.078,75.646,1780.219,andhra pradesh|anantapur|olherian|site|32.0777...
19072,andhra pradesh,anantapur,barota,32.058,75.721,1776.319,andhra pradesh|anantapur|barota|site|32.05833|...
3044,andhra pradesh,anantapur,aithabari,26.735,91.358,1637.130,andhra pradesh|anantapur|aithabari|site|26.734...



FARTHEST LOCATIONS FROM CHHATTISGARH MEDIAN


,State,District,Site_name,Latitude,Longitude,Distance_from_state_median_km,_spatial_group_id
20924,chhattisgarh,dhamtari,nagari1,13.327,79.617,988.641,chhattisgarh|dhamtari|nagari1|site|13.32667|79...
3531,chhattisgarh,dhamtari,bugga agraharam-dw14,13.331,79.622,988.005,chhattisgarh|dhamtari|bugga agraharam-dw14|sit...
6342,chhattisgarh,dantewada,kerlapal,18.288,81.554,404.760,chhattisgarh|dantewada|kerlapal|site|18.2875|8...
3855,chhattisgarh,dantewada,ramawaram,18.312,81.562,401.879,chhattisgarh|dantewada|ramawaram|site|18.3125|...
6633,chhattisgarh,dantewada,sukma,18.392,81.661,391.705,chhattisgarh|dantewada|sukma|site|18.39167|81....
14075,chhattisgarh,dantewada,rudraram,18.838,80.414,384.100,chhattisgarh|dantewada|rudraram|site|18.8375|8...
4955,chhattisgarh,dantewada,gaddiras,18.490,81.544,382.717,chhattisgarh|dantewada|gaddiras|site|18.48972|...
696,chhattisgarh,dantewada,dotpal,18.539,81.538,377.447,chhattisgarh|dantewada|dotpal|site|18.53889|81...
11681,chhattisgarh,dantewada,kirandul,18.629,81.258,373.719,chhattisgarh|dantewada|kirandul|site|18.62917|...
2879,chhattisgarh,dantewada,chhindgarh,18.550,81.758,373.006,chhattisgarh|dantewada|chhindgarh|site|18.55|8...



AUDIT COMPLETE — SOURCE RECORDS UNCHANGED


In [57]:
# CELL 54 — Add automatic map viewport fitting

map_path = SRC_DIR / "MapExplorer.jsx"
map_source = map_path.read_text(encoding="utf-8")

fit_component = r'''
function FitMapToPoints({ points, selectedState }) {
  const map = useMap();

  useEffect(() => {
    if (!points?.length) return;

    const locations = points
      .map((point) => [
        Number(point.Latitude),
        Number(point.Longitude)
      ])
      .filter(([lat, lng]) =>
        Number.isFinite(lat) &&
        Number.isFinite(lng) &&
        lat >= -90 && lat <= 90 &&
        lng >= -180 && lng <= 180
      );

    if (!locations.length) return;

    if (locations.length === 1) {
      map.setView(locations[0], 9, { animate: false });
      return;
    }

    const bounds = L.latLngBounds(locations);

    if (bounds.isValid()) {
      map.fitBounds(bounds, {
        padding: [45, 45],
        maxZoom: selectedState ? 8 : 5,
        animate: false
      });
    }
  }, [map, points, selectedState]);

  return null;
}
'''

if "function FitMapToPoints" not in map_source:
    anchor = "function MapMarkers({ points, onSelect })"

    if anchor not in map_source:
        raise ValueError("Could not locate MapMarkers component.")

    map_source = map_source.replace(
        anchor,
        fit_component + "\n\n" + anchor,
        1
    )

# Add viewport fitting inside MapContainer.
map_anchor = '''          <TileLayer
            attribution='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
            url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png"
            maxZoom={19}
          />'''

map_replacement = map_anchor + '''

          <FitMapToPoints
            points={mapData?.points || []}
            selectedState={state}
          />
'''

if "<FitMapToPoints" not in map_source:
    if map_anchor not in map_source:
        raise ValueError("Could not locate map tile layer.")

    map_source = map_source.replace(
        map_anchor,
        map_replacement,
        1
    )

map_path.write_text(
    map_source,
    encoding="utf-8"
)

print("AUTOMATIC MAP VIEWPORT FITTING ADDED")
print("Component present:", "function FitMapToPoints" in map_source)
print("Map integration present:", "<FitMapToPoints" in map_source)

AUTOMATIC MAP VIEWPORT FITTING ADDED
Component present: True
Map integration present: True


In [58]:
# CELL 55 — Verify map viewport update

import subprocess
import shutil

npm_executable = shutil.which("npm") or shutil.which("npm.cmd")

result = subprocess.run(
    [npm_executable, "run", "build"],
    cwd=str(FRONTEND_ROOT),
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr[-8000:])
    raise RuntimeError("Map viewport update failed to build.")

print("\nPASS: Updated map frontend compiled successfully.")


> borewell-sentinel-frontend@0.1.0 build
> vite build

vite v8.3.8 building client environment for production...
transforming...
âœ“ 2575 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                   0.61 kB â”‚ gzip:   0.36 kB
dist/assets/index-BwRtUluW.css   30.94 kB â”‚ gzip:  10.15 kB
dist/assets/index-CJCOuWWj.js   854.29 kB â”‚ gzip: 248.13 kB

âœ“ built in 1.01s


PASS: Updated map frontend compiled successfully.


In [59]:
# CELL 56 — Correct duplicate-coordinate counts and export review register

from pathlib import Path
import pandas as pd

MAP_QUALITY_DIR = PROJECT_ROOT / "reports" / "map_quality"
MAP_QUALITY_DIR.mkdir(parents=True, exist_ok=True)

# One row for each exact coordinate pair.
coordinate_groups = (
    coordinate_df
    .groupby(["Latitude", "Longitude"], as_index=False)
    .agg(
        Location_count=("_spatial_group_id", "nunique"),
        State_count=("State", "nunique"),
        District_count=("District", "nunique"),
        States=("State", lambda x: " | ".join(sorted(set(x.dropna())))),
        Districts=("District", lambda x: " | ".join(sorted(set(x.dropna())))),
        Site_names=(
            "_spatial_group_id",
            lambda x: " | ".join(
                sorted({
                    str(value).split("|")[2]
                    for value in x
                    if len(str(value).split("|")) > 2
                })
            )
        ),
        Spatial_group_IDs=(
            "_spatial_group_id",
            lambda x: " || ".join(sorted(set(x)))
        )
    )
)

coordinate_groups["Extra_location_IDs"] = (
    coordinate_groups["Location_count"] - 1
).clip(lower=0)

coordinate_groups["Review_status"] = "Unique coordinate"

coordinate_groups.loc[
    coordinate_groups["Location_count"] > 1,
    "Review_status"
] = "Repeated exact coordinate"

coordinate_groups.loc[
    coordinate_groups["State_count"] > 1,
    "Review_status"
] = "Cross-state coordinate conflict"

repeated_coordinate_register = (
    coordinate_groups.loc[
        coordinate_groups["Location_count"] > 1
    ]
    .sort_values(
        ["State_count", "Location_count"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("CORRECTED DUPLICATE SUMMARY")
print("Mapped location records:", len(coordinate_df))
print("Unique coordinate pairs:", len(coordinate_groups))
print("Repeated coordinate pairs:", len(repeated_coordinate_register))
print(
    "Locations participating in repeats:",
    int(repeated_coordinate_register["Location_count"].sum())
)
print(
    "True excess location IDs:",
    int(repeated_coordinate_register["Extra_location_IDs"].sum())
)
print(
    "Cross-state coordinate conflicts:",
    int(repeated_coordinate_register["State_count"].gt(1).sum())
)

print("\nTOP REPEATED COORDINATE GROUPS")
display(repeated_coordinate_register.head(20))

duplicate_register_path = (
    MAP_QUALITY_DIR / "phase10_exact_coordinate_review.csv"
)

repeated_coordinate_register.to_csv(
    duplicate_register_path,
    index=False
)

print("\nSAVED:", duplicate_register_path)

assert int(
    repeated_coordinate_register["Extra_location_IDs"].sum()
) == len(coordinate_df) - len(coordinate_groups)

print("PASS: Coordinate-group counts reconciled.")

CORRECTED DUPLICATE SUMMARY
Mapped location records: 24172
Unique coordinate pairs: 23280
Repeated coordinate pairs: 808
Locations participating in repeats: 1700
True excess location IDs: 892
Cross-state coordinate conflicts: 1

TOP REPEATED COORDINATE GROUPS


,Latitude,Longitude,Location_count,State_count,District_count,States,Districts,Site_names,Spatial_group_IDs,Extra_location_IDs,Review_status
0,17.383333,78.433333,2,2,2,andhra pradesh | telangana,anantapur | hyderabad,gudimalkapur-pz | gudimalkapur2,andhra pradesh|anantapur|gudimalkapur-pz|site|...,1,Cross-state coordinate conflict
1,27.500000,94.544444,9,1,1,assam,dhemaji,akajan | bhagaban charali | bordoloni | dhemaj...,assam|dhemaji|akajan|site|27.5|94.54444 || ass...,8,Repeated exact coordinate
2,10.686111,76.869444,8,1,1,kerala,palakkad,gopalapuram | kambilichungam | kongad1 | kozhi...,kerala|palakkad|gopalapuram|site|10.68611|76.8...,7,Repeated exact coordinate
3,11.250000,75.791667,6,1,1,kerala,kozhikode,chevayur | karaparamba | koodathumpara | kozhi...,kerala|kozhikode|chevayur|site|11.25|75.79167 ...,5,Repeated exact coordinate
4,9.919444,76.274722,5,1,1,kerala,ernakulam,cheria kadavu | kallorkad | palluruthy | patti...,kerala|ernakulam|cheria kadavu|site|9.91944|76...,4,Repeated exact coordinate
5,10.857222,76.515556,5,1,1,kerala,palakkad,chakkanthara | kootanad | kumaranallur | mattu...,kerala|palakkad|chakkanthara|site|10.85722|76....,4,Repeated exact coordinate
6,24.731111,92.890278,5,1,1,assam,cachar,lakhipur1 | poilapul | razabazar | shivtila | ...,assam|cachar|lakhipur1|site|24.73111|92.89028 ...,4,Repeated exact coordinate
7,10.152778,76.724722,4,1,1,kerala,ernakulam,keerampara | kuthukuzhi | mullankunnu | nellik...,kerala|ernakulam|keerampara|site|10.15278|76.7...,3,Repeated exact coordinate
8,10.543056,76.276944,4,1,1,kerala,trichur,arimpur1 | dyyal ambalam jn | edathrinji | map...,kerala|trichur|arimpur1|site|10.54306|76.27694...,3,Repeated exact coordinate
9,13.206667,79.101667,4,1,1,andhra pradesh,chittoor,chittoor | chittoor-alt | puttur-alt | srikala...,andhra pradesh|chittoor|chittoor-alt|site|13.2...,3,Repeated exact coordinate



SAVED: c:\career\projects\BoreWell Sentinel India\reports\map_quality\phase10_exact_coordinate_review.csv
PASS: Coordinate-group counts reconciled.


In [60]:
# CELL 57 — Retrieve Government of India state-boundary reference

import requests
import json

BOUNDARY_URL = (
    "https://mapservice.gov.in/gismapservice/rest/services/"
    "BharatMapService/Admin_Boundary_District/MapServer/0/query"
)

BOUNDARY_DIR = MAP_QUALITY_DIR / "boundary_reference"
BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY_GEOJSON_PATH = (
    BOUNDARY_DIR / "india_state_boundaries_reference.geojson"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "resultRecordCount": "1000",
    "f": "geojson"
}

print("REQUESTING STATE BOUNDARIES")
print("Source: Government of India BharatMapService")
print("Layer: State (Layer 0)")
print("Reference vintage: Published service metadata, October 2017")

response = requests.get(
    BOUNDARY_URL,
    params=params,
    timeout=120
)

response.raise_for_status()

boundary_data = response.json()

if "error" in boundary_data:
    raise RuntimeError(
        f"Boundary service returned an error: {boundary_data['error']}"
    )

features = boundary_data.get("features", [])

if not features:
    raise RuntimeError(
        "No state-boundary features were returned. "
        "Inspect the service response before continuing."
    )

if not all(
    feature.get("geometry") is not None
    for feature in features
):
    raise RuntimeError("One or more state features have no geometry.")

BOUNDARY_GEOJSON_PATH.write_text(
    json.dumps(boundary_data),
    encoding="utf-8"
)

print("\nBOUNDARY DOWNLOAD PASSED")
print("Feature count:", len(features))
print("Saved file:", BOUNDARY_GEOJSON_PATH)
print("First feature property names:")
print(list(features[0].get("properties", {}).keys()))
print("\nFirst feature attribute preview:")
display(pd.DataFrame([features[0].get("properties", {})]))

REQUESTING STATE BOUNDARIES
Source: Government of India BharatMapService
Layer: State (Layer 0)
Reference vintage: Published service metadata, October 2017


SSLError: HTTPSConnectionPool(host='mapservice.gov.in', port=443): Max retries exceeded with url: /gismapservice/rest/services/BharatMapService/Admin_Boundary_District/MapServer/0/query?where=1%3D1&outFields=%2A&returnGeometry=true&outSR=4326&resultRecordCount=1000&f=geojson (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1082)')))

In [61]:
# CELL 59 — Install and verify truststore

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("truststore") is None:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "truststore"],
        capture_output=True,
        text=True,
        timeout=180
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr[-5000:])
        raise RuntimeError("Truststore installation failed.")

import truststore
import ssl
import urllib3

print("Truststore:", truststore.__version__ if hasattr(truststore, "__version__") else "Available")
print("Python:", sys.version)
print("TLS verification: REQUIRED")
print("Windows native certificate store: enabled through truststore")

Truststore: 0.10.5
Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jun 18 2026, 21:18:42) [MSC v.1942 64 bit (AMD64)]
TLS verification: REQUIRED
Windows native certificate store: enabled through truststore


In [62]:
# CELL 60 — Securely retrieve India state boundaries

import json
from pathlib import Path
import ssl
import truststore
import urllib3

BOUNDARY_URL = (
    "https://mapservice.gov.in/gismapservice/rest/services/"
    "BharatMapService/Admin_Boundary_District/MapServer"
)

# Use Windows' native certificate store, with certificate checks enabled.
tls_context = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)

http = urllib3.PoolManager(
    ssl_context=tls_context,
    cert_reqs=ssl.CERT_REQUIRED,
    timeout=urllib3.Timeout(connect=20, read=120),
    retries=urllib3.Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    )
)

def secure_arcgis_get(url, params=None):
    response = http.request(
        "GET",
        url,
        fields=params or {},
        headers={"User-Agent": "BoreWell-Sentinel-India/1.0"}
    )

    if response.status != 200:
        raise RuntimeError(
            f"Boundary service HTTP status: {response.status}"
        )

    payload = json.loads(response.data.decode("utf-8"))

    if "error" in payload:
        raise RuntimeError(
            f"ArcGIS service error: {payload['error']}"
        )

    return payload

# First inspect service and layer metadata.
service_metadata = secure_arcgis_get(
    f"{BOUNDARY_URL}/0",
    {"f": "pjson"}
)

print("LAYER:", service_metadata.get("name"))
print("GEOMETRY:", service_metadata.get("geometryType"))
print("MAX RECORD COUNT:", service_metadata.get("maxRecordCount"))

boundary_fields = [
    item["name"]
    for item in service_metadata.get("fields", [])
]

print("FIELDS:", boundary_fields)

# Retrieve polygon features in WGS84 longitude/latitude.
boundary_geojson = secure_arcgis_get(
    f"{BOUNDARY_URL}/0/query",
    {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "resultRecordCount": "1000",
        "f": "geojson"
    }
)

boundary_features = boundary_geojson.get("features", [])

if not boundary_features:
    raise RuntimeError("No state boundary polygons were returned.")

if len(boundary_features) > 1000:
    raise RuntimeError(
        "Boundary response exceeded the expected feature limit; "
        "pagination needs to be implemented."
    )

if any(feature.get("geometry") is None for feature in boundary_features):
    raise RuntimeError("At least one state feature has missing geometry.")

BOUNDARY_DIR = MAP_QUALITY_DIR / "boundary_reference"
BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY_GEOJSON_PATH = (
    BOUNDARY_DIR / "india_state_boundaries_reference.geojson"
)

BOUNDARY_GEOJSON_PATH.write_text(
    json.dumps(boundary_geojson),
    encoding="utf-8"
)

boundary_properties_preview = pd.DataFrame([
    feature.get("properties", {})
    for feature in boundary_features
])

print("\nBOUNDARY DOWNLOAD PASSED")
print("Features retrieved:", len(boundary_features))
print("Saved to:", BOUNDARY_GEOJSON_PATH)

print("\nFEATURE PROPERTY NAMES")
print(boundary_properties_preview.columns.tolist())

print("\nSTATE NAME PREVIEW")
name_field = next(
    (field for field in ["STNAME", "stname", "STATE_NAME"]
     if field in boundary_properties_preview.columns),
    None
)

if name_field:
    display(
        boundary_properties_preview[[name_field]]
        .drop_duplicates()
        .sort_values(name_field)
    )
else:
    display(boundary_properties_preview.head())

print("\nBoundary vintage: October 2017, according to service metadata.")
print("TLS verification remained enabled.")

LAYER: State
GEOMETRY: esriGeometryPolygon
MAX RECORD COUNT: 1000
FIELDS: ['OBJECTID', 'Shape', 'STNAME', 'STCODE11', 'STNAME_SH', 'State_LGD', 'Shape_Length', 'Shape_Area', 'Remarks', 'Zone']

BOUNDARY DOWNLOAD PASSED
Features retrieved: 36
Saved to: c:\career\projects\BoreWell Sentinel India\reports\map_quality\boundary_reference\india_state_boundaries_reference.geojson

FEATURE PROPERTY NAMES
['OBJECTID', 'STNAME', 'STCODE11', 'STNAME_SH', 'State_LGD', 'Shape_Length', 'Shape_Area', 'Remarks', 'Zone']

STATE NAME PREVIEW


,STNAME
0,Andaman & Nicobar
1,Andhra Pradesh
2,Arunachal Pradesh
3,Assam
4,Bihar
5,Chandigarh
6,Chhattisgarh
35,"Dadra,Nagar Haveli,Daman & Diu"
7,Delhi
8,Goa



Boundary vintage: October 2017, according to service metadata.
TLS verification remained enabled.


In [63]:
# CELL 61 — Load boundary polygons and site register

from pathlib import Path
import json
import sys
import subprocess
import importlib.util

import pandas as pd
import numpy as np

# Install Shapely only if it is not already available
if importlib.util.find_spec("shapely") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "shapely"
    ])

from shapely.geometry import shape, Point
from shapely.prepared import prep

# Find the project root without assuming the notebook's current directory
current_path = Path.cwd()

PROJECT_ROOT = None

for candidate in [current_path, *current_path.parents]:
    required_files = [
        candidate / "reports/testing_prioritization/phase_8/phase8_site_priority_register.csv",
        candidate / "reports/map_quality/phase10_exact_coordinate_review.csv",
        candidate / "reports/map_quality/boundary_reference/india_state_boundaries_reference.geojson",
    ]

    if all(path.exists() for path in required_files):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root or one of the expected Phase 10 files."
    )

SITE_PATH = (
    PROJECT_ROOT
    / "reports/testing_prioritization/phase_8/phase8_site_priority_register.csv"
)

BOUNDARY_PATH = (
    PROJECT_ROOT
    / "reports/map_quality/boundary_reference/india_state_boundaries_reference.geojson"
)

# Load site data
sites_df = pd.read_csv(SITE_PATH, low_memory=False)

# Load GeoJSON boundaries
with open(BOUNDARY_PATH, "r", encoding="utf-8") as file:
    boundary_geojson = json.load(file)

# Validate required columns
required_columns = ["State", "Latitude", "Longitude"]

missing_columns = [
    column for column in required_columns
    if column not in sites_df.columns
]

if missing_columns:
    raise ValueError(f"Missing site columns: {missing_columns}")

if boundary_geojson.get("type") != "FeatureCollection":
    raise ValueError("The downloaded boundary file is not a GeoJSON FeatureCollection.")

# Prepare boundary geometries
state_boundaries = []

for feature in boundary_geojson["features"]:
    properties = feature.get("properties", {})
    state_name = properties.get("STNAME")
    geometry_data = feature.get("geometry")

    if not state_name or not geometry_data:
        continue

    geometry = shape(geometry_data)

    state_boundaries.append({
        "state": str(state_name),
        "geometry": prep(geometry)
    })

print("Project root:", PROJECT_ROOT)
print("Site records:", f"{len(sites_df):,}")
print("Boundary features:", len(state_boundaries))
print("Shapely version:", __import__("shapely").__version__)

print("\nRequired columns found:")
print(required_columns)

print("\nBoundary state names:")
print(sorted(item["state"] for item in state_boundaries))

Project root: c:\career\projects\BoreWell Sentinel India
Site records: 24,199
Boundary features: 36
Shapely version: 2.2.0

Required columns found:
['State', 'Latitude', 'Longitude']

Boundary state names:
['Andaman & Nicobar', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra,Nagar Haveli,Daman & Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [64]:
# CELL 62 — Point-in-polygon state validation

import re

def normalize_state(value):
    """Normalize state names for comparison without changing source data."""

    if pd.isna(value):
        return None

    key = re.sub(r"[^a-z0-9]", "", str(value).lower())

    aliases = {
        "andamanandnicobarislands": "andamannicobar",
        "andamannicobarislands": "andamannicobar",
        "jammuandkashmir": "jammukashmir",
        "nctofdelhi": "delhi",
        "nationalcapitalterritoryofdelhi": "delhi",
        "orissa": "odisha",
        "uttaranchal": "uttarakhand",
        "pondicherry": "puducherry",
        "dadraandnagarhavelianddamananddiu": "dadranagarhavellidamananddiu",
    }

    return aliases.get(key, key)


# Normalize boundary names for matching
boundary_names = {
    normalize_state(item["state"]): item["state"]
    for item in state_boundaries
}

validation_results = []

for latitude, longitude, recorded_state in sites_df[
    ["Latitude", "Longitude", "State"]
].itertuples(index=False, name=None):

    # Check missing coordinates
    if pd.isna(latitude) or pd.isna(longitude):
        validation_results.append({
            "Boundary_state_matches": "",
            "Boundary_match_status": "MISSING_COORDINATES"
        })
        continue

    try:
        latitude = float(latitude)
        longitude = float(longitude)
    except (TypeError, ValueError):
        validation_results.append({
            "Boundary_state_matches": "",
            "Boundary_match_status": "INVALID_COORDINATES"
        })
        continue

    # Validate coordinate ranges
    if (
        not np.isfinite(latitude)
        or not np.isfinite(longitude)
        or not (-90 <= latitude <= 90)
        or not (-180 <= longitude <= 180)
    ):
        validation_results.append({
            "Boundary_state_matches": "",
            "Boundary_match_status": "INVALID_COORDINATES"
        })
        continue

    point = Point(longitude, latitude)

    # Find every state polygon containing or touching this point
    matching_states = [
        item["state"]
        for item in state_boundaries
        if item["geometry"].covers(point)
    ]

    matching_states_normalized = {
        normalize_state(name) for name in matching_states
    }

    recorded_normalized = normalize_state(recorded_state)

    # Classify the result
    if recorded_normalized is None or recorded_normalized not in boundary_names:
        status = "UNKNOWN_RECORDED_STATE"

    elif recorded_normalized in matching_states_normalized:
        if len(matching_states) > 1:
            status = "ON_MULTIPLE_STATE_BOUNDARIES"
        else:
            status = "MATCH_RECORDED_STATE"

    elif matching_states:
        status = "OUTSIDE_RECORDED_STATE"

    else:
        status = "NO_STATE_POLYGON_MATCH"

    validation_results.append({
        "Boundary_state_matches": "; ".join(matching_states),
        "Boundary_match_status": status
    })


# Attach results to a separate audit copy
boundary_audit_df = sites_df.copy()

validation_df = pd.DataFrame(validation_results)

boundary_audit_df["Boundary_state_matches"] = (
    validation_df["Boundary_state_matches"]
)

boundary_audit_df["Boundary_match_status"] = (
    validation_df["Boundary_match_status"]
)

boundary_audit_df["Boundary_reference_vintage"] = "October 2017 metadata"

# Save the complete audit
output_dir = PROJECT_ROOT / "reports/map_quality"
output_dir.mkdir(parents=True, exist_ok=True)

audit_path = output_dir / "phase10_state_boundary_audit.csv"

boundary_audit_df.to_csv(audit_path, index=False)

# Save a separate manual-review register
review_statuses = [
    "OUTSIDE_RECORDED_STATE",
    "NO_STATE_POLYGON_MATCH",
    "UNKNOWN_RECORDED_STATE",
    "MISSING_COORDINATES",
    "INVALID_COORDINATES",
]

review_df = boundary_audit_df[
    boundary_audit_df["Boundary_match_status"].isin(review_statuses)
].copy()

review_path = output_dir / "phase10_state_boundary_review.csv"
review_df.to_csv(review_path, index=False)

# Display results
print("BOUNDARY VALIDATION COMPLETE")
print("=" * 55)

print("\nStatus counts:")
display(
    boundary_audit_df["Boundary_match_status"]
    .value_counts()
    .rename_axis("Status")
    .to_frame("Locations")
)

print("\nTotal locations audited:", f"{len(boundary_audit_df):,}")
print("Locations requiring review:", f"{len(review_df):,}")

print("\nReview sample:")
preview_columns = [
    "_spatial_group_id",
    "State",
    "District",
    "Latitude",
    "Longitude",
    "Boundary_state_matches",
    "Boundary_match_status"
]

preview_columns = [
    column for column in preview_columns
    if column in review_df.columns
]

display(review_df[preview_columns].head(30))

print("\nSaved:")
print(audit_path)
print(review_path)

BOUNDARY VALIDATION COMPLETE

Status counts:


,Locations
Status,
MATCH_RECORDED_STATE,23829
OUTSIDE_RECORDED_STATE,294
MISSING_COORDINATES,27
NO_STATE_POLYGON_MATCH,26
UNKNOWN_RECORDED_STATE,23



Total locations audited: 24,199
Locations requiring review: 370

Review sample:


,_spatial_group_id,State,District,Latitude,Longitude,Boundary_state_matches,Boundary_match_status
95,andaman and nicobar islands|south andaman|sipp...,andaman and nicobar islands,south andaman,11.604444,92.686111,,NO_STATE_POLYGON_MATCH
107,andhra pradesh|anantapur|aithabari|site|26.734...,andhra pradesh,anantapur,26.734722,91.358333,Assam,OUTSIDE_RECORDED_STATE
108,andhra pradesh|anantapur|akola|site|20.73667|7...,andhra pradesh,anantapur,20.736667,77.007222,Maharashtra,OUTSIDE_RECORDED_STATE
109,andhra pradesh|anantapur|akot1|site|21.11667|7...,andhra pradesh,anantapur,21.116667,77.050000,Maharashtra,OUTSIDE_RECORDED_STATE
110,andhra pradesh|anantapur|akot|site|21.11667|77.05,andhra pradesh,anantapur,21.116667,77.050000,Maharashtra,OUTSIDE_RECORDED_STATE
114,andhra pradesh|anantapur|alampur|site|26.02944...,andhra pradesh,anantapur,26.029444,78.797222,Madhya Pradesh,OUTSIDE_RECORDED_STATE
115,andhra pradesh|anantapur|alegaon|site|20.40972...,andhra pradesh,anantapur,20.409722,76.853333,Maharashtra,OUTSIDE_RECORDED_STATE
119,andhra pradesh|anantapur|ambashi|site|20.45667...,andhra pradesh,anantapur,20.456667,76.859167,Maharashtra,OUTSIDE_RECORDED_STATE
120,andhra pradesh|anantapur|ambikhalsa|site|19.35...,andhra pradesh,anantapur,19.350000,74.175000,Maharashtra,OUTSIDE_RECORDED_STATE
123,andhra pradesh|anantapur|andarsul|site|20.0013...,andhra pradesh,anantapur,20.001389,74.591667,Maharashtra,OUTSIDE_RECORDED_STATE



Saved:
c:\career\projects\BoreWell Sentinel India\reports\map_quality\phase10_state_boundary_audit.csv
c:\career\projects\BoreWell Sentinel India\reports\map_quality\phase10_state_boundary_review.csv


In [65]:
# CELL 63 — Investigate state-boundary mismatches

outside_df = boundary_audit_df[
    boundary_audit_df["Boundary_match_status"] == "OUTSIDE_RECORDED_STATE"
].copy()

unknown_df = boundary_audit_df[
    boundary_audit_df["Boundary_match_status"] == "UNKNOWN_RECORDED_STATE"
].copy()

no_polygon_df = boundary_audit_df[
    boundary_audit_df["Boundary_match_status"] == "NO_STATE_POLYGON_MATCH"
].copy()

# Identify an available station/site-name column
name_candidates = ["Station", "station", "Site_name", "site_name"]
name_col = next(
    (column for column in name_candidates if column in boundary_audit_df.columns),
    None
)

# 1. Which recorded states account for the mismatches?
print("1. MISMATCHES BY RECORDED STATE")
display(
    outside_df.groupby("State", dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("Locations")
    .to_frame()
)

# 2. Recorded state vs state containing the coordinates
print("\n2. RECORDED STATE → BOUNDARY STATE")
aggregation = {
    "Locations": ("State", "size"),
    "Districts": ("District", "nunique"),
}

if name_col:
    aggregation["Example_sites"] = (
        name_col,
        lambda values: "; ".join(
            values.dropna().astype(str).unique()[:5]
        )
    )

mismatch_pairs = (
    outside_df
    .groupby(
        ["State", "District", "Boundary_state_matches"],
        dropna=False
    )
    .agg(**aggregation)
    .reset_index()
    .sort_values("Locations", ascending=False)
)

display(mismatch_pairs.head(40))

# 3. Exact values responsible for unknown state names
print("\n3. UNKNOWN RECORDED STATE VALUES")
display(
    unknown_df.groupby("State", dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("Locations")
    .to_frame()
)

# 4. Locations with no containing state polygon
print("\n4. NO POLYGON MATCH — STATE AND DISTRICT")
display(
    no_polygon_df.groupby(
        ["State", "District"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .rename("Locations")
    .to_frame()
    .head(30)
)

# 5. Show the full exception sample, including coordinates
print("\n5. SAMPLE OF NON-MATCHING LOCATIONS")

sample_columns = [
    "_spatial_group_id",
    "State",
    "District",
    name_col,
    "Latitude",
    "Longitude",
    "Boundary_state_matches",
    "Boundary_match_status"
]

sample_columns = [
    column for column in sample_columns
    if column and column in boundary_audit_df.columns
]

display(
    boundary_audit_df[
        boundary_audit_df["Boundary_match_status"] != "MATCH_RECORDED_STATE"
    ][sample_columns].head(40)
)

print("\nAudit counts:")
print("Outside recorded state:", len(outside_df))
print("Unknown recorded state:", len(unknown_df))
print("No polygon match:", len(no_polygon_df))
print("Missing coordinates:", (
    boundary_audit_df["Boundary_match_status"] == "MISSING_COORDINATES"
).sum())

1. MISMATCHES BY RECORDED STATE


,Locations
State,
andhra pradesh,241
telangana,13
gujarat,11
kerala,9
chhattisgarh,2
maharashtra,2
odisha,2
goa,2
haryana,2



2. RECORDED STATE → BOUNDARY STATE


,State,District,Boundary_state_matches,Locations,Districts
8,andhra pradesh,anantapur,Maharashtra,161,1
9,andhra pradesh,anantapur,Rajasthan,16,1
10,andhra pradesh,anantapur,Telangana,14,1
33,telangana,khammam,Andhra Pradesh,12,1
2,andhra pradesh,anantapur,Gujarat,12,1
20,gujarat,daman,"Dadra,Nagar Haveli,Daman & Diu",9,1
11,andhra pradesh,anantapur,Uttar Pradesh,9,1
4,andhra pradesh,anantapur,Himachal Pradesh,9,1
26,kerala,palakkad,Tamil Nadu,8,1
3,andhra pradesh,anantapur,Haryana,6,1



3. UNKNOWN RECORDED STATE VALUES


,Locations
State,
dadra and nagar haveli,23



4. NO POLYGON MATCH — STATE AND DISTRICT


Locations
State                       District                     
tripura                     south tripura              11
andhra pradesh              visakhapatnam               2
andaman and nicobar islands south andaman               1
assam                       cachar                      1
goa                         south goa                   1
gujarat                     diu                         1
andhra pradesh              east godavari               1
gujarat                     jamnagar                    1
                            kachchh                     1
tamil nadu                  chennai                     1
kerala                      thiruvananthapuram          1
tamil nadu                  kanyakumari                 1
                            ramanathapuram              1
tripura                     dhalai                      1
                            gomati                      1


5. SAMPLE OF NON-MATCHING LOCATIONS


,_spatial_group_id,State,District,Latitude,Longitude,Boundary_state_matches,Boundary_match_status
95,andaman and nicobar islands|south andaman|sipp...,andaman and nicobar islands,south andaman,11.604444,92.686111,,NO_STATE_POLYGON_MATCH
107,andhra pradesh|anantapur|aithabari|site|26.734...,andhra pradesh,anantapur,26.734722,91.358333,Assam,OUTSIDE_RECORDED_STATE
108,andhra pradesh|anantapur|akola|site|20.73667|7...,andhra pradesh,anantapur,20.736667,77.007222,Maharashtra,OUTSIDE_RECORDED_STATE
109,andhra pradesh|anantapur|akot1|site|21.11667|7...,andhra pradesh,anantapur,21.116667,77.050000,Maharashtra,OUTSIDE_RECORDED_STATE
110,andhra pradesh|anantapur|akot|site|21.11667|77.05,andhra pradesh,anantapur,21.116667,77.050000,Maharashtra,OUTSIDE_RECORDED_STATE
114,andhra pradesh|anantapur|alampur|site|26.02944...,andhra pradesh,anantapur,26.029444,78.797222,Madhya Pradesh,OUTSIDE_RECORDED_STATE
115,andhra pradesh|anantapur|alegaon|site|20.40972...,andhra pradesh,anantapur,20.409722,76.853333,Maharashtra,OUTSIDE_RECORDED_STATE
119,andhra pradesh|anantapur|ambashi|site|20.45667...,andhra pradesh,anantapur,20.456667,76.859167,Maharashtra,OUTSIDE_RECORDED_STATE
120,andhra pradesh|anantapur|ambikhalsa|site|19.35...,andhra pradesh,anantapur,19.350000,74.175000,Maharashtra,OUTSIDE_RECORDED_STATE
123,andhra pradesh|anantapur|andarsul|site|20.0013...,andhra pradesh,anantapur,20.001389,74.591667,Maharashtra,OUTSIDE_RECORDED_STATE



Audit counts:
Outside recorded state: 294
Unknown recorded state: 23
No polygon match: 26
Missing coordinates: 27


In [66]:
# CELL 64 — Inspect Andhra Pradesh / Anantapur mismatches

anantapur_mismatch = outside_df[
    outside_df["State"].astype(str).str.strip().str.lower().eq("andhra pradesh")
    & outside_df["District"].astype(str).str.strip().str.lower().eq("anantapur")
].copy()

# The spatial group ID format contains the station identifier as its third segment.
anantapur_mismatch["Station_from_group_id"] = (
    anantapur_mismatch["_spatial_group_id"]
    .astype(str)
    .str.split("|", regex=False)
    .str[2]
)

print("Anantapur mismatch locations:", len(anantapur_mismatch))

print("\n1. Distribution by boundary state")
display(
    anantapur_mismatch.groupby("Boundary_state_matches")
    .agg(
        Locations=("_spatial_group_id", "size"),
        Unique_stations=("Station_from_group_id", "nunique"),
        Example_stations=(
            "Station_from_group_id",
            lambda x: ", ".join(x.dropna().astype(str).unique()[:12])
        )
    )
    .sort_values("Locations", ascending=False)
)

print("\n2. Sample of Maharashtra-mapped locations")
maharashtra_sample = anantapur_mismatch[
    anantapur_mismatch["Boundary_state_matches"].eq("Maharashtra")
][
    [
        "Station_from_group_id",
        "Latitude",
        "Longitude",
        "State",
        "District",
        "Boundary_state_matches"
    ]
].sort_values("Station_from_group_id")

display(maharashtra_sample.head(50))

print("\n3. Sample of all other Anantapur mismatches")
display(
    anantapur_mismatch[
        ~anantapur_mismatch["Boundary_state_matches"].eq("Maharashtra")
    ][
        [
            "Station_from_group_id",
            "Latitude",
            "Longitude",
            "Boundary_state_matches"
        ]
    ].head(40)
)

Anantapur mismatch locations: 238

1. Distribution by boundary state


,Locations,Unique_stations,Example_stations
Boundary_state_matches,,,
Maharashtra,161,161,"akola, akot1, akot, alegaon, ambashi, ambikhal..."
Rajasthan,16,16,"badki, bhukarka1, bhukarka, biramsar, bisrasar..."
Telangana,14,14,"annarugudem, erramanzil-pz, gudimalkapur-pz, g..."
Gujarat,12,12,"atul, dungri, kasal, katradevi, malda, mandvi2..."
Uttar Pradesh,9,9,"khaga(airwa) q, khaga2, khundauli q, khundauli..."
Himachal Pradesh,9,9,"andaura, barota, bhalad, bharmar, channaur, ka..."
Haryana,6,6,"badli, bahadurgarh, chhara, dulhera, kulasi, s..."
Madhya Pradesh,5,5,"alampur, balaji, daboh, dewri, lahar"
Jharkhand,3,3,"jaitgarh, mosabani1, mosabani"



2. Sample of Maharashtra-mapped locations


,Station_from_group_id,Latitude,Longitude,State,District,Boundary_state_matches
108,akola,20.736667,77.007222,andhra pradesh,anantapur,Maharashtra
110,akot,21.116667,77.050000,andhra pradesh,anantapur,Maharashtra
109,akot1,21.116667,77.050000,andhra pradesh,anantapur,Maharashtra
115,alegaon,20.409722,76.853333,andhra pradesh,anantapur,Maharashtra
119,ambashi,20.456667,76.859167,andhra pradesh,anantapur,Maharashtra
120,ambikhalsa,19.350000,74.175000,andhra pradesh,anantapur,Maharashtra
123,andarsul,20.001389,74.591667,andhra pradesh,anantapur,Maharashtra
126,andura,20.883333,76.852778,andhra pradesh,anantapur,Maharashtra
125,andura1,20.883333,76.852778,andhra pradesh,anantapur,Maharashtra
127,angangaon,20.041111,74.468333,andhra pradesh,anantapur,Maharashtra



3. Sample of all other Anantapur mismatches


,Station_from_group_id,Latitude,Longitude,Boundary_state_matches
107,aithabari,26.734722,91.358333,Assam
114,alampur,26.029444,78.797222,Madhya Pradesh
124,andaura,32.125000,75.691667,Himachal Pradesh
129,annarugudem,17.263611,80.429722,Telangana
134,atul,20.541667,72.966667,Gujarat
137,badki,23.581944,74.050000,Rajasthan
138,badli,28.591667,76.816667,Haryana
140,bahadurgarh,28.691667,76.933333,Haryana
141,balaji,26.296111,78.980000,Madhya Pradesh
146,barota,32.058333,75.720833,Himachal Pradesh


In [67]:
# CELL 65 — Trace flagged spatial groups to original processed records

MASTER_PATH = (
    PROJECT_ROOT
    / "data/processed/groundwater_master_processed.csv"
)

CONTEXT_PATH = (
    PROJECT_ROOT
    / "reports/feature_engineering/classification_context_labels.csv"
)

if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Master dataset not found: {MASTER_PATH}")

if not CONTEXT_PATH.exists():
    raise FileNotFoundError(f"Context labels file not found: {CONTEXT_PATH}")

# Identify the relevant Anantapur site IDs
target_spatial_ids = set(
    anantapur_mismatch["_spatial_group_id"].dropna().astype(str)
)

# Read only record ID and spatial group ID from the context file
context_df = pd.read_csv(
    CONTEXT_PATH,
    usecols=["_record_id", "_spatial_group_id"],
    low_memory=False
)

context_subset = context_df[
    context_df["_spatial_group_id"].astype(str).isin(target_spatial_ids)
].copy()

print("Flagged spatial groups:", len(target_spatial_ids))
print("Underlying record IDs found:", len(context_subset))

# Inspect master schema before loading selected columns
master_columns = pd.read_csv(MASTER_PATH, nrows=0).columns.tolist()

candidate_columns = [
    "_record_id",
    "State",
    "District",
    "Station",
    "Agency",
    "Latitude",
    "Longitude",
    "Data Acquisition Time"
]

available_columns = [
    column for column in candidate_columns
    if column in master_columns
]

if "_record_id" not in available_columns:
    raise ValueError(
        "The processed master does not contain _record_id; "
        "we need to inspect its record linkage before proceeding."
    )

print("\nSource columns available:")
print(available_columns)

# Load only relevant source columns
master_subset = pd.read_csv(
    MASTER_PATH,
    usecols=available_columns,
    low_memory=False
)

# Restrict to records associated with the flagged sites
source_records = master_subset.merge(
    context_subset,
    on="_record_id",
    how="inner",
    validate="one_to_one"
)

print("\nUnderlying source records:", f"{len(source_records):,}")

# Check the actual source State/District combinations
if {"State", "District"}.issubset(source_records.columns):
    print("\nSource State / District combinations")
    display(
        source_records.groupby(
            ["State", "District"],
            dropna=False
        )
        .size()
        .sort_values(ascending=False)
        .rename("Source_records")
        .to_frame()
        .head(30)
    )

# Show sample source records with the linked site identity
display(source_records.head(40))

print("\nNo source records have been modified.")

Flagged spatial groups: 238
Underlying record IDs found: 2165

Source columns available:
['_record_id', 'State', 'District', 'Station', 'Agency', 'Latitude', 'Longitude', 'Data Acquisition Time']

Underlying source records: 2,165

Source State / District combinations


,,Source_records
State,District,
Andhra Pradesh,ANANTAPUR,2165


,Station,Agency,State,District,Latitude,Longitude,Data Acquisition Time,_record_id,_spatial_group_id
0,Aithabari,CGWB,Andhra Pradesh,ANANTAPUR,26.734722,91.358333,25-05-2005 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|aithabari|site|26.734...
1,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2000 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
2,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
3,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2003 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
4,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2004 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
5,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2005 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
6,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2006 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
7,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2007 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
8,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2008 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...
9,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222,25-05-2011 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,andhra pradesh|anantapur|akola|site|20.73667|7...



No source records have been modified.


In [68]:
# CELL 66 — Identify original source files for Anantapur mismatches

import re
import zipfile
from pathlib import Path
from collections import Counter

# Inspect complete record IDs (not truncated dataframe display values)
record_ids = (
    source_records["_record_id"]
    .dropna()
    .astype(str)
)

print("1. SAMPLE COMPLETE RECORD IDs")
for record_id in record_ids.head(8):
    print(record_id)

print("\n2. COMMON RECORD-ID PREFIX")
print(
    os.path.commonprefix(record_ids.tolist())
    if len(record_ids) else "No record IDs found"
)

# Find CSV files while excluding generated outputs and installed packages
excluded_dirs = {
    "processed",
    "reports",
    "node_modules",
    ".venv",
    "__pycache__"
}

csv_files = []

for path in (PROJECT_ROOT / "data").rglob("*.csv"):
    if not any(part.lower() in excluded_dirs for part in path.parts):
        csv_files.append(path)

print("\n3. POSSIBLE ORIGINAL CSV FILES")
for path in sorted(csv_files):
    print(path.relative_to(PROJECT_ROOT))

# Find likely source files using ID tokens
id_text = " ".join(record_ids.head(100).tolist()).lower()

print("\n4. POSSIBLE CSV NAME MATCHES")
for path in sorted(csv_files):
    stem = re.sub(r"[^a-z0-9]", "", path.stem.lower())

    if (
        "cgwb" in stem
        or "chemical" in stem
        or "groundwater" in stem
        or any(token in id_text for token in [stem])
    ):
        print(path.relative_to(PROJECT_ROOT))

# Inspect ZIP contents without extracting or modifying files
zip_files = list(PROJECT_ROOT.rglob("1961-2025.zip"))

print("\n5. ORIGINAL ZIP FILES")
for zip_path in zip_files:
    print(zip_path.relative_to(PROJECT_ROOT))

    with zipfile.ZipFile(zip_path, "r") as archive:
        matching_names = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and (
                "cgwb" in name.lower()
                or "chemical" in name.lower()
                or "groundwater" in name.lower()
            )
        ]

        print("Likely source CSV entries:")
        for name in matching_names[:50]:
            print(" ", name)

print("\nNo source files were modified.")

1. SAMPLE COMPLETE RECORD IDs
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::120
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::139
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::140
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::141
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::142
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::143
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::144
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::145

2. COMMON RECORD-ID PREFIX
gwq_chemical_parameter_manual_cgwb_ap_1961_2025::

3. POSSIBLE ORIGINAL CSV FILES
data\extracted\1961-2025\gwq_chemical_parameter_manual_cgwb_an_1961_2025.csv
data\extracted\1961-2025\gwq_chemical_parameter_manual_cgwb_ap_1961_2025.csv
data\extracted\1961-2025\gwq_chemical_parameter_manual_cgwb_ar_1961_2025.csv
data\extracted\1961-2025\gwq_chemical_parameter_manual_cgwb_as_1961_2025.csv
data\extracted\1961-2025\gwq_chemical_parameter_manual_cgwb_br_1961_2025.csv
data\extracted\1961-2025\gwq_chemical_parameter_man

In [69]:
# CELL 67 — Inspect original CGWB Andhra Pradesh source rows

SOURCE_CSV = (
    PROJECT_ROOT
    / "data/extracted/1961-2025/"
      "gwq_chemical_parameter_manual_cgwb_ap_1961_2025.csv"
)

if not SOURCE_CSV.exists():
    raise FileNotFoundError(SOURCE_CSV)

raw_ap_df = pd.read_csv(SOURCE_CSV, low_memory=False)

# Recover original row positions from the record IDs
id_map = context_subset.copy()

id_parts = id_map["_record_id"].astype(str).str.rsplit("::", n=1)

id_map["source_row_position"] = id_parts.str[-1].astype(int)

expected_prefix = "gwq_chemical_parameter_manual_cgwb_ap_1961_2025"

if not id_parts.str[0].eq(expected_prefix).all():
    raise ValueError("Some flagged records do not belong to the expected source CSV.")

if id_map["source_row_position"].max() >= len(raw_ap_df):
    raise IndexError(
        "Record ID positions exceed the number of rows in the source CSV."
    )

# Retrieve the original rows by their source positions
raw_rows = raw_ap_df.iloc[
    id_map["source_row_position"].to_numpy()
].copy().reset_index(drop=True)

raw_rows.insert(
    0, "_source_row_position",
    id_map["source_row_position"].to_numpy()
)

raw_rows.insert(
    1, "_record_id",
    id_map["_record_id"].to_numpy()
)

raw_rows.insert(
    2, "_spatial_group_id",
    id_map["_spatial_group_id"].to_numpy()
)

print("Original source file:", SOURCE_CSV.name)
print("Original source row count:", f"{len(raw_ap_df):,}")
print("Flagged source records retrieved:", f"{len(raw_rows):,}")

print("\nOriginal source columns:")
print(raw_ap_df.columns.tolist())

print("\n1. ORIGINAL SOURCE STATE / DISTRICT COMBINATIONS")

display(
    raw_rows.groupby(
        ["State", "District"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .rename("Original_source_records")
    .to_frame()
    .head(30)
)

print("\n2. ORIGINAL SOURCE STATION / COORDINATE SAMPLE")

source_preview_columns = [
    "_source_row_position",
    "Station",
    "Agency",
    "State",
    "District",
    "Latitude",
    "Longitude"
]

source_preview_columns = [
    column for column in source_preview_columns
    if column in raw_rows.columns
]

display(
    raw_rows[source_preview_columns]
    .drop_duplicates()
    .head(50)
)

print("\nOriginal extracted CSV was read-only.")

Original source file: gwq_chemical_parameter_manual_cgwb_ap_1961_2025.csv
Original source row count: 10,950
Flagged source records retrieved: 2,165

Original source columns:
['SlNo', 'Station', 'Agency', 'State LGD Code', 'State', 'District LGD Code', 'District', 'Tehsil', 'Block', 'Village', 'River', 'Basin', 'Tributary', 'Subtributary', 'SubSubtributary', 'Local River', 'Latitude', 'Longitude', 'Data Acquisition Time', 'Potential of Hydrogen (pH)', 'Electric Conductivity (μS/cm)', 'Total Dissolved Solids (mg/L)', 'Carbonate (mg/L)', 'Bicarbonate (mg/L)', 'Total Alkalinity (mg/L as CaCO3)', 'Chloride (mg/L)', 'Nitrate N (mgN/L)', 'Sulphate (mg/L)', 'Phosphate(mg/L)', 'Silica(mg/L)', 'Fluoride (mg/L)', 'Total Hardness (mgCaCO3/L)', 'Calcium (mg/L)', 'Magnesium (mg/L)', 'Sodium (mg/L)', 'Potassium (mg/L)', 'Iron(mg/L)', 'Arsenic (mg/L)', 'Uranium(mg/L)', 'Manganese (mg/L)', 'Copper (mg/L)', 'Lead (mg/L)', 'Zinc (mg/L)', 'Nickel (mg/L)', 'Cadmium (mg/L)', 'Chromium (mg/L)']

1. ORIGINAL 

,,Original_source_records
State,District,
Andhra Pradesh,ANANTAPUR,2165



2. ORIGINAL SOURCE STATION / COORDINATE SAMPLE


,_source_row_position,Station,Agency,State,District,Latitude,Longitude
0,120,Aithabari,CGWB,Andhra Pradesh,ANANTAPUR,26.734722,91.358333
1,139,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
2,140,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
3,141,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
4,142,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
5,143,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
6,144,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
7,145,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
8,146,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222
9,147,Akola,CGWB,Andhra Pradesh,ANANTAPUR,20.736667,77.007222



Original extracted CSV was read-only.


In [70]:
# CELL 68 — Original vs processed record comparison

comparison_fields = [
    "Station",
    "Agency",
    "State",
    "District",
    "Latitude",
    "Longitude"
]

comparison_fields = [
    column for column in comparison_fields
    if column in raw_rows.columns and column in source_records.columns
]

original_compare = raw_rows[
    ["_record_id"] + comparison_fields
].copy()

processed_compare = source_records[
    ["_record_id"] + comparison_fields
].copy()

comparison = original_compare.merge(
    processed_compare,
    on="_record_id",
    how="outer",
    suffixes=("_original", "_processed"),
    indicator=True,
    validate="one_to_one"
)

print("Comparison records:", f"{len(comparison):,}")

print("\n1. Record linkage")
display(comparison["_merge"].value_counts().to_frame("Records"))

# Compare text columns after trimming whitespace and normalizing case.
for column in comparison_fields:
    original_col = f"{column}_original"
    processed_col = f"{column}_processed"

    if column in ["Latitude", "Longitude"]:
        original_values = pd.to_numeric(
            comparison[original_col], errors="coerce"
        )
        processed_values = pd.to_numeric(
            comparison[processed_col], errors="coerce"
        )

        equal = np.isclose(
            original_values.to_numpy(dtype=float),
            processed_values.to_numpy(dtype=float),
            equal_nan=True,
            rtol=0,
            atol=1e-8
        )

    else:
        original_values = (
            comparison[original_col]
            .fillna("<NA>")
            .astype(str)
            .str.strip()
            .str.casefold()
        )

        processed_values = (
            comparison[processed_col]
            .fillna("<NA>")
            .astype(str)
            .str.strip()
            .str.casefold()
        )

        equal = original_values.eq(processed_values).to_numpy()

    comparison[f"{column}_unchanged"] = equal

print("\n2. Field-level comparison")
for column in comparison_fields:
    check_col = f"{column}_unchanged"
    changed_count = int((~comparison[check_col]).sum())

    print(
        f"{column}: "
        f"{len(comparison) - changed_count:,} unchanged, "
        f"{changed_count:,} changed"
    )

# Display any mismatching records
unchanged_columns = [
    f"{column}_unchanged"
    for column in comparison_fields
]

changed_mask = ~comparison[unchanged_columns].all(axis=1)
changed_records = comparison[changed_mask]

print("\n3. Records with any changed field:", len(changed_records))

if len(changed_records):
    display(
        changed_records[
            ["_record_id"]
            + [
                item
                for column in comparison_fields
                for item in (
                    f"{column}_original",
                    f"{column}_processed"
                )
            ]
        ].head(30)
    )
else:
    print("All compared source fields match the processed master.")

print("\nNo source or processed records were modified.")

Comparison records: 2,165

1. Record linkage


,Records
_merge,
both,2165
left_only,0
right_only,0



2. Field-level comparison
Station: 2,165 unchanged, 0 changed
Agency: 2,165 unchanged, 0 changed
State: 2,165 unchanged, 0 changed
District: 2,165 unchanged, 0 changed
Latitude: 2,165 unchanged, 0 changed
Longitude: 2,165 unchanged, 0 changed

3. Records with any changed field: 0
All compared source fields match the processed master.

No source or processed records were modified.


In [71]:
# CELL 69 — Compare extracted CSV with original ZIP entry

import io
import zipfile
import pandas as pd
import numpy as np

ZIP_PATH = PROJECT_ROOT / "1961-2025.zip"

if not ZIP_PATH.exists():
    candidates = list(PROJECT_ROOT.rglob("1961-2025.zip"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not identify one original ZIP. Found: {candidates}"
        )
    ZIP_PATH = candidates[0]

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    matches = [
        name for name in archive.namelist()
        if name.replace("\\", "/").endswith(
            "gwq_chemical_parameter_manual_cgwb_ap_1961_2025.csv"
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one matching CSV in ZIP; found: {matches}"
        )

    archive_member = matches[0]

    archived_ap_df = pd.read_csv(
        io.BytesIO(archive.read(archive_member)),
        low_memory=False
    )

print("Original ZIP:", ZIP_PATH)
print("Archive member:", archive_member)
print("Extracted rows:", len(raw_ap_df))
print("Archived rows:", len(archived_ap_df))

# Compare columns and values
common_columns = [
    column for column in raw_ap_df.columns
    if column in archived_ap_df.columns
]

same_columns = raw_ap_df.columns.tolist() == archived_ap_df.columns.tolist()
same_shape = raw_ap_df.shape == archived_ap_df.shape

print("\n1. ZIP versus extracted CSV")
print("Same columns and order:", same_columns)
print("Same shape:", same_shape)

if same_shape and same_columns:
    equal_rows = (
        raw_ap_df.astype("string")
        .fillna("<NULL>")
        .eq(
            archived_ap_df.astype("string").fillna("<NULL>")
        )
        .all(axis=1)
    )

    print("Identical rows:", int(equal_rows.sum()))
    print("Different rows:", int((~equal_rows).sum()))

    if (~equal_rows).any():
        print("\nFirst differing row positions:")
        print(np.flatnonzero((~equal_rows).to_numpy())[:20])
else:
    print("The extracted and archived files have different structure.")

# Inspect all source state/district/LGD combinations
print("\n2. Full source file: State, District and LGD code combinations")

lgd_columns = [
    "State",
    "District",
    "State LGD Code",
    "District LGD Code"
]

display(
    raw_ap_df.groupby(lgd_columns, dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("Source_records")
    .to_frame()
    .head(60)
)

# Inspect codes specifically for the flagged records
flagged_positions = set(
    id_map["source_row_position"].astype(int)
)

flagged_source_rows = raw_ap_df.iloc[
    sorted(flagged_positions)
].copy()

print("\n3. LGD code values among the 2,165 flagged records")

for column in ["State LGD Code", "District LGD Code"]:
    if column in flagged_source_rows.columns:
        print(f"\n{column}:")
        display(
            flagged_source_rows[column]
            .value_counts(dropna=False)
            .rename("Records")
            .to_frame()
        )

print("\n4. Sample of non-Anantapur source rows")

other_rows = raw_ap_df[
    ~(
        raw_ap_df["State"].astype(str).str.strip().str.lower().eq(
            "andhra pradesh"
        )
        & raw_ap_df["District"].astype(str).str.strip().str.lower().eq(
            "anantapur"
        )
    )
]

display(
    other_rows[
        [
            column for column in [
                "SlNo",
                "Station",
                "State LGD Code",
                "State",
                "District LGD Code",
                "District",
                "Latitude",
                "Longitude"
            ]
            if column in other_rows.columns
        ]
    ].head(30)
)

print("\nAll operations were read-only.")

Original ZIP: c:\career\projects\BoreWell Sentinel India\data\raw\1961-2025.zip
Archive member: 1961-2025/gwq_chemical_parameter_manual_cgwb_ap_1961_2025.csv
Extracted rows: 10950
Archived rows: 10950

1. ZIP versus extracted CSV
Same columns and order: True
Same shape: True
Identical rows: 10950
Different rows: 0

2. Full source file: State, District and LGD code combinations


Source_records
State          District      State LGD Code District LGD Code                
Andhra Pradesh ANANTAPUR     28             502                          3095
               GUNTUR        28             506                           950
               KRISHNA       28             510                           886
               EAST GODAVARI 28             505                           844
               VISAKHAPATNAM 28             520                           835
               NELLORE       28             515                           642
               PRAKASAM      28             517                           614
               CHITTOOR      28             503                           579
               WEST GODAVARI 28             523                           568
               SRIKAKULAM    28             519                           527
               VIZIANAGARAM  28             521                           510
               KURNOOL       28             511                           481
               CUDDAPAH      28             504                           404
               Khammam       28             509                            15


3. LGD code values among the 2,165 flagged records

State LGD Code:


,Records
State LGD Code,
28,2165



District LGD Code:


,Records
District LGD Code,
502,2165



4. Sample of non-Anantapur source rows


,SlNo,Station,State LGD Code,State,District LGD Code,District,Latitude,Longitude
0,1,Addanki,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
1,2,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
2,3,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
3,4,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
4,5,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
5,6,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
6,7,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
7,8,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
8,9,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000
9,10,Addanki-new,28,Andhra Pradesh,517,PRAKASAM,15.833333,80.000000



All operations were read-only.


In [72]:
# CELL 70 — Cross-check flagged coordinates against all original state CSVs

EXTRACTED_DIR = PROJECT_ROOT / "data/extracted/1961-2025"

source_files = sorted(EXTRACTED_DIR.glob("*.csv"))

if not source_files:
    raise FileNotFoundError(f"No source CSV files found in {EXTRACTED_DIR}")

target_sites = anantapur_mismatch[
    ["_spatial_group_id", "Station_from_group_id",
     "Latitude", "Longitude", "Boundary_state_matches"]
].drop_duplicates("_spatial_group_id").copy()

target_sites["coord_key"] = list(zip(
    target_sites["Latitude"].round(6),
    target_sites["Longitude"].round(6)
))

target_coordinate_keys = set(target_sites["coord_key"])

search_columns = [
    "Station",
    "State LGD Code",
    "State",
    "District LGD Code",
    "District",
    "Latitude",
    "Longitude"
]

cross_file_matches = []
scanned_files = 0

for csv_path in source_files:

    # The Andhra Pradesh file is already the file under investigation.
    if csv_path.name == SOURCE_CSV.name:
        continue

    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    available = [column for column in search_columns if column in header]

    if not {"Latitude", "Longitude"}.issubset(available):
        continue

    frame = pd.read_csv(
        csv_path,
        usecols=available,
        low_memory=False
    )

    frame["Latitude"] = pd.to_numeric(frame["Latitude"], errors="coerce")
    frame["Longitude"] = pd.to_numeric(frame["Longitude"], errors="coerce")

    frame = frame.dropna(subset=["Latitude", "Longitude"]).copy()

    frame["coord_key"] = list(zip(
        frame["Latitude"].round(6),
        frame["Longitude"].round(6)
    ))

    matches = frame[frame["coord_key"].isin(target_coordinate_keys)].copy()

    if not matches.empty:
        matches["Matched_source_file"] = csv_path.name
        cross_file_matches.append(matches)

    scanned_files += 1

if cross_file_matches:
    cross_file_df = pd.concat(cross_file_matches, ignore_index=True)

    target_lookup = target_sites.merge(
        cross_file_df,
        on="coord_key",
        how="inner",
        suffixes=("_flagged", "_source")
    )

    print("Other source files scanned:", scanned_files)
    print("Matched source rows:", f"{len(cross_file_df):,}")
    print(
        "Flagged site groups with another-source coordinate match:",
        target_lookup["_spatial_group_id"].nunique()
    )

    print("\n1. Matches by source file and recorded state")

    display(
        cross_file_df.groupby(
            ["Matched_source_file", "State", "District"],
            dropna=False
        )
        .size()
        .sort_values(ascending=False)
        .rename("Matching_source_records")
        .to_frame()
        .head(40)
    )

    print("\n2. Examples linking flagged locations to other source records")

    result_columns = [
        "_spatial_group_id",
        "Station_from_group_id",
        "Latitude_flagged",
        "Longitude_flagged",
        "Boundary_state_matches",
        "Station",
        "State",
        "District",
        "State LGD Code",
        "District LGD Code",
        "Matched_source_file"
    ]

    result_columns = [
        column for column in result_columns
        if column in target_lookup.columns
    ]

    display(target_lookup[result_columns].drop_duplicates().head(50))

else:
    cross_file_df = pd.DataFrame()
    target_lookup = pd.DataFrame()

    print("Other source files scanned:", scanned_files)
    print("No exact coordinate matches found in other state CSVs.")

print("\nOriginal source files were not modified.")

Other source files scanned: 30
Matched source rows: 1
Flagged site groups with another-source coordinate match: 1

1. Matches by source file and recorded state


,,,Matching_source_records
Matched_source_file,State,District,
gwq_chemical_parameter_manual_cgwb_ts_1961_2025.csv,Telangana,HYDERABAD,1



2. Examples linking flagged locations to other source records


,_spatial_group_id,Station_from_group_id,Latitude_flagged,Longitude_flagged,Boundary_state_matches,Station,State,District,State LGD Code,District LGD Code,Matched_source_file
0,andhra pradesh|anantapur|gudimalkapur-pz|site|...,gudimalkapur-pz,17.383333,78.433333,Telangana,Gudimalkapur2,Telangana,HYDERABAD,36,507,gwq_chemical_parameter_manual_cgwb_ts_1961_202...



Original source files were not modified.


In [73]:
 # CELL 71 — Save source metadata anomaly register

source_counts = (
    raw_rows.groupby("_spatial_group_id")
    .agg(
        Source_measurement_records=("_record_id", "nunique"),
        Source_state_lgd_codes=(
            "State LGD Code",
            lambda x: "; ".join(sorted(x.dropna().astype(str).unique()))
        ),
        Source_district_lgd_codes=(
            "District LGD Code",
            lambda x: "; ".join(sorted(x.dropna().astype(str).unique()))
        ),
        Source_record_id_count=("_record_id", "count")
    )
)

anomaly_register = anantapur_mismatch.merge(
    source_counts,
    left_on="_spatial_group_id",
    right_index=True,
    how="left",
    validate="one_to_one"
)

anomaly_register["Original_source_file"] = SOURCE_CSV.name
anomaly_register["Original_zip_match"] = "IDENTICAL"
anomaly_register["Processed_fields_match_source"] = "ALL_CHECKED_FIELDS_MATCH"
anomaly_register["Source_metadata_review_status"] = (
    "INPUT_METADATA_COORDINATE_CONFLICT"
)
anomaly_register["Automatic_correction_applied"] = False
anomaly_register["Boundary_reference"] = (
    "BharatMapService state boundaries; October 2017 metadata"
)

register_columns = [
    "_spatial_group_id",
    "Station_from_group_id",
    "State",
    "District",
    "Latitude",
    "Longitude",
    "Boundary_state_matches",
    "Boundary_match_status",
    "Source_measurement_records",
    "Source_state_lgd_codes",
    "Source_district_lgd_codes",
    "Original_source_file",
    "Original_zip_match",
    "Processed_fields_match_source",
    "Source_metadata_review_status",
    "Automatic_correction_applied",
    "Boundary_reference"
]

register_columns = [
    column for column in register_columns
    if column in anomaly_register.columns
]

anomaly_register = anomaly_register[register_columns].copy()

register_path = (
    PROJECT_ROOT
    / "reports/map_quality/phase10_source_metadata_anomaly_register.csv"
)

register_path.parent.mkdir(parents=True, exist_ok=True)
anomaly_register.to_csv(register_path, index=False)

print("SOURCE METADATA ANOMALY REGISTER SAVED")
print("=" * 55)
print("Flagged site groups:", f"{len(anomaly_register):,}")
print(
    "Source measurement records:",
    f"{anomaly_register['Source_measurement_records'].sum():,}"
)
print(
    "Unique source LGD code combinations:",
    anomaly_register[
        ["Source_state_lgd_codes", "Source_district_lgd_codes"]
    ].drop_duplicates().shape[0]
)
print("Automatic corrections:", anomaly_register["Automatic_correction_applied"].sum())
print("\nSaved to:")
print(register_path)

display(anomaly_register.head(15))

SOURCE METADATA ANOMALY REGISTER SAVED
Flagged site groups: 238
Source measurement records: 2,165
Unique source LGD code combinations: 1
Automatic corrections: 0

Saved to:
c:\career\projects\BoreWell Sentinel India\reports\map_quality\phase10_source_metadata_anomaly_register.csv


,_spatial_group_id,Station_from_group_id,State,District,Latitude,Longitude,Boundary_state_matches,Boundary_match_status,Source_measurement_records,Source_state_lgd_codes,Source_district_lgd_codes,Original_source_file,Original_zip_match,Processed_fields_match_source,Source_metadata_review_status,Automatic_correction_applied,Boundary_reference
107,andhra pradesh|anantapur|aithabari|site|26.734...,aithabari,andhra pradesh,anantapur,26.734722,91.358333,Assam,OUTSIDE_RECORDED_STATE,1,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
108,andhra pradesh|anantapur|akola|site|20.73667|7...,akola,andhra pradesh,anantapur,20.736667,77.007222,Maharashtra,OUTSIDE_RECORDED_STATE,15,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
109,andhra pradesh|anantapur|akot1|site|21.11667|7...,akot1,andhra pradesh,anantapur,21.116667,77.050000,Maharashtra,OUTSIDE_RECORDED_STATE,2,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
110,andhra pradesh|anantapur|akot|site|21.11667|77.05,akot,andhra pradesh,anantapur,21.116667,77.050000,Maharashtra,OUTSIDE_RECORDED_STATE,2,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
114,andhra pradesh|anantapur|alampur|site|26.02944...,alampur,andhra pradesh,anantapur,26.029444,78.797222,Madhya Pradesh,OUTSIDE_RECORDED_STATE,18,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
115,andhra pradesh|anantapur|alegaon|site|20.40972...,alegaon,andhra pradesh,anantapur,20.409722,76.853333,Maharashtra,OUTSIDE_RECORDED_STATE,7,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
119,andhra pradesh|anantapur|ambashi|site|20.45667...,ambashi,andhra pradesh,anantapur,20.456667,76.859167,Maharashtra,OUTSIDE_RECORDED_STATE,7,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
120,andhra pradesh|anantapur|ambikhalsa|site|19.35...,ambikhalsa,andhra pradesh,anantapur,19.350000,74.175000,Maharashtra,OUTSIDE_RECORDED_STATE,17,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
123,andhra pradesh|anantapur|andarsul|site|20.0013...,andarsul,andhra pradesh,anantapur,20.001389,74.591667,Maharashtra,OUTSIDE_RECORDED_STATE,14,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...
124,andhra pradesh|anantapur|andaura|site|32.125|7...,andaura,andhra pradesh,anantapur,32.125000,75.691667,Himachal Pradesh,OUTSIDE_RECORDED_STATE,15,28,502,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,IDENTICAL,ALL_CHECKED_FIELDS_MATCH,INPUT_METADATA_COORDINATE_CONFLICT,False,BharatMapService state boundaries; October 201...


In [74]:
# CELL 72 — Reconcile Anantapur mismatch rows and unique site groups

expected_ids = set(
    anantapur_mismatch["_spatial_group_id"].dropna().astype(str)
)

registered_ids = set(
    anomaly_register["_spatial_group_id"].dropna().astype(str)
)

id_counts = (
    anantapur_mismatch["_spatial_group_id"]
    .value_counts(dropna=False)
)

duplicate_group_ids = id_counts[id_counts > 1]

missing_from_register = expected_ids - registered_ids
unexpected_in_register = registered_ids - expected_ids

print("ANANTAPUR MISMATCH RECONCILIATION")
print("=" * 55)
print("Mismatch rows:", len(anantapur_mismatch))
print("Unique spatial group IDs:", len(expected_ids))
print("Rows beyond unique group count:", len(anantapur_mismatch) - len(expected_ids))
print("Unique IDs in saved register:", len(registered_ids))
print("IDs missing from saved register:", len(missing_from_register))
print("Unexpected IDs in saved register:", len(unexpected_in_register))
print("Repeated spatial group IDs:", len(duplicate_group_ids))

if len(duplicate_group_ids):
    print("\nRepeated group IDs and row counts:")
    display(
        anantapur_mismatch[
            anantapur_mismatch["_spatial_group_id"].isin(
                duplicate_group_ids.index
            )
        ][
            [
                "_spatial_group_id",
                "Station_from_group_id",
                "Latitude",
                "Longitude",
                "Boundary_state_matches"
            ]
        ].sort_values("_spatial_group_id")
    )

if missing_from_register:
    print("\nIDs missing from the saved register:")
    display(
        anantapur_mismatch[
            anantapur_mismatch["_spatial_group_id"].isin(
                missing_from_register
            )
        ]
    )

if unexpected_in_register:
    print("\nUnexpected IDs in the saved register:")
    display(
        anomaly_register[
            anomaly_register["_spatial_group_id"].isin(
                unexpected_in_register
            )
        ]
    )

print("\nSource measurement records represented in register:",
      f"{anomaly_register['Source_measurement_records'].sum():,}")

ANANTAPUR MISMATCH RECONCILIATION
Mismatch rows: 238
Unique spatial group IDs: 238
Rows beyond unique group count: 0
Unique IDs in saved register: 238
IDs missing from saved register: 0
Unexpected IDs in saved register: 0
Repeated spatial group IDs: 0

Source measurement records represented in register: 2,165


In [75]:
# CELL 73 — Inspect current map API and frontend implementation

from pathlib import Path

files_to_inspect = {
    "BACKEND MAIN": PROJECT_ROOT / "app/backend/main.py",
    "DATA SERVICE": PROJECT_ROOT / "app/backend/services/data_service.py",
    "MAP EXPLORER": PROJECT_ROOT / "app/frontend/src/MapExplorer.jsx",
}

def print_relevant_sections(path, keywords, context=30, max_lines=650):
    print(f"\n{'=' * 80}")
    print(path.name)
    print("=" * 80)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        return

    lines = path.read_text(encoding="utf-8").splitlines()

    matched = []
    for index, line in enumerate(lines):
        if any(keyword.lower() in line.lower() for keyword in keywords):
            start = max(0, index - context)
            end = min(len(lines), index + context + 1)
            matched.append((start, end))

    if not matched:
        print("No matching sections found.")
        return

    # Merge overlapping ranges
    merged = []
    for start, end in matched:
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    printed = 0
    for start, end in merged:
        print(f"\n--- Lines {start + 1} to {end} ---")

        for index in range(start, end):
            if printed >= max_lines:
                print("\nOUTPUT CAPPED.")
                return

            print(f"{index + 1:4}: {lines[index]}")
            printed += 1


print_relevant_sections(
    files_to_inspect["BACKEND MAIN"],
    ["map/sites", "map_sites", "site_priority"],
    context=45
)

print_relevant_sections(
    files_to_inspect["DATA SERVICE"],
    ["site_priority", "load_site", "get_site"],
    context=30
)

print_relevant_sections(
    files_to_inspect["MAP EXPLORER"],
    ["fetch(", "MapContainer", "Marker", "Popup", "useEffect", "selectedSite"],
    context=35
)


main.py

--- Lines 284 to 375 ---
 284:             },
 285:             {
 286:                 "name": "Electric Conductivity (μS/cm)",
 287:                 "unit": "μS/cm",
 288:                 "transformation": "log1p"
 289:             },
 290:             {
 291:                 "name": "Chloride (mg/L)",
 292:                 "unit": "mg/L",
 293:                 "transformation": "log1p"
 294:             },
 295:             {
 296:                 "name": "Bicarbonate (mg/L)",
 297:                 "unit": "mg/L",
 298:                 "transformation": "log1p"
 299:             },
 300:             {
 301:                 "name": "Sulphate (mg/L)",
 302:                 "unit": "mg/L",
 303:                 "transformation": "log1p",
 304:                 "conditional_screening": True
 305:             },
 306:             {
 307:                 "name": "Calcium (mg/L)",
 308:                 "unit": "mg/L",
 309:                 "transformation": "log1p"
 310:          

In [76]:
# CELL 74 — Locate map endpoint and frontend component sections

import re
from pathlib import Path

main_path = PROJECT_ROOT / "app/backend/main.py"
service_path = PROJECT_ROOT / "app/backend/services/data_service.py"
map_path = PROJECT_ROOT / "app/frontend/src/MapExplorer.jsx"

# 1. Locate the map API route
main_lines = main_path.read_text(encoding="utf-8").splitlines()

print("=" * 65)
print("BACKEND MAP ENDPOINT")
print("=" * 65)

matches = [
    i for i, line in enumerate(main_lines)
    if "/api/v1/map/sites" in line or "map_sites" in line.lower()
]

if matches:
    for i in matches:
        start = max(0, i - 5)
        end = min(len(main_lines), i + 12)
        for j in range(start, end):
            print(f"{j + 1:4}: {main_lines[j]}")
        print("---")
else:
    print("No exact map endpoint match found.")

# 2. List data-service function locations
service_lines = service_path.read_text(encoding="utf-8").splitlines()

print("\n" + "=" * 65)
print("DATA SERVICE FUNCTIONS")
print("=" * 65)

for i, line in enumerate(service_lines):
    if re.match(r"\s*(async\s+)?def\s+", line):
        print(f"{i + 1:4}: {line.strip()}")

# 3. List frontend component and function locations
map_lines = map_path.read_text(encoding="utf-8").splitlines()

print("\n" + "=" * 65)
print("MAP EXPLORER STRUCTURE")
print("=" * 65)
print("Total lines:", len(map_lines))

for i, line in enumerate(map_lines):
    if re.match(
        r"\s*(export\s+default\s+)?function\s+|"
        r"\s*(export\s+)?const\s+\w+\s*=\s*(async\s*)?\(",
        line
    ):
        print(f"{i + 1:4}: {line.strip()}")

print("\nKEY MAP REFERENCES")

keywords = [
    "MapContainer",
    "MarkerClusterGroup",
    "MapMarkers",
    "Marker",
    "Popup",
    "selectedSite",
    "setSelectedSite",
    "/api/v1/map/sites",
    "fetch("
]

for i, line in enumerate(map_lines):
    if any(keyword.lower() in line.lower() for keyword in keywords):
        print(f"{i + 1:4}: {line.strip()[:180]}")

BACKEND MAP ENDPOINT
 324:             "or certifications of groundwater safety."
 325:         )
 326:     }
 327: 
 328: 
 329: @app.get("/api/v1/map/sites")
 330: def map_sites(
 331:     state: str | None = Query(default=None, max_length=100),
 332:     tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
 333: ):
 334:     import pandas as pd
 335: 
 336:     sites = get_data()["sites"]
 337:     filtered = sites.copy()
 338: 
 339:     if state:
 340:         filtered = filtered.loc[
---
 325:         )
 326:     }
 327: 
 328: 
 329: @app.get("/api/v1/map/sites")
 330: def map_sites(
 331:     state: str | None = Query(default=None, max_length=100),
 332:     tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
 333: ):
 334:     import pandas as pd
 335: 
 336:     sites = get_data()["sites"]
 337:     filtered = sites.copy()
 338: 
 339:     if state:
 340:         filtered = filtered.loc[
 341:             filtered["State"].astype(str).str.casefold()
---



In [77]:
# CELL 75 — Extract exact backend and frontend map code

import re

def show_lines(path, start, end, title):
    lines = path.read_text(encoding="utf-8").splitlines()
    print(f"\n{'=' * 75}\n{title} ({start + 1}-{min(end, len(lines))})\n{'=' * 75}")

    for i in range(start, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")


# 1. Backend map endpoint
main_path = PROJECT_ROOT / "app/backend/main.py"
main_lines = main_path.read_text(encoding="utf-8").splitlines()

route_start = next(
    i for i, line in enumerate(main_lines)
    if '@app.get("/api/v1/map/sites")' in line
)

route_end = next(
    (
        i for i in range(route_start + 1, len(main_lines))
        if main_lines[i].startswith("@app.")
    ),
    min(route_start + 100, len(main_lines))
)

show_lines(main_path, route_start, route_end, "BACKEND MAP ROUTE")


# 2. Data service
service_path = PROJECT_ROOT / "app/backend/services/data_service.py"
service_lines = service_path.read_text(encoding="utf-8").splitlines()

show_lines(
    service_path,
    0,
    len(service_lines),
    "DATA SERVICE"
)


# 3. Frontend: imports, state, API calls, marker rendering and map
map_path = PROJECT_ROOT / "app/frontend/src/MapExplorer.jsx"
map_lines = map_path.read_text(encoding="utf-8").splitlines()

patterns = [
    r"^import ",
    r"^export default",
    r"function ",
    r"const .*=",
    r"useState",
    r"useEffect",
    r"fetch\(",
    r"MapContainer",
    r"MapMarkers",
    r"MarkerClusterGroup",
    r"<Marker",
    r"<Popup",
    r"selectedSite",
    r"setSelectedSite",
    r"fitBounds",
]

matched_ranges = []

for i, line in enumerate(map_lines):
    if any(re.search(pattern, line) for pattern in patterns):
        matched_ranges.append((
            max(0, i - 5),
            min(len(map_lines), i + 9)
        ))

merged_ranges = []

for start, end in matched_ranges:
    if merged_ranges and start <= merged_ranges[-1][1]:
        merged_ranges[-1] = (
            merged_ranges[-1][0],
            max(merged_ranges[-1][1], end)
        )
    else:
        merged_ranges.append((start, end))

print("\nFRONTEND FILE LENGTH:", len(map_lines))

total_printed = 0

for start, end in merged_ranges:
    if total_printed >= 260:
        print("\nFrontend output capped at 260 lines.")
        break

    actual_end = min(end, start + (260 - total_printed))

    show_lines(
        map_path,
        start,
        actual_end,
        "FRONTEND MAP SECTION"
    )

    total_printed += actual_end - start


BACKEND MAP ROUTE (329-399)
 329: @app.get("/api/v1/map/sites")
 330: def map_sites(
 331:     state: str | None = Query(default=None, max_length=100),
 332:     tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
 333: ):
 334:     import pandas as pd
 335: 
 336:     sites = get_data()["sites"]
 337:     filtered = sites.copy()
 338: 
 339:     if state:
 340:         filtered = filtered.loc[
 341:             filtered["State"].astype(str).str.casefold()
 342:             == state.strip().casefold()
 343:         ]
 344: 
 345:     if tier:
 346:         filtered = filtered.loc[
 347:             filtered["Priority_tier"]
 348:             .fillna("")
 349:             .astype(str)
 350:             .str.startswith(tier.upper())
 351:         ]
 352: 
 353:     total_filtered = len(filtered)
 354: 
 355:     latitude = pd.to_numeric(
 356:         filtered["Latitude"], errors="coerce"
 357:     )
 358: 
 359:     longitude = pd.to_numeric(
 360:         filtered["Longitud

In [78]:
# CELL 76 — Read only the complete map endpoint

main_path = PROJECT_ROOT / "app/backend/main.py"
lines = main_path.read_text(encoding="utf-8").splitlines()

for i in range(328, 399):
    print(f"{i + 1:4}: {lines[i]}")

 329: @app.get("/api/v1/map/sites")
 330: def map_sites(
 331:     state: str | None = Query(default=None, max_length=100),
 332:     tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
 333: ):
 334:     import pandas as pd
 335: 
 336:     sites = get_data()["sites"]
 337:     filtered = sites.copy()
 338: 
 339:     if state:
 340:         filtered = filtered.loc[
 341:             filtered["State"].astype(str).str.casefold()
 342:             == state.strip().casefold()
 343:         ]
 344: 
 345:     if tier:
 346:         filtered = filtered.loc[
 347:             filtered["Priority_tier"]
 348:             .fillna("")
 349:             .astype(str)
 350:             .str.startswith(tier.upper())
 351:         ]
 352: 
 353:     total_filtered = len(filtered)
 354: 
 355:     latitude = pd.to_numeric(
 356:         filtered["Latitude"], errors="coerce"
 357:     )
 358: 
 359:     longitude = pd.to_numeric(
 360:         filtered["Longitude"], errors="coerce"
 361:   

In [79]:
# CELL 77A — Route lines 354–369
for i in range(353, 369):
    print(f"{i + 1:4}: {lines[i]}")

 354: 
 355:     latitude = pd.to_numeric(
 356:         filtered["Latitude"], errors="coerce"
 357:     )
 358: 
 359:     longitude = pd.to_numeric(
 360:         filtered["Longitude"], errors="coerce"
 361:     )
 362: 
 363:     valid_coordinates = (
 364:         latitude.between(-90, 90)
 365:         & longitude.between(-180, 180)
 366:     )
 367: 
 368:     geo = filtered.loc[valid_coordinates].copy()
 369: 


In [80]:
# CELL 77B — Route lines 370–384
for i in range(369, 384):
    print(f"{i + 1:4}: {lines[i]}")

 370:     geo["Latitude"] = latitude.loc[valid_coordinates]
 371:     geo["Longitude"] = longitude.loc[valid_coordinates]
 372: 
 373:     map_columns = [
 374:         "_spatial_group_id",
 375:         "State",
 376:         "District",
 377:         "Latitude",
 378:         "Longitude",
 379:         "Priority_rank",
 380:         "Priority_tier",
 381:         "Testing_priority_score",
 382:         "Primary_priority_driver",
 383:         "Total_records",
 384:         "Distinct_years"


In [81]:
# CELL 77C — Route lines 385–399
for i in range(384, 399):
    print(f"{i + 1:4}: {lines[i]}")

 385:     ]
 386: 
 387:     geo = geo[map_columns].sort_values(
 388:         "Priority_rank",
 389:         ascending=True
 390:     )
 391: 
 392:     return {
 393:         "filtered_locations": int(total_filtered),
 394:         "mapped_locations": int(len(geo)),
 395:         "excluded_invalid_coordinates": int(
 396:             total_filtered - len(geo)
 397:         ),
 398:         "points": records_from_frame(geo)
 399:     }


In [82]:
# CELL 79 — Read MapExplorer.jsx

frontend_path = PROJECT_ROOT / "app/frontend/src/MapExplorer.jsx"

frontend_lines = frontend_path.read_text(encoding="utf-8").splitlines()

print("Frontend file:", frontend_path)
print("Total lines:", len(frontend_lines))

for i, line in enumerate(frontend_lines[:180]):
    print(f"{i + 1:4}: {line}")

Frontend file: c:\career\projects\BoreWell Sentinel India\app\frontend\src\MapExplorer.jsx
Total lines: 498
   1: import React, { useCallback, useEffect, useMemo, useRef, useState } from "react";
   2: import L from "leaflet";
   3: import "leaflet.markercluster";
   4: import {
   5:   MapContainer, TileLayer, useMap
   6: } from "react-leaflet";
   7: 
   8: import {
   9:   AlertTriangle, Crosshair, Layers3, LoaderCircle,
  10:   MapPin, RefreshCw, X
  11: } from "lucide-react";
  12: 
  13: import "leaflet/dist/leaflet.css";
  14: import "leaflet.markercluster/dist/MarkerCluster.css";
  15: import "leaflet.markercluster/dist/MarkerCluster.Default.css";
  16: 
  17: const API = import.meta.env.VITE_API_BASE_URL || "http://127.0.0.1:8000";
  18: 
  19: const number = (value) =>
  20:   Number(value || 0).toLocaleString("en-IN");
  21: 
  22: const tierClass = (tier) => {
  23:   if (String(tier).startsWith("P1")) return "p1";
  24:   if (String(tier).startsWith("P2")) return "p2";
  

In [83]:
# CELL 80 — Locate frontend rendering and selection code

keywords = [
    "function ",
    "const Map",
    "const [",
    "setPoints",
    "setDetail",
    "setSelectedSite",
    "fetch(",
    "MapMarkers",
    "MarkerClusterGroup",
    "circleMarker",
    "L.marker",
    "bindPopup",
    "setLatLng",
    "MapContainer",
    "Popup",
    "selectedSite",
    "onClick",
    "fitBounds",
]

print("FRONTEND COMPONENT STRUCTURE")
for i, line in enumerate(frontend_lines):
    if any(word.lower() in line.lower() for word in keywords):
        print(f"{i + 1:4}: {line.strip()[:180]}")

print("\nFRONTEND LINES 150–300")
for i in range(149, min(300, len(frontend_lines))):
    print(f"{i + 1:4}: {frontend_lines[i]}")

print("\nFRONTEND LINES 301–498")
for i in range(300, len(frontend_lines)):
    print(f"{i + 1:4}: {frontend_lines[i]}")

FRONTEND COMPONENT STRUCTURE
   5: MapContainer, TileLayer, useMap
  38: function FitMapToPoints({ points, selectedState }) {
  39: const map = useMap();
  66: map.fitBounds(bounds, {
  78: function MapMarkers({ points, onSelect }) {
  79: const map = useMap();
  89: const clusters = L.markerClusterGroup({
  96: zoomToBoundsOnClick: true,
 135: const marker = L.marker([lat, lng], {
 140: const popup = `
 141: <div class="map-popup">
 142: <div class="map-popup-title">
 145: <div class="map-popup-subtitle">
 148: <div class="map-popup-grid">
 156: <div class="map-popup-hint">Click marker to open the site profile</div>
 160: marker.bindPopup(popup, {
 183: function MapLegend() {
 194: function SiteProfile({ detail, loading, onClose }) {
 217: <button className="icon-button" onClick={onClose} title="Close profile">
 269: export default function MapExplorer() {
 270: const [mapData, setMapData] = useState(null);
 271: const [states, setStates] = useState([]);
 272: const [state, setState] 

In [84]:
for i in range(77, 183):
    print(f"{i + 1:4}: {frontend_lines[i]}")

  78: function MapMarkers({ points, onSelect }) {
  79:   const map = useMap();
  80:   const selectRef = useRef(onSelect);
  81: 
  82:   useEffect(() => {
  83:     selectRef.current = onSelect;
  84:   }, [onSelect]);
  85: 
  86:   useEffect(() => {
  87:     if (!points?.length) return;
  88: 
  89:     const clusters = L.markerClusterGroup({
  90:       chunkedLoading: true,
  91:       chunkInterval: 150,
  92:       chunkDelay: 35,
  93:       maxClusterRadius: 55,
  94:       spiderfyOnMaxZoom: true,
  95:       showCoverageOnHover: false,
  96:       zoomToBoundsOnClick: true,
  97:       animate: false,
  98:       removeOutsideVisibleBounds: true,
  99: 
 100:       iconCreateFunction(cluster) {
 101:         const count = cluster.getChildCount();
 102:         const size = count < 10 ? "small" : count < 100 ? "medium" : "large";
 103: 
 104:         return L.divIcon({
 105:           html: `<div class="cluster-bubble ${size}"><span>${number(count)}</span></div>`,
 106:    

In [85]:
for i in range(268, 390):
    print(f"{i + 1:4}: {frontend_lines[i]}")

 269: export default function MapExplorer() {
 270:   const [mapData, setMapData] = useState(null);
 271:   const [states, setStates] = useState([]);
 272:   const [state, setState] = useState("");
 273:   const [tier, setTier] = useState("ALL");
 274: 
 275:   const [loading, setLoading] = useState(true);
 276:   const [error, setError] = useState("");
 277: 
 278:   const [selectedId, setSelectedId] = useState(null);
 279:   const [detail, setDetail] = useState(null);
 280:   const [detailLoading, setDetailLoading] = useState(false);
 281: 
 282:   const loadMap = useCallback(async () => {
 283:     setLoading(true);
 284:     setError("");
 285: 
 286:     try {
 287:       const params = new URLSearchParams();
 288: 
 289:       if (state) params.set("state", state);
 290:       if (tier !== "ALL") params.set("tier", tier);
 291: 
 292:       const suffix = params.toString()
 293:         ? `?${params.toString()}`
 294:         : "";
 295: 
 296:       const [mapResponse, statesRes

In [86]:
for i in range(389, 498):
    print(f"{i + 1:4}: {frontend_lines[i]}")

 390:             onChange={(event) => setState(event.target.value)}
 391:           >
 392:             <option value="">All states</option>
 393:             {states.map((item) => (
 394:               <option value={item} key={item}>{item}</option>
 395:             ))}
 396:           </select>
 397:         </label>
 398: 
 399:         <label>
 400:           <span>PRIORITY TIER</span>
 401:           <select
 402:             value={tier}
 403:             onChange={(event) => setTier(event.target.value)}
 404:           >
 405:             <option value="ALL">All tiers</option>
 406:             <option value="P1">P1 — Highest 10%</option>
 407:             <option value="P2">P2 — Next 15%</option>
 408:             <option value="P3">P3 — Remaining</option>
 409:           </select>
 410:         </label>
 411: 
 412:         <div className="map-filter-summary">
 413:           <Layers3 size={16} />
 414:           {summaryLabel}
 415:         </div>
 416:       </div>
 417: 


In [87]:
for i in range(100, 145):
    print(f"{i + 1:4}: {frontend_lines[i]}")

 101:         const count = cluster.getChildCount();
 102:         const size = count < 10 ? "small" : count < 100 ? "medium" : "large";
 103: 
 104:         return L.divIcon({
 105:           html: `<div class="cluster-bubble ${size}"><span>${number(count)}</span></div>`,
 106:           className: "cluster-icon-wrapper",
 107:           iconSize: [42, 42],
 108:           iconAnchor: [21, 21]
 109:         });
 110:       }
 111:     });
 112: 
 113:     for (const point of points) {
 114:       const lat = Number(point.Latitude);
 115:       const lng = Number(point.Longitude);
 116: 
 117:       if (
 118:         !Number.isFinite(lat) ||
 119:         !Number.isFinite(lng) ||
 120:         lat < -90 || lat > 90 ||
 121:         lng < -180 || lng > 180
 122:       ) {
 123:         continue;
 124:       }
 125: 
 126:       const tier = tierClass(point.Priority_tier);
 127: 
 128:       const icon = L.divIcon({
 129:         html: '<span class="site-marker-core"></span>',
 130:    

In [88]:
for i in range(145, 183):
    print(f"{i + 1:4}: {frontend_lines[i]}")

 146:             ${escapeHTML(point.State)}
 147:           </div>
 148:           <div class="map-popup-grid">
 149:             <span>Priority rank</span>
 150:             <strong>#${number(point.Priority_rank)}</strong>
 151:             <span>Testing score</span>
 152:             <strong>${Number(point.Testing_priority_score || 0).toFixed(2)}</strong>
 153:             <span>Tier</span>
 154:             <strong>${escapeHTML(String(point.Priority_tier || "").split(" — ")[0])}</strong>
 155:           </div>
 156:           <div class="map-popup-hint">Click marker to open the site profile</div>
 157:         </div>
 158:       `;
 159: 
 160:       marker.bindPopup(popup, {
 161:         maxWidth: 250,
 162:         closeButton: true
 163:       });
 164: 
 165:       marker.on("click", () => {
 166:         selectRef.current(point._spatial_group_id);
 167:       });
 168: 
 169:       clusters.addLayer(marker);
 170:     }
 171: 
 172:     map.addLayer(clusters);
 173: 
 174:   

In [89]:
for i in range(282, 370):
    print(f"{i + 1:4}: {frontend_lines[i]}")

 283:     setLoading(true);
 284:     setError("");
 285: 
 286:     try {
 287:       const params = new URLSearchParams();
 288: 
 289:       if (state) params.set("state", state);
 290:       if (tier !== "ALL") params.set("tier", tier);
 291: 
 292:       const suffix = params.toString()
 293:         ? `?${params.toString()}`
 294:         : "";
 295: 
 296:       const [mapResponse, statesResponse] = await Promise.all([
 297:         fetch(`${API}/api/v1/map/sites${suffix}`),
 298:         fetch(`${API}/api/v1/dashboard/states?limit=40`)
 299:       ]);
 300: 
 301:       if (!mapResponse.ok) {
 302:         throw new Error(`Map endpoint returned ${mapResponse.status}`);
 303:       }
 304: 
 305:       if (!statesResponse.ok) {
 306:         throw new Error(`State endpoint returned ${statesResponse.status}`);
 307:       }
 308: 
 309:       const mapJson = await mapResponse.json();
 310:       const statesJson = await statesResponse.json();
 311: 
 312:       setMapData(mapJson

In [90]:
# CELL 83 — Add exact-coordinate grouping API

from pathlib import Path
import sys
import importlib
import subprocess

main_path = PROJECT_ROOT / "app/backend/main.py"
main_source = main_path.read_text(encoding="utf-8")

route_name = '@app.get("/api/v1/map/coordinate-groups")'

endpoint_code = '''

@app.get("/api/v1/map/coordinate-groups")
def map_coordinate_groups(
    state: str | None = Query(default=None, max_length=100),
    tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
):
    """
    Return one map marker per exact latitude/longitude pair.
    Every original site remains available in the group's members list.
    """
    import pandas as pd

    sites = get_data()["sites"]
    filtered = sites.copy()

    if state:
        filtered = filtered.loc[
            filtered["State"].astype(str).str.casefold()
            == state.strip().casefold()
        ]

    if tier:
        filtered = filtered.loc[
            filtered["Priority_tier"]
            .fillna("")
            .astype(str)
            .str.startswith(tier.upper())
        ]

    total_filtered = len(filtered)

    latitude = pd.to_numeric(filtered["Latitude"], errors="coerce")
    longitude = pd.to_numeric(filtered["Longitude"], errors="coerce")

    valid_coordinates = (
        latitude.between(-90, 90)
        & longitude.between(-180, 180)
    )

    geo = filtered.loc[valid_coordinates].copy()

    geo["Latitude"] = latitude.loc[valid_coordinates]
    geo["Longitude"] = longitude.loc[valid_coordinates]

    map_columns = [
        "_spatial_group_id",
        "State",
        "District",
        "Latitude",
        "Longitude",
        "Priority_rank",
        "Priority_tier",
        "Testing_priority_score",
        "Primary_priority_driver",
        "Total_records",
        "Distinct_years"
    ]

    geo = geo[map_columns].copy()

    # Sort so the highest-priority site is the representative site
    # for each coordinate group.
    geo["_rank_sort"] = pd.to_numeric(
        geo["Priority_rank"], errors="coerce"
    )

    geo = (
        geo.sort_values("_rank_sort", ascending=True, na_position="last")
        .drop(columns=["_rank_sort"])
        .reset_index(drop=True)
    )

    # Serialize the individual site records once.
    serialized_sites = records_from_frame(geo)

    grouped = geo.groupby(
        ["Latitude", "Longitude"],
        sort=False,
        dropna=False
    )

    coordinate_groups = []

    for (lat, lng), positions in grouped.indices.items():
        positions = [int(position) for position in positions]

        member_records = [
            serialized_sites[position]
            for position in positions
        ]

        group_frame = geo.iloc[positions]
        representative = member_records[0]

        states = sorted({
            str(value).strip()
            for value in group_frame["State"].dropna()
            if str(value).strip()
        })

        districts = sorted({
            str(value).strip()
            for value in group_frame["District"].dropna()
            if str(value).strip()
        })

        tier_values = (
            group_frame["Priority_tier"]
            .fillna("")
            .astype(str)
            .str.extract(r"^(P[123])", expand=False)
        )

        priority_counts = {
            priority: int(tier_values.eq(priority).sum())
            for priority in ("P1", "P2", "P3")
        }

        coordinate_groups.append({
            "Latitude": float(lat),
            "Longitude": float(lng),

            "location_count": len(member_records),

            "State": states[0] if len(states) == 1 else "Multiple states",
            "District": (
                districts[0]
                if len(districts) == 1
                else "Multiple districts"
            ),

            "states": states,
            "districts": districts,
            "state_count": len(states),
            "district_count": len(districts),
            "has_state_conflict": len(states) > 1,

            "Priority_rank": representative.get("Priority_rank"),
            "Priority_tier": representative.get("Priority_tier"),
            "Testing_priority_score": representative.get(
                "Testing_priority_score"
            ),
            "Primary_priority_driver": representative.get(
                "Primary_priority_driver"
            ),

            "priority_counts": priority_counts,

            # Preserve every original location ID and its site-level data.
            "members": member_records
        })

    duplicate_groups = [
        group for group in coordinate_groups
        if group["location_count"] > 1
    ]

    return {
        "filtered_locations": int(total_filtered),
        "mapped_locations": int(len(geo)),
        "excluded_invalid_coordinates": int(total_filtered - len(geo)),

        "coordinate_groups": int(len(coordinate_groups)),
        "duplicate_coordinate_groups": int(len(duplicate_groups)),
        "locations_in_duplicate_groups": int(
            sum(group["location_count"] for group in duplicate_groups)
        ),
        "extra_locations_at_duplicate_coordinates": int(
            sum(group["location_count"] - 1 for group in duplicate_groups)
        ),

        "groups": coordinate_groups
    }

'''

if route_name in main_source:
    raise RuntimeError(
        "The coordinate-groups endpoint already exists. "
        "No file changes were made."
    )

main_path.write_text(
    main_source.rstrip() + "\n\n" + endpoint_code,
    encoding="utf-8"
)

print("Added endpoint:", "/api/v1/map/coordinate-groups")
print("Updated:", main_path)

Added endpoint: /api/v1/map/coordinate-groups
Updated: c:\career\projects\BoreWell Sentinel India\app\backend\main.py


In [92]:
# CELL 84 — Validate coordinate groups against existing map API
# Corrected: explicitly start the FastAPI application lifecycle.

import importlib
import pandas as pd
from fastapi.testclient import TestClient

import backend.main as backend_main

backend_main = importlib.reload(backend_main)


def validate_group_response(client, params=None):
    params = params or {}

    original_response = client.get(
        "/api/v1/map/sites",
        params=params
    )

    grouped_response = client.get(
        "/api/v1/map/coordinate-groups",
        params=params
    )

    assert original_response.status_code == 200, (
        f"Original endpoint failed: {original_response.text}"
    )

    assert grouped_response.status_code == 200, (
        f"Grouped endpoint failed: {grouped_response.text}"
    )

    original = original_response.json()
    grouped = grouped_response.json()
    groups = grouped["groups"]

    # Every original mapped location must be represented.
    member_count = sum(
        group["location_count"] for group in groups
    )

    assert member_count == original["mapped_locations"]
    assert grouped["mapped_locations"] == original["mapped_locations"]
    assert grouped["filtered_locations"] == original["filtered_locations"]

    assert (
        grouped["excluded_invalid_coordinates"]
        == original["excluded_invalid_coordinates"]
    )

    # One marker per exact coordinate pair.
    coordinate_keys = [
        (group["Latitude"], group["Longitude"])
        for group in groups
    ]

    assert len(coordinate_keys) == len(set(coordinate_keys))

    # Every individual site ID must remain present and unique.
    all_member_ids = []

    for group in groups:
        members = group["members"]

        member_ids = [
            member["_spatial_group_id"]
            for member in members
        ]

        assert len(member_ids) == group["location_count"]
        assert len(member_ids) == len(set(member_ids))

        for member in members:
            assert float(member["Latitude"]) == group["Latitude"]
            assert float(member["Longitude"]) == group["Longitude"]

        all_member_ids.extend(member_ids)

    assert len(all_member_ids) == len(set(all_member_ids))

    return grouped


test_cases = [
    ("All locations", {}),
    ("P1", {"tier": "P1"}),
    ("P2", {"tier": "P2"}),
    ("P3", {"tier": "P3"}),
    ("Andhra Pradesh", {"state": "Andhra Pradesh"}),
    ("Telangana", {"state": "Telangana"}),
]

test_results = []

# IMPORTANT: Entering this context starts the FastAPI application.
with TestClient(backend_main.app) as client:

    print("FastAPI startup completed. Running API tests...\n")

    for label, params in test_cases:
        result = validate_group_response(client, params)

        test_results.append({
            "Filter": label,
            "Filtered locations": result["filtered_locations"],
            "Mapped locations": result["mapped_locations"],
            "Coordinate groups": result["coordinate_groups"],
            "Duplicate coordinate groups": (
                result["duplicate_coordinate_groups"]
            ),
            "Extra locations at duplicates": (
                result["extra_locations_at_duplicate_coordinates"]
            )
        })

    print("All six API tests passed.")

display(pd.DataFrame(test_results))

FastAPI startup completed. Running API tests...

All six API tests passed.


,Filter,Filtered locations,Mapped locations,Coordinate groups,Duplicate coordinate groups,Extra locations at duplicates
0,All locations,24199,24172,23280,808,892
1,P1,2421,2416,2377,38,39
2,P2,3629,3623,3533,84,90
3,P3,18149,18133,17759,345,374
4,Andhra Pradesh,1569,1550,1462,80,88
5,Telangana,947,946,895,50,51


In [93]:
# CELL 85 — Upgrade frontend to coordinate-group markers

from pathlib import Path
import re

frontend_path = PROJECT_ROOT / "app/frontend/src/MapExplorer.jsx"
styles_path = PROJECT_ROOT / "app/frontend/src/styles.css"

frontend_source = frontend_path.read_text(encoding="utf-8")
styles_source = styles_path.read_text(encoding="utf-8")

start_token = "function MapMarkers({ points, onSelect }) {"
end_token = "function MapLegend() {"

if frontend_source.count(start_token) != 1:
    raise RuntimeError("Could not uniquely locate MapMarkers. No changes made.")

start = frontend_source.index(start_token)
end = frontend_source.index(end_token, start)

if end <= start:
    raise RuntimeError("MapMarkers boundaries are invalid. No changes made.")

# Verify and prepare the endpoint replacement.
old_endpoint = "/api/v1/map/sites"
new_endpoint = "/api/v1/map/coordinate-groups"

if frontend_source.count(old_endpoint) != 1:
    raise RuntimeError(
        "Expected exactly one existing map API URL. No changes made."
    )

new_map_markers = r'''
function MapMarkers({ points, onSelect }) {
  const map = useMap();
  const selectRef = useRef(onSelect);

  useEffect(() => {
    selectRef.current = onSelect;
  }, [onSelect]);

  useEffect(() => {
    const clusters = L.markerClusterGroup({
      chunkedLoading: true,
      chunkInterval: 150,
      chunkDelay: 35,
      maxClusterRadius: 55,
      spiderfyOnMaxZoom: true,
      showCoverageOnHover: false,
      zoomToBoundsOnClick: true,
      animate: false,
      removeOutsideVisibleBounds: true,

      iconCreateFunction(cluster) {
        const count = cluster.getChildCount();
        const size = count < 10 ? "small" : count < 100 ? "medium" : "large";

        return L.divIcon({
          html: `<div class="cluster-bubble ${size}"><span>${number(count)}</span></div>`,
          className: "cluster-icon-wrapper",
          iconSize: [42, 42],
          iconAnchor: [21, 21]
        });
      }
    });

    for (const group of points || []) {
      const lat = Number(group.Latitude);
      const lng = Number(group.Longitude);

      if (
        !Number.isFinite(lat) ||
        !Number.isFinite(lng) ||
        lat < -90 || lat > 90 ||
        lng < -180 || lng > 180
      ) {
        continue;
      }

      const members = Array.isArray(group.members) ? group.members : [];
      const locationCount = Number(group.location_count || members.length || 1);

      const tier = String(group.Priority_tier || "")
        .split(" — ")[0]
        .toLowerCase();

      const markerIcon = L.divIcon({
        html: `
          <div class="coordinate-marker ${tier} ${group.has_state_conflict ? "conflict" : ""}">
            <span>${number(locationCount)}</span>
          </div>
        `,
        className: "coordinate-marker-wrapper",
        iconSize: [36, 36],
        iconAnchor: [18, 18]
      });

      const marker = L.marker([lat, lng], {
        icon: markerIcon,
        title: `${number(locationCount)} site location(s)`
      });

      const stateLabel = Array.isArray(group.states) && group.states.length
        ? group.states.join(", ")
        : String(group.State || "Unknown state");

      const districtLabel = Array.isArray(group.districts) && group.districts.length
        ? group.districts.join(", ")
        : String(group.District || "Unknown district");

      const counts = group.priority_counts || {};

      const prioritySummary = `
        <div class="map-popup-priorities">
          <span>P1: ${number(counts.P1)}</span>
          <span>P2: ${number(counts.P2)}</span>
          <span>P3: ${number(counts.P3)}</span>
        </div>
      `;

      const memberButtons = members.map((member, index) => {
        const siteId = String(member._spatial_group_id || "");
        const siteParts = siteId.split("|");
        const station = siteParts[2] || `Site ${index + 1}`;

        const memberTier = String(member.Priority_tier || "")
          .split(" — ")[0];

        const rankText = member.Priority_rank == null
          ? "Unranked"
          : `Rank #${number(member.Priority_rank)}`;

        return `
          <button
            type="button"
            class="map-popup-member"
            data-site-id="${escapeHTML(siteId)}"
          >
            <span>${escapeHTML(station)}</span>
            <small>
              ${escapeHTML(String(member.District || districtLabel))}
              · ${escapeHTML(memberTier)}
              · ${escapeHTML(rankText)}
            </small>
          </button>
        `;
      }).join("");

      const conflictWarning = group.has_state_conflict
        ? `<div class="map-popup-conflict">State metadata conflict: this coordinate has records from multiple states.</div>`
        : "";

      const popup = `
        <div class="map-popup coordinate-group-popup">
          <div class="map-popup-title">
            ${number(locationCount)} site${locationCount === 1 ? "" : "s"} at this coordinate
          </div>

          <div class="map-popup-subtitle">
            ${escapeHTML(districtLabel)}
          </div>

          <div class="map-popup-subtitle">
            ${escapeHTML(stateLabel)}
          </div>

          ${conflictWarning}

          ${prioritySummary}

          <div class="map-popup-member-heading">
            SELECT AN INDIVIDUAL SITE
          </div>

          <div class="map-popup-members">
            ${memberButtons || '<div class="map-popup-empty">No individual site records returned.</div>'}
          </div>
        </div>
      `;

      marker.bindPopup(popup, {
        maxWidth: 320,
        maxHeight: 300,
        closeButton: true
      });

      // Bind once per popup element. Selecting a member opens its
      // existing site profile through the original selection callback.
      marker.on("popupopen", (popupEvent) => {
        const popupElement = popupEvent.popup.getElement();

        if (
          !popupElement ||
          popupElement.dataset.siteSelectionBound === "true"
        ) {
          return;
        }

        popupElement.dataset.siteSelectionBound = "true";

        popupElement.addEventListener("click", (event) => {
          const button = event.target.closest("[data-site-id]");

          if (!button) return;

          const siteId = button.getAttribute("data-site-id");

          if (siteId) {
            selectRef.current?.(siteId);
            map.closePopup();
          }
        });
      });

      clusters.addLayer(marker);
    }

    map.addLayer(clusters);

    return () => {
      map.removeLayer(clusters);
    };
  }, [map, points]);

  return null;
}
'''

# Prepare updated source in memory first.
updated_frontend = (
    frontend_source[:start]
    + new_map_markers
    + "\n\n"
    + frontend_source[end:]
)

updated_frontend = updated_frontend.replace(old_endpoint, new_endpoint, 1)

# Change map-data references to use the new endpoint's "groups" array.
old_optional_refs = updated_frontend.count("mapData?.points")
old_direct_refs = updated_frontend.count("mapData.points")

updated_frontend = updated_frontend.replace(
    "mapData?.points", "mapData?.groups"
)
updated_frontend = updated_frontend.replace(
    "mapData.points", "mapData.groups"
)

if old_optional_refs + old_direct_refs == 0:
    raise RuntimeError(
        "No mapData.points references found. No files were written."
    )

# Preserve the old frontend as a backup.
backup_path = frontend_path.with_name(
    "MapExplorer.jsx.before_coordinate_groups.bak"
)

if not backup_path.exists():
    backup_path.write_text(frontend_source, encoding="utf-8")

# Add styles for grouped coordinate markers and selectable member buttons.
group_styles = r"""

/* Phase 10: exact-coordinate group markers */
.coordinate-marker-wrapper {
  background: transparent !important;
  border: none !important;
}

.coordinate-marker {
  width: 34px;
  height: 34px;
  display: flex;
  align-items: center;
  justify-content: center;
  border: 3px solid #ffffff;
  border-radius: 50%;
  color: #ffffff;
  font-size: 11px;
  font-weight: 800;
  box-shadow: 0 2px 8px rgba(15, 23, 42, 0.35);
}

.coordinate-marker.p1 {
  background: #dc2626;
}

.coordinate-marker.p2 {
  background: #d97706;
}

.coordinate-marker.p3 {
  background: #2563eb;
}

.coordinate-marker.conflict {
  outline: 3px solid #7c3aed;
  outline-offset: 2px;
}

.coordinate-group-popup {
  min-width: 220px;
}

.map-popup-priorities {
  display: flex;
  flex-wrap: wrap;
  gap: 6px;
  margin: 10px 0;
}

.map-popup-priorities span {
  padding: 4px 7px;
  border-radius: 6px;
  background: #f1f5f9;
  color: #334155;
  font-size: 11px;
  font-weight: 700;
}

.map-popup-conflict {
  margin-top: 8px;
  padding: 7px 8px;
  border-radius: 6px;
  background: #fef3c7;
  color: #92400e;
  font-size: 11px;
  line-height: 1.4;
}

.map-popup-member-heading {
  margin: 10px 0 6px;
  color: #64748b;
  font-size: 10px;
  font-weight: 800;
  letter-spacing: 0.06em;
}

.map-popup-members {
  display: flex;
  flex-direction: column;
  gap: 6px;
  max-height: 190px;
  overflow-y: auto;
  padding-right: 3px;
}

.map-popup-member {
  display: flex;
  flex-direction: column;
  gap: 3px;
  width: 100%;
  padding: 8px 9px;
  border: 1px solid #e2e8f0;
  border-radius: 7px;
  background: #ffffff;
  color: #0f172a;
  text-align: left;
  cursor: pointer;
}

.map-popup-member:hover {
  border-color: #64748b;
  background: #f8fafc;
}

.map-popup-member span {
  font-size: 12px;
  font-weight: 700;
}

.map-popup-member small {
  color: #64748b;
  font-size: 10px;
}

.map-popup-empty {
  padding: 8px;
  color: #64748b;
  font-size: 11px;
}
"""

if ".coordinate-marker-wrapper" not in styles_source:
    updated_styles = styles_source.rstrip() + "\n\n" + group_styles + "\n"
else:
    updated_styles = styles_source

# Write only after all transformations and checks succeeded.
frontend_path.write_text(updated_frontend, encoding="utf-8")
styles_path.write_text(updated_styles, encoding="utf-8")

print("FRONTEND UPDATE COMPLETE")
print("=" * 55)
print("Updated:", frontend_path)
print("Updated:", styles_path)
print("Backup:", backup_path)
print("Old map API references replaced: 1")
print("mapData?.points references replaced:", old_optional_refs)
print("mapData.points references replaced:", old_direct_refs)
print("Map markers now use exact-coordinate groups.")
print("Every individual site remains selectable from its marker popup.")

FRONTEND UPDATE COMPLETE
Updated: c:\career\projects\BoreWell Sentinel India\app\frontend\src\MapExplorer.jsx
Updated: c:\career\projects\BoreWell Sentinel India\app\frontend\src\styles.css
Backup: c:\career\projects\BoreWell Sentinel India\app\frontend\src\MapExplorer.jsx.before_coordinate_groups.bak
Old map API references replaced: 1
mapData?.points references replaced: 2
mapData.points references replaced: 1
Map markers now use exact-coordinate groups.
Every individual site remains selectable from its marker popup.


In [94]:
# CELL 86 — Build and verify the updated frontend

import shutil
import subprocess

frontend_dir = PROJECT_ROOT / "app/frontend"

npm_executable = shutil.which("npm") or shutil.which("npm.cmd")

if not npm_executable:
    raise FileNotFoundError(
        "npm was not found. Verify Node.js is installed and available in PATH."
    )

result = subprocess.run(
    [npm_executable, "run", "build"],
    cwd=str(frontend_dir),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\nBUILD STDERR:")
    print(result.stderr[-6000:])

if result.returncode != 0:
    raise RuntimeError(
        f"Frontend build failed with exit code {result.returncode}."
    )

print("\nFrontend production build passed.")


> borewell-sentinel-frontend@0.1.0 build
> vite build

vite v8.3.8 building client environment for production...
transforming...
✓ 2575 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                   0.61 kB │ gzip:   0.36 kB
dist/assets/index-BKSuR_Bg.css   32.43 kB │ gzip:  10.48 kB
dist/assets/index-Ch8o1N8B.js   855.75 kB │ gzip: 248.62 kB

✓ built in 1.27s


BUILD STDERR:
[plugin builtin:vite-reporter] 
(!) Some chunks are larger than 500 kB after minification. Consider:
- Using dynamic import() to code-split the application
- Use build.rolldownOptions.output.codeSplitting to improve chunking: https://rolldown.rs/reference/OutputOptions.codeSplitting
- Adjust chunk size limit for this warning via build.chunkSizeWarningLimit.


Frontend production build passed.


In [95]:
# CELL 87 — Verify duplicate and cross-state coordinate groups

from fastapi.testclient import TestClient
import pandas as pd

with TestClient(backend_main.app) as client:
    response = client.get("/api/v1/map/coordinate-groups")

    assert response.status_code == 200, response.text
    data = response.json()

groups = data["groups"]

# Chittoor coordinate previously identified in the audit
chittoor_group = next(
    (
        g for g in groups
        if round(g["Latitude"], 6) == 13.206667
        and round(g["Longitude"], 6) == 79.101667
    ),
    None
)

# Gudimalkapur cross-state coordinate
gudimalkapur_group = next(
    (
        g for g in groups
        if round(g["Latitude"], 6) == 17.383333
        and round(g["Longitude"], 6) == 78.433333
    ),
    None
)

print("TOTAL COORDINATE GROUPS:", f"{len(groups):,}")

print("\n1. CHITTOOR DUPLICATE COORDINATE")
if chittoor_group:
    print("Location count:", chittoor_group["location_count"])
    display(pd.DataFrame([
        {
            "Site ID": m["_spatial_group_id"],
            "State": m["State"],
            "District": m["District"],
            "Priority": m["Priority_tier"]
        }
        for m in chittoor_group["members"]
    ]))
else:
    print("The expected Chittoor coordinate was not found.")

print("\n2. GUDIMALKAPUR CROSS-STATE COORDINATE")
if gudimalkapur_group:
    print("Location count:", gudimalkapur_group["location_count"])
    print("State conflict:", gudimalkapur_group["has_state_conflict"])
    print("States:", gudimalkapur_group["states"])

    display(pd.DataFrame([
        {
            "Site ID": m["_spatial_group_id"],
            "State": m["State"],
            "District": m["District"]
        }
        for m in gudimalkapur_group["members"]
    ]))
else:
    print("The expected Gudimalkapur coordinate was not found.")

assert chittoor_group is not None, "Chittoor coordinate group missing."
assert gudimalkapur_group is not None, "Gudimalkapur group missing."
assert gudimalkapur_group["has_state_conflict"] is True

print("\nBoth coordinate-group checks passed.")

TOTAL COORDINATE GROUPS: 23,280

1. CHITTOOR DUPLICATE COORDINATE
Location count: 4


,Site ID,State,District,Priority
0,andhra pradesh|chittoor|chittoor|site|13.20667...,andhra pradesh,chittoor,P2 — Next 15%
1,andhra pradesh|chittoor|srikalahasthi alt|site...,andhra pradesh,chittoor,P3 — Remaining locations
2,andhra pradesh|chittoor|puttur-alt|site|13.206...,andhra pradesh,chittoor,P3 — Remaining locations
3,andhra pradesh|chittoor|chittoor-alt|site|13.2...,andhra pradesh,chittoor,P3 — Remaining locations



2. GUDIMALKAPUR CROSS-STATE COORDINATE
Location count: 2
State conflict: True
States: ['andhra pradesh', 'telangana']


,Site ID,State,District
0,telangana|hyderabad|gudimalkapur2|site|17.3833...,telangana,hyderabad
1,andhra pradesh|anantapur|gudimalkapur-pz|site|...,andhra pradesh,anantapur



Both coordinate-group checks passed.


In [96]:
# CELL 88 — Diagnose the live backend response

import requests
import time
import json

BASE_URL = "http://127.0.0.1:8000"

def check_live_endpoint(path, timeout=180):
    url = BASE_URL + path
    print(f"\nTesting: {path}")

    start = time.perf_counter()

    try:
        response = requests.get(url, timeout=(5, timeout))
        elapsed = time.perf_counter() - start

        print("HTTP status:", response.status_code)
        print("Response time:", f"{elapsed:.2f} seconds")
        print("Response size:", f"{len(response.content) / 1024**2:.2f} MB")

        response.raise_for_status()
        return response.json()

    except Exception as error:
        print("ERROR:", repr(error))
        return None


health = check_live_endpoint("/health", timeout=10)

states_data = check_live_endpoint(
    "/api/v1/dashboard/states?limit=40",
    timeout=30
)

map_data = check_live_endpoint(
    "/api/v1/map/coordinate-groups",
    timeout=180
)

if map_data:
    print("\nLIVE MAP DATA SUMMARY")
    print("=" * 45)

    groups = map_data.get("groups", [])

    print("Filtered locations:", map_data.get("filtered_locations"))
    print("Mapped locations:", map_data.get("mapped_locations"))
    print("Coordinate groups:", len(groups))
    print("Duplicate groups:", map_data.get("duplicate_coordinate_groups"))
    print("Total group members:", sum(
        group.get("location_count", 0) for group in groups
    ))

    print("Expected locations:", 24199)
    print("Expected mapped locations:", 24172)

    assert len(groups) == map_data["coordinate_groups"]
    assert sum(g["location_count"] for g in groups) == 24172

    print("\nLIVE MAP API: DATA CHECK PASSED")


Testing: /health
HTTP status: 200
Response time: 5.65 seconds
Response size: 0.00 MB

Testing: /api/v1/dashboard/states?limit=40
HTTP status: 200
Response time: 0.63 seconds
Response size: 0.00 MB

Testing: /api/v1/map/coordinate-groups
HTTP status: 200
Response time: 68.47 seconds
Response size: 17.50 MB

LIVE MAP DATA SUMMARY
Filtered locations: 24199
Mapped locations: 24172
Coordinate groups: 23280
Duplicate groups: 808
Total group members: 24172
Expected locations: 24199
Expected mapped locations: 24172

LIVE MAP API: DATA CHECK PASSED


In [97]:
# CELL 92 — Apply Andhra Pradesh-only map corrections

import ast
import pandas as pd
from pathlib import Path

# 1. Build the override register from the completed boundary audit.
ap_mask = (
    boundary_audit_df["State"].astype(str).str.strip().str.casefold().eq("andhra pradesh")
    & boundary_audit_df["Boundary_match_status"].eq("OUTSIDE_RECORDED_STATE")
)

ap_overrides = boundary_audit_df.loc[
    ap_mask,
    ["_spatial_group_id", "State", "Boundary_state_matches"]
].copy()

ap_overrides = ap_overrides.rename(columns={
    "State": "Original_State",
    "Boundary_state_matches": "Map_state"
})

ap_overrides["Map_state"] = (
    ap_overrides["Map_state"].astype(str).str.strip().str.casefold()
)

if len(ap_overrides) != 241:
    raise ValueError(
        f"Expected 241 Andhra Pradesh overrides, found {len(ap_overrides)}."
    )

if ap_overrides["_spatial_group_id"].nunique() != 241:
    raise ValueError("Duplicate spatial group IDs found in overrides.")

if ap_overrides["Map_state"].isin(["", "nan"]).any():
    raise ValueError("Some Andhra Pradesh overrides have no destination state.")

override_path = (
    PROJECT_ROOT
    / "reports/map_quality/phase10_andhra_pradesh_map_overrides.csv"
)

ap_overrides["Correction_scope"] = "ANDHRA_PRADESH_ONLY"
ap_overrides["Correction_basis"] = "2017 state boundary coordinate match"
ap_overrides["Original_metadata_preserved"] = True

ap_overrides.to_csv(override_path, index=False)

print("Andhra Pradesh override register:", override_path)
print("Override locations:", len(ap_overrides))
display(
    ap_overrides.groupby("Map_state")
    .size()
    .sort_values(ascending=False)
    .rename("Locations")
    .to_frame()
)

# 2. Replace only the coordinate-groups endpoint in main.py.
main_path = PROJECT_ROOT / "app/backend/main.py"
main_source = main_path.read_text(encoding="utf-8")
tree = ast.parse(main_source)

target = None

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        for decorator in node.decorator_list:
            if (
                isinstance(decorator, ast.Call)
                and isinstance(decorator.func, ast.Attribute)
                and decorator.func.attr == "get"
                and decorator.args
                and isinstance(decorator.args[0], ast.Constant)
                and decorator.args[0].value == "/api/v1/map/coordinate-groups"
            ):
                if target is not None:
                    raise RuntimeError("Duplicate coordinate-groups routes.")
                target = node

if target is None:
    raise RuntimeError("Coordinate-groups endpoint not found.")

endpoint_start = min(
    [target.lineno] + [d.lineno for d in target.decorator_list]
) - 1
endpoint_end = target.end_lineno

optimized_endpoint = r'''
@app.get("/api/v1/map/coordinate-groups")
def map_coordinate_groups(
    state: str | None = Query(default=None, max_length=100),
    tier: str | None = Query(default=None, pattern="^(P1|P2|P3)$")
):
    """Group exact coordinates, applying map-only Andhra Pradesh overrides."""
    import pandas as pd
    from pathlib import Path
    from collections import Counter

    sites = get_data()["sites"]

    override_path = (
        Path(__file__).resolve().parents[2]
        / "reports/map_quality/phase10_andhra_pradesh_map_overrides.csv"
    )

    overrides = pd.read_csv(
        override_path,
        usecols=["_spatial_group_id", "Map_state"],
        dtype=str
    )

    override_map = dict(zip(
        overrides["_spatial_group_id"],
        overrides["Map_state"]
    ))

    filtered = sites.copy()

    # All non-Andhra-Pradesh records retain their original state.
    # Only the 241 audited Andhra Pradesh site IDs receive overrides.
    filtered["Map_state"] = (
        filtered["_spatial_group_id"].astype(str)
        .map(override_map)
        .fillna(filtered["State"].astype(str))
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    if state:
        filtered = filtered.loc[
            filtered["Map_state"] == state.strip().casefold()
        ]

    if tier:
        filtered = filtered.loc[
            filtered["Priority_tier"]
            .fillna("")
            .astype(str)
            .str.startswith(tier.upper())
        ]

    total_filtered = len(filtered)

    latitude = pd.to_numeric(filtered["Latitude"], errors="coerce")
    longitude = pd.to_numeric(filtered["Longitude"], errors="coerce")

    valid = (
        latitude.between(-90, 90)
        & longitude.between(-180, 180)
    )

    geo = filtered.loc[valid, [
        "_spatial_group_id",
        "State",
        "Map_state",
        "District",
        "Priority_rank",
        "Priority_tier",
        "Latitude",
        "Longitude"
    ]].copy()

    geo["Latitude"] = latitude.loc[valid]
    geo["Longitude"] = longitude.loc[valid]

    geo["Priority_rank"] = pd.to_numeric(
        geo["Priority_rank"], errors="coerce"
    )

    geo["Station"] = (
        geo["_spatial_group_id"]
        .astype(str)
        .str.split("|")
        .str[2]
        .fillna("Unknown")
    )

    geo = geo.sort_values(
        "Priority_rank",
        ascending=True,
        na_position="last"
    ).reset_index(drop=True)

    compact_columns = [
        "_spatial_group_id",
        "Station",
        "State",
        "Map_state",
        "District",
        "Priority_rank",
        "Priority_tier"
    ]

    serialized_members = records_from_frame(
        geo[compact_columns]
    )

    coordinate_indices = geo.groupby(
        ["Latitude", "Longitude"],
        sort=False,
        dropna=False
    ).indices

    groups = []

    for (lat, lng), positions in coordinate_indices.items():
        members = [
            serialized_members[int(position)]
            for position in positions
        ]

        states = sorted({
            str(member["Map_state"]).strip().casefold()
            for member in members
            if member.get("Map_state")
        })

        source_states = sorted({
            str(member["State"]).strip().casefold()
            for member in members
            if member.get("State")
        })

        districts = sorted({
            str(member["District"]).strip()
            for member in members
            if member.get("District")
        })

        metadata_conflict = any(
            str(member.get("State") or "").strip().casefold()
            != str(member.get("Map_state") or "").strip().casefold()
            for member in members
        )

        counts = Counter(
            str(member.get("Priority_tier") or "").split(" — ", 1)[0]
            for member in members
        )

        representative = members[0]

        groups.append({
            "Latitude": float(lat),
            "Longitude": float(lng),
            "location_count": len(members),

            "State": states[0] if len(states) == 1 else "Multiple states",
            "District": (
                districts[0] if len(districts) == 1
                else "Multiple districts"
            ),

            "states": states,
            "source_states": source_states,
            "districts": districts,

            "state_count": len(states),
            "district_count": len(districts),
            "has_state_conflict": len(states) > 1,
            "has_metadata_conflict": metadata_conflict,

            "Priority_rank": representative.get("Priority_rank"),
            "Priority_tier": representative.get("Priority_tier"),

            "priority_counts": {
                "P1": int(counts.get("P1", 0)),
                "P2": int(counts.get("P2", 0)),
                "P3": int(counts.get("P3", 0))
            },

            "members": members
        })

    duplicate_groups = [
        group for group in groups
        if group["location_count"] > 1
    ]

    return {
        "filtered_locations": int(total_filtered),
        "mapped_locations": int(len(geo)),
        "excluded_invalid_coordinates": int(total_filtered - len(geo)),

        "coordinate_groups": int(len(groups)),
        "duplicate_coordinate_groups": int(len(duplicate_groups)),
        "locations_in_duplicate_groups": int(
            sum(group["location_count"] for group in duplicate_groups)
        ),
        "extra_locations_at_duplicate_coordinates": int(
            sum(group["location_count"] - 1 for group in duplicate_groups)
        ),

        "groups": groups
    }
'''

source_lines = main_source.splitlines(keepends=True)

updated_source = (
    "".join(source_lines[:endpoint_start])
    + optimized_endpoint.lstrip("\n")
    + "\n"
    + "".join(source_lines[endpoint_end:])
)

ast.parse(updated_source)

backend_backup = main_path.with_name(
    "main.py.before_andhra_pradesh_map_correction.bak"
)

if not backend_backup.exists():
    backend_backup.write_text(main_source, encoding="utf-8")

main_path.write_text(updated_source, encoding="utf-8")

print("\nBACKEND UPDATED")
print("Only the coordinate-groups endpoint was changed.")
print("Original site State and District columns remain untouched.")
print("Other states retain their existing map assignments.")
print("Backup:", backend_backup)

Andhra Pradesh override register: c:\career\projects\BoreWell Sentinel India\reports\map_quality\phase10_andhra_pradesh_map_overrides.csv
Override locations: 241


,Locations
Map_state,
maharashtra,161
rajasthan,16
telangana,16
gujarat,12
himachal pradesh,9
uttar pradesh,9
haryana,6
madhya pradesh,5
jharkhand,3



BACKEND UPDATED
Only the coordinate-groups endpoint was changed.
Original site State and District columns remain untouched.
Other states retain their existing map assignments.
Backup: c:\career\projects\BoreWell Sentinel India\app\backend\main.py.before_andhra_pradesh_map_correction.bak


In [98]:
# CELL 93 — Validate Andhra Pradesh correction and unaffected states

import requests
import time

BASE_URL = "http://127.0.0.1:8000"

# Wait for Uvicorn's automatic reload to finish.
time.sleep(3)

def get_group_data(params=None):
    response = requests.get(
        f"{BASE_URL}/api/v1/map/coordinate-groups",
        params=params or {},
        timeout=(10, 120)
    )
    response.raise_for_status()
    return response.json()


# All locations
all_data = get_group_data()

assert all_data["filtered_locations"] == 24199
assert all_data["mapped_locations"] == 24172

# Andhra Pradesh after map-only corrections
ap_data = get_group_data({"state": "andhra pradesh"})

# Karnataka remains unchanged
ka_data = get_group_data({"state": "karnataka"})

# Telangana after accepting the Andhra Pradesh overrides
ts_data = get_group_data({"state": "telangana"})

results = pd.DataFrame([
    {
        "Check": "All states",
        "Filtered locations": all_data["filtered_locations"],
        "Mapped locations": all_data["mapped_locations"],
        "Coordinate groups": all_data["coordinate_groups"]
    },
    {
        "Check": "Andhra Pradesh — corrected map state",
        "Filtered locations": ap_data["filtered_locations"],
        "Mapped locations": ap_data["mapped_locations"],
        "Coordinate groups": ap_data["coordinate_groups"]
    },
    {
        "Check": "Karnataka — unchanged",
        "Filtered locations": ka_data["filtered_locations"],
        "Mapped locations": ka_data["mapped_locations"],
        "Coordinate groups": ka_data["coordinate_groups"]
    },
    {
        "Check": "Telangana — includes mapped AP overrides",
        "Filtered locations": ts_data["filtered_locations"],
        "Mapped locations": ts_data["mapped_locations"],
        "Coordinate groups": ts_data["coordinate_groups"]
    }
])

display(results)

# Verify all 241 override IDs are present under their map destination states.
ap_override_ids = set(ap_overrides["_spatial_group_id"])

all_member_map = {
    member["_spatial_group_id"]: member
    for group in all_data["groups"]
    for member in group["members"]
}

assert ap_override_ids.issubset(all_member_map)

for row in ap_overrides.itertuples(index=False):
    member = all_member_map[row._0] if hasattr(row, "_0") else None

# Verify unaffected source states retain their original map state.
for group in all_data["groups"]:
    for member in group["members"]:
        site_id = member["_spatial_group_id"]

        if site_id not in ap_override_ids:
            assert (
                str(member["State"]).casefold()
                == str(member["Map_state"]).casefold()
            )

# Verify Gudimalkapur now maps both site records to Telangana.
gudimalkapur = next(
    group for group in all_data["groups"]
    if round(group["Latitude"], 6) == 17.383333
    and round(group["Longitude"], 6) == 78.433333
)

assert gudimalkapur["states"] == ["telangana"]
assert gudimalkapur["has_metadata_conflict"] is True

# Check the number of Andhra Pradesh records moved out of its map filter.
assert ap_data["filtered_locations"] == 1569 - 241

print("\nALL ANDHRA PRADESH-ONLY VALIDATIONS PASSED")
print("Corrected site IDs:", len(ap_override_ids))
print("Other site Map_state values unchanged.")
print("Gudimalkapur now maps to Telangana.")
print("Original State and District metadata preserved.")

,Check,Filtered locations,Mapped locations,Coordinate groups
0,All states,24199,24172,23280
1,Andhra Pradesh — corrected map state,1328,1309,1227
2,Karnataka — unchanged,2044,2044,1960
3,Telangana — includes mapped AP overrides,963,962,910



ALL ANDHRA PRADESH-ONLY VALIDATIONS PASSED
Corrected site IDs: 241
Other site Map_state values unchanged.
Gudimalkapur now maps to Telangana.
Original State and District metadata preserved.


In [99]:
# CELL 94 — Update popup conflict explanation

frontend_path = PROJECT_ROOT / "app/frontend/src/MapExplorer.jsx"
frontend_source = frontend_path.read_text(encoding="utf-8")

old_conflict = '''      const conflictWarning = group.has_state_conflict
        ? `<div class="map-popup-conflict">State metadata conflict: this coordinate has records from multiple states.</div>`
        : "";'''

new_conflict = '''      const conflictWarning = group.has_metadata_conflict
        ? `<div class="map-popup-conflict">Source state metadata differs from the map location for one or more sites. Original CGWB metadata is preserved.</div>`
        : group.has_state_conflict
          ? `<div class="map-popup-conflict">Multiple map states share this coordinate.</div>`
          : "";'''

if frontend_source.count(old_conflict) != 1:
    raise RuntimeError("Popup warning section did not match expected code.")

frontend_source = frontend_source.replace(
    old_conflict, new_conflict, 1
)

old_member = '''        const rankText = member.Priority_rank == null
          ? "Unranked"
          : `Rank #${number(member.Priority_rank)}`;

        return `
          <button
            type="button"
            class="map-popup-member"
            data-site-id="${escapeHTML(siteId)}"
          >
            <span>${escapeHTML(station)}</span>
            <small>
              ${escapeHTML(String(member.District || districtLabel))}
              · ${escapeHTML(memberTier)}
              · ${escapeHTML(rankText)}
            </small>
          </button>
        `;'''

new_member = '''        const rankText = member.Priority_rank == null
          ? "Unranked"
          : `Rank #${number(member.Priority_rank)}`;

        const sourceState = String(member.State || "");
        const mapState = String(member.Map_state || group.State || "");

        const stateDescription = sourceState.toLowerCase() !== mapState.toLowerCase()
          ? `Source: ${sourceState} → Map: ${mapState}`
          : `Map: ${mapState}`;

        return `
          <button
            type="button"
            class="map-popup-member"
            data-site-id="${escapeHTML(siteId)}"
          >
            <span>${escapeHTML(station)}</span>
            <small>
              ${escapeHTML(String(member.District || districtLabel))}
              · ${escapeHTML(memberTier)}
              · ${escapeHTML(rankText)}
              · ${escapeHTML(stateDescription)}
            </small>
          </button>
        `;'''

if frontend_source.count(old_member) != 1:
    raise RuntimeError("Popup member section did not match expected code.")

frontend_source = frontend_source.replace(
    old_member, new_member, 1
)

# Verify JavaScript source contains the new fields before saving.
assert "group.has_metadata_conflict" in frontend_source
assert "member.Map_state" in frontend_source

frontend_backup = frontend_path.with_name(
    "MapExplorer.jsx.before_ap_popup_update.bak"
)

if not frontend_backup.exists():
    frontend_backup.write_text(
        frontend_path.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

frontend_path.write_text(frontend_source, encoding="utf-8")

# Build the frontend.
import shutil
import subprocess

npm = shutil.which("npm") or shutil.which("npm.cmd")

build = subprocess.run(
    [npm, "run", "build"],
    cwd=str(PROJECT_ROOT / "app/frontend"),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(build.stdout)

if build.stderr:
    print(build.stderr[-3000:])

if build.returncode != 0:
    raise RuntimeError("Frontend build failed.")

print("Andhra Pradesh-only map correction build passed.")


> borewell-sentinel-frontend@0.1.0 build
> vite build

vite v8.3.8 building client environment for production...
transforming...
✓ 2575 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                   0.61 kB │ gzip:   0.36 kB
dist/assets/index-BKSuR_Bg.css   32.43 kB │ gzip:  10.48 kB
dist/assets/index-CBT2oajG.js   856.05 kB │ gzip: 248.73 kB

✓ built in 1.20s

[plugin builtin:vite-reporter] 
(!) Some chunks are larger than 500 kB after minification. Consider:
- Using dynamic import() to code-split the application
- Use build.rolldownOptions.output.codeSplitting to improve chunking: https://rolldown.rs/reference/OutputOptions.codeSplitting
- Adjust chunk size limit for this warning via build.chunkSizeWarningLimit.

Andhra Pradesh-only map correction build passed.
